# MediSense-Cardio -- Traffic-light safeguard for ECG digitisers (demo, click Run All)

**The problem:** a program that turns a photo of an ECG back into a signal sometimes gives a wrong heart rate, and nothing tells the user.
**What this tool does:** it looks only at the digitised ECG line (never at the true signal) and shows a light: **GREEN** = no warning, **AMBER** = check it, **RED** = do not trust the heart rate.
**What you will see (about 1-2 minutes):** (1) six real examples from our own phone photos of ECG pages on a screen, each with the photo, the signal the digitiser drew, the light and the reasons; (2) how often each light was right on 1,200+ competition captures and on our 60 own-phone photos; (3) the limits.

*Research prototype, not a medical device. Built and frozen before it was tested on the real photos.*


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neurokit2'], capture_output=True)
import os, io, json, base64, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings('ignore')
os.makedirs('safeguard', exist_ok=True)
open('safeguard/ecg_traffic_light.py', 'w', encoding='utf-8').write('"""ECG digitiser traffic light: is the heart rate read from a digitised ECG trustworthy?\n\nInput : one digitised 12-lead ECG as the Open-ECG-Digitizer saves it (1000 Hz, microvolts, 10 s, NaN where a lead was not read), as a dict\n        {"I": array, "II": array, ... "V6": array} or an .npz file with those keys.\nOutput: GREEN (no warning), AMBER (check by a person) or RED (do not trust the heart rate), a score, and plain-language reasons.\nIt NEVER needs the true signal. It looks only at the digitised signal: beat regularity, agreement between the 12 leads (Einthoven and Goldberger rules),\nhow complete the leads are, and whether five standard beat detectors agree.\n\nModel and thresholds: safeguard/model.json (built once by safeguard/train_safeguard.py from the simulated new 100 only; rules fixed in\ndocs/plan/analysis_plan_changes.md, 2026-11-10). Needs: numpy, neurokit2.   This is a research prototype, not a medical device.\n"""\nimport json\nimport os\nimport warnings\n\nimport numpy as np\n\nwarnings.filterwarnings("ignore")\nFS = 1000\nLEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]\nFEATURES = ["ii_complete", "n_unreadable_leads", "mean_other_lead_complete", "n_beats", "hr_ii", "hr_implausible", "rr_cv", "rr_max_ratio", "rr_min_ratio",\n            "amp_log", "hf_share", "einthoven", "goldberger", "xlead_count_match", "xlead_count_absdiff"]\n_MODEL_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "model.json")\n\n\n# ------------------------------------------------------------------ features (identical to scripts/analysis/section_D2_features.py)\ndef _prep(sig, min_s=1.5, min_fill=0.9):\n    sig = np.asarray(sig, dtype=float)\n    idx = np.where(np.isfinite(sig))[0]\n    if len(idx) == 0:\n        return None\n    a0, b0 = int(idx[0]), int(idx[-1]) + 1\n    seg = sig[a0:b0].copy()\n    fin = np.isfinite(seg)\n    if len(seg) < min_s * FS or fin.mean() < min_fill:\n        return None\n    if not fin.all():\n        seg[~fin] = np.interp(np.where(~fin)[0], np.where(fin)[0], seg[fin])\n    return a0, seg\n\n\ndef _rms(x):\n    return float(np.sqrt(np.mean(np.square(x))))\n\n\ndef output_features(leads):\n    """The 15 output features of one digitised ECG (leads: dict of 12 arrays)."""\n    import neurokit2 as nk\n    L = {k: np.asarray(leads[k], dtype=float) if k in leads else np.full(10000, np.nan) for k in LEADS}\n    out = {}\n    ii = L["II"]\n    out["ii_complete"] = float(np.isfinite(ii).mean())\n    cnt = {k: int(np.isfinite(v).sum()) for k, v in L.items()}\n    out["n_unreadable_leads"] = int(sum(1 for k in LEADS if cnt[k] < 1500))\n    out["mean_other_lead_complete"] = float(np.mean([min(1.0, cnt[k] / 2500.0) for k in LEADS if k != "II"]))\n    nan = np.nan\n    for k in FEATURES[3:]:\n        out[k] = nan\n    p2 = _prep(ii, min_s=3.0)\n    if p2 is None:\n        return out\n    a2, seg2 = p2\n    try:\n        cl = nk.ecg_clean(seg2, sampling_rate=FS, method="neurokit")\n        rpk = np.asarray(nk.ecg_peaks(cl, sampling_rate=FS)[1]["ECG_R_Peaks"])\n    except Exception:\n        return out\n    out["n_beats"] = int(len(rpk))\n    out["amp_log"] = float(np.log10(max(1e-6, np.percentile(seg2, 95) - np.percentile(seg2, 5))))\n    f = np.abs(np.fft.rfft(seg2 - seg2.mean())) ** 2\n    fr = np.fft.rfftfreq(len(seg2), 1 / FS)\n    out["hf_share"] = float(f[fr > 40].sum() / max(1e-12, f.sum()))\n    if len(rpk) >= 3:\n        rr = np.diff(rpk) / FS\n        hr = float(np.mean(60.0 / rr))\n        out["hr_ii"] = hr\n        out["hr_implausible"] = float(hr < 30 or hr > 200)\n        out["rr_cv"] = float(np.std(rr) / np.mean(rr))\n        out["rr_max_ratio"] = float(rr.max() / np.median(rr))\n        out["rr_min_ratio"] = float(rr.min() / np.median(rr))\n    m = np.isfinite(L["I"]) & np.isfinite(L["II"]) & np.isfinite(L["III"])\n    if m.sum() >= 1000:\n        i_, ii_, iii_ = (L[k][m] - L[k][m].mean() for k in ("I", "II", "III"))\n        out["einthoven"] = float(_rms(ii_ - i_ - iii_) / max(1e-9, _rms(ii_)))\n    m = np.isfinite(L["aVR"]) & np.isfinite(L["aVL"]) & np.isfinite(L["aVF"])\n    if m.sum() >= 1000:\n        a_, b_, c_ = (L[k][m] - L[k][m].mean() for k in ("aVR", "aVL", "aVF"))\n        out["goldberger"] = float(_rms(a_ + b_ + c_) / max(1e-9, np.mean([_rms(a_), _rms(b_), _rms(c_)])))\n    match, diffs = [], []\n    for k in LEADS:\n        if k == "II":\n            continue\n        pr = _prep(L[k])\n        if pr is None:\n            continue\n        a0, seg = pr\n        try:\n            c = nk.ecg_clean(seg, sampling_rate=FS, method="neurokit")\n            pk = np.asarray(nk.ecg_peaks(c, sampling_rate=FS)[1]["ECG_R_Peaks"]) + a0\n        except Exception:\n            continue\n        lo, hi = a0 + 100, a0 + len(seg) - 100\n        n_lead = int(((pk >= lo) & (pk < hi)).sum())\n        n_ref = int((((rpk + a2) >= lo) & ((rpk + a2) < hi)).sum())\n        match.append(float(n_lead == n_ref))\n        diffs.append(abs(n_lead - n_ref))\n    if match:\n        out["xlead_count_match"] = float(np.mean(match))\n        out["xlead_count_absdiff"] = float(np.mean(diffs))\n    return out\n\n\ndef detector_spread(leads):\n    """Heart rate from five standard beat detectors on lead II; returns (spread in bpm, list of the five rates). For the reasons only."""\n    import neurokit2 as nk\n    s = np.asarray(leads["II"], dtype=float)\n    s = s[np.isfinite(s)] / 1000.0\n    hrs = []\n    for m, c in (("neurokit", False), ("neurokit", True), ("pantompkins1985", True), ("hamilton2002", True), ("elgendi2010", True)):\n        try:\n            x = nk.ecg_clean(s, sampling_rate=FS, method="neurokit") if c else s\n            r = np.asarray(nk.ecg_peaks(x, sampling_rate=FS, method=m)[1]["ECG_R_Peaks"], dtype=float) / FS\n            hrs.append(float(60.0 / np.diff(r).mean()) if len(r) >= 3 else np.nan)\n        except Exception:\n            hrs.append(np.nan)\n    a = np.array(hrs)\n    ok = np.isfinite(a)\n    return (float(a[ok].max() - a[ok].min()) if ok.sum() >= 2 else np.nan), hrs\n\n\n# ------------------------------------------------------------------ model (numpy only)\ndef load_model(path=_MODEL_PATH):\n    with open(path, encoding="utf-8") as f:\n        return json.load(f)\n\n\ndef score_from_features(feat, model=None):\n    """Logistic risk score in [0, 1] from the 15 features (median imputation, standardisation, logistic regression: all stored in model.json)."""\n    model = model or load_model()\n    x = np.array([feat.get(k, np.nan) for k in model["features"]], dtype=float)\n    med = np.array(model["impute_median"], dtype=float)\n    x = np.where(np.isfinite(x), x, med)\n    z = (x - np.array(model["scaler_mean"])) / np.array(model["scaler_scale"])\n    s = float(np.dot(z, np.array(model["coef"])) + model["intercept"])\n    return float(1.0 / (1.0 + np.exp(-s)))\n\n\ndef load_npz(path):\n    z = np.load(path, allow_pickle=False)\n    return {k: np.asarray(z[k], dtype=float) for k in z.files}\n\n\ndef check(leads, model=None):\n    """Returns {\'light\': \'GREEN\'|\'AMBER\'|\'RED\', \'score\': float, \'reasons\': [str], \'features\': dict, \'detector_spread_bpm\': float}."""\n    model = model or load_model()\n    feat = output_features(leads)\n    reasons = []\n    n_beats, hr = feat.get("n_beats"), feat.get("hr_ii")\n    no_hr = (n_beats is None) or not np.isfinite(n_beats) or n_beats < 3 or hr is None or not np.isfinite(hr)\n    score = score_from_features(feat, model)\n    if no_hr:\n        light = "RED"\n        reasons.append("no usable heart rate: fewer than 3 beats found in lead II")\n    else:\n        light = "RED" if score >= model["red_threshold"] else ("AMBER" if score >= model["amber_threshold"] else "GREEN")\n    p90 = model["reason_p90"]\n    msgs = {"rr_cv": "beat spacing is irregular", "rr_max_ratio": "one beat gap is much longer than the others (a missed beat?)",\n            "rr_min_ratio": "one beat gap is much shorter than the others (an extra beat?)", "einthoven": "leads I, II and III disagree with each other (should satisfy II = I + III)",\n            "goldberger": "aVR, aVL and aVF disagree with each other (should add up to zero)", "xlead_count_absdiff": "other leads count a different number of beats than lead II",\n            "n_unreadable_leads": "several leads could not be read"}\n    for k, m in msgs.items():\n        v = feat.get(k)\n        if v is not None and np.isfinite(v) and v > p90.get(k, np.inf):\n            reasons.append(m)\n    sp = np.nan\n    if not no_hr and "II" in leads:\n        sp, _ = detector_spread(leads)\n        if np.isfinite(sp) and sp > model["detector_spread_note_bpm"]:\n            reasons.append(f"five beat detectors disagree about the heart rate by {sp:.0f} bpm")\n    if light == "GREEN":                                    # out-of-range guard (rule added 2026-11-10, before any real-data evaluation)\n        odd = [k for k in model.get("guard_features", []) if feat.get(k) is not None and np.isfinite(feat[k])\n               and (feat[k] < model["train_min"][k] or feat[k] > model["train_max"][k])]\n        if odd:\n            light = "AMBER"\n            reasons.append("signal looks unlike the data the tool was trained on (" + ", ".join(odd[:3]) + ")")\n    if light == "GREEN" and not reasons:\n        reasons.append("beats are regular and the leads agree")\n    elif light == "GREEN":                                  # minor notes only: they did not change the light\n        reasons = ["minor note (did not change the light): " + x for x in reasons]\n    return dict(light=light, score=score, reasons=reasons, features=feat, detector_spread_bpm=sp)\n\n\ndef check_npz(path, model=None):\n    return check(load_npz(path), model)\n\n\nif __name__ == "__main__":\n    import sys\n    if len(sys.argv) < 2:\n        print("usage: python safeguard/ecg_traffic_light.py file1.npz [file2.npz ...]")\n        raise SystemExit(1)\n    for p in sys.argv[1:]:\n        r = check_npz(p)\n        print(f"{os.path.basename(p)}: {r[\'light\']}  (score {r[\'score\']:.2f})  " + "; ".join(r["reasons"]))\n')
open('safeguard/model.json', 'w', encoding='utf-8').write('{\n "features": [\n  "ii_complete",\n  "n_unreadable_leads",\n  "mean_other_lead_complete",\n  "n_beats",\n  "hr_ii",\n  "hr_implausible",\n  "rr_cv",\n  "rr_max_ratio",\n  "rr_min_ratio",\n  "amp_log",\n  "hf_share",\n  "einthoven",\n  "goldberger",\n  "xlead_count_match",\n  "xlead_count_absdiff"\n ],\n "impute_median": [\n  0.9994,\n  0.0,\n  0.9951818181818182,\n  12.0,\n  74.0216700184688,\n  0.0,\n  0.0358498192961257,\n  1.046626984126984,\n  0.9454136184580751,\n  2.5255899390823826,\n  0.0007362414010553,\n  0.06707477429236305,\n  0.0937501659393144,\n  0.7272727272727273,\n  0.2857142857142857\n ],\n "scaler_mean": [\n  0.9802214738510302,\n  0.13946117274167988,\n  0.9805955337847573,\n  11.973058637083994,\n  76.39663993696156,\n  0.0,\n  0.08702864148607564,\n  1.1821164973366252,\n  0.8724298837405502,\n  2.5514492274239564,\n  0.0017205214182889083,\n  0.16190988804720569,\n  0.25605878934248116,\n  0.68351147883636,\n  0.3893814760692732\n ],\n "scaler_scale": [\n  0.06573172622150693,\n  0.5927820155049183,\n  0.05241503976647709,\n  2.8145371993142985,\n  17.305919303926245,\n  1.0,\n  0.1099168813363056,\n  0.32656962006669327,\n  0.1522239800671506,\n  0.18606471574603514,\n  0.006109157949884285,\n  0.260652723570789,\n  0.3882022333416513,\n  0.21482536730344057,\n  0.320431901563923\n ],\n "coef": [\n  -0.6467277303355301,\n  -1.1594121430017297,\n  -1.5026317513104093,\n  0.5494552503565914,\n  -0.40538266345909857,\n  0.0,\n  -0.3034850461038481,\n  1.3338496529228427,\n  -1.1866426406072825,\n  -0.47692804666276484,\n  -0.21289836142501706,\n  0.18851247008068142,\n  0.3137492572878113,\n  -0.3123621816856542,\n  0.13817252337666516\n ],\n "intercept": -2.0975653428294114,\n "amber_threshold": 0.2584530234508664,\n "red_threshold": 0.782317952100413,\n "reason_p90": {\n  "rr_cv": 0.18281209119133132,\n  "rr_max_ratio": 1.3863319386331938,\n  "rr_min_ratio": 0.670401077362063,\n  "einthoven": 0.3154104700270292,\n  "goldberger": 0.5665886410231794,\n  "xlead_count_absdiff": 0.7272727272727273,\n  "n_unreadable_leads": 0.0\n },\n "detector_spread_note_bpm": 15.0,\n "trained_on": "simulated new 100 (1,262 images with a reading, 113 wrong)",\n "rules": "docs/plan/analysis_plan_changes.md 2026-11-10",\n "oof_light_table": {\n  "AMBER": {\n   "readings": 219,\n   "wrong": 31\n  },\n  "GREEN": {\n   "readings": 903,\n   "wrong": 12\n  },\n  "RED": {\n   "readings": 140,\n   "wrong": 70\n  }\n },\n "guard_features": [\n  "ii_complete",\n  "n_unreadable_leads",\n  "mean_other_lead_complete",\n  "n_beats",\n  "hr_ii",\n  "hr_implausible",\n  "rr_cv",\n  "rr_max_ratio",\n  "rr_min_ratio",\n  "hf_share",\n  "einthoven",\n  "goldberger",\n  "xlead_count_match",\n  "xlead_count_absdiff"\n ],\n "train_min": {\n  "ii_complete": 0.2484,\n  "n_unreadable_leads": 0.0,\n  "mean_other_lead_complete": 0.6003636363636363,\n  "n_beats": 4.0,\n  "hr_ii": 39.216155613948736,\n  "hr_implausible": 0.0,\n  "rr_cv": 0.0021822168542874,\n  "rr_max_ratio": 1.0016051364365972,\n  "rr_min_ratio": 0.1995565410199556,\n  "hf_share": 1.695686793477554e-05,\n  "einthoven": 0.010235190439559,\n  "goldberger": 0.0096872426562925,\n  "xlead_count_match": 0.0,\n  "xlead_count_absdiff": 0.0\n },\n "train_max": {\n  "ii_complete": 1.0,\n  "n_unreadable_leads": 4.0,\n  "mean_other_lead_complete": 1.0,\n  "n_beats": 22.0,\n  "hr_ii": 142.87236117905448,\n  "hr_implausible": 0.0,\n  "rr_cv": 0.7459213384601792,\n  "rr_max_ratio": 4.947460595446586,\n  "rr_min_ratio": 0.998389694041868,\n  "hf_share": 0.0823617341352785,\n  "einthoven": 2.6189649315492365,\n  "goldberger": 2.542778922147121,\n  "xlead_count_match": 1.0,\n  "xlead_count_absdiff": 2.333333333333333\n }\n}')
sys.path.insert(0, 'safeguard')
import ecg_traffic_light as T
print('Tool loaded. Thresholds: AMBER from score', round(T.load_model()['amber_threshold'], 3), '| RED from score', round(T.load_model()['red_threshold'], 3))


## 1. Six real examples (our own phone photos of ECG pages on a laptop screen)
*Chosen by a fixed rule, not by how nice they look: the first two GREEN straight photos, the first AMBER photo, the two RED photos with the largest heart-rate error, and the first RED photo with no usable heart rate.*


In [ ]:
EXAMPLES = {'IMG_20261110_132617': '...', 'IMG_20261110_132655': '...', 'IMG_20261110_132704': '...', 'IMG_20261110_133206': '...', 'IMG_20261110_132921': '...', 'IMG_20261110_132723': 'UEsDBC0AAAAIAAAAIQA9SwZY//////////8FABQASS5ucHkBABAAoE4AAAAAAACPBgAAAAAAAO2YeWxVRRTGD2VpBaSUVQShKOWBVCmLIggE2QRKEalo1ERTpShGaWkBWWURNRqjUeISE6JGY1yixriAyCqyCSiCbAJlKSBlsZSW0tKW4zdz77y7vPtaQPzvvF9K2nfnzpw5c5ZvWDhqbNroB2vRVJoZGpeZ+3hOqE9iqO+E7qHkxND4rJzJORkTH83KGZepvh+a8XRuJr7PfTIjOxN/d+qWgk9y5+TE2YlX+Km/jteCX8Aa/hmsBqt4JVgBlvMy8BNYyj+CJTaLNT9cIovDqHfVPEuBmlXNvlyvtBKrrsLayoY1sEZZtQ6s5w1gI/8KNvFmsAX8Bn4HW/kPsA1s5z/BDrCTd4HdYA/4C+wF+8B+kAcOaA6CQ+CwJj/MEZujNsc0f9scD1Pg40QgJ12cuizcb5rZzFrGBmPVMQ9HAzlSA/lXlcNXgUNXyMH/kQM+vE9rsiz6XqN7sqZzu1pERsylRL4/zk30nvbxT7UU+jhj43zjjPXO688Xd56YDHGyIjLS/SdgPGH2XqD3dkrbeIaL+CyX8Dk+z2V8gSu4ii8yUS2qTXWoLtWjWBBH11B9agAa0rX4LQ5PYoioCuPL8G4x5jmNOY9jlXzEzH5UqN2oW9tQzzaj1q1FBVyB2riEv+dv+Cv+gj/jT/gj/oAX8Xv8Dr/Fb/Lr/Cq/wi/xCzyf5/BsnsHTeArnghwwCWSDLDARP9n4OwfPJvNUfhYjp2P8LLw3jxfwi/wy5nqN3+CF/Da/ixUW8fv8IX+MNT/lz/lL/hpWfMvfoY4vQdVepmv1atTodbB1E+rwVtTendjBXlRWf0V1e9PEkImayKrojhQnDoq018/CbyXa96VA+b8cqDOoBOoc1EedRYx9Gs5JqHOIp8aUQE2oKTWj5tSSrqNWdD21phuoLSVSe7qROlAShagT3UxdKJluoVupK6VQN+pBPek2up16UW/qQ32pH/WnAXQXDaLBNISG0t00jIbTCEqlkZRGo+geGk33gjGUTveBsXQ/eCAq6ulYoMam460x+u3RmGcUSMOsqZh9BNYYhtWGYNWBWH0ArOgHa/rAql6wryd1h7VdYXcX7KATdcRuOmBX7bG7tthlG+y2FXbdklpg/83gh6bwRgK8Ek+N4J+G8JOK1VhEcV34rzb8GANvqg/Du1WgEt6ugM8vaN+XgfOgVHNOU2JT7OFsIEVROOMjqAZcHv7q4q0xJsZqwm9XEF5L/TUwsmZF6/funu/t/JHd39vzg/p5UEf9Lz0vL6yirJ5nZbuV6wXYfSkiJQax1AiR1hoRGEI+pSBGeyNiByKGRyC20xH5D9EjlEHj6Al6ip6hbMqFCp5OM8Esmk3P0RyaS/NoPj1PC/DvXHwzi2bQNJqCkZMoC+9MoPH0GGZ5GLOlI2dGIkcGIyt6Ig+SEPXNEdl16SIsKtQV9wjs3QdVuAOqcQtU5fpANtiac6OtO5Xy3GSrT0d/GgW6NaxAt4cVqKVBLRVq6VBLiTrsjcI+F/tt8lyq9YBHafjVg7+bH/P08Mj+7e7eTg32Z01QppgsLq6W4GyvKXsKa8z56Dnlzx+3CjD5UZ1OvTxFWf1Yf94EEzyjX5VEdtETtjI5pU+s0NYnReFTcXdLizJNud05DRU2lYFURXDRhdUVKu0ZTGcw3cF0CNMlTIdwYsMdD5GV06/rIlWdOc/Dtuet2qRyRuWPyieVayr7dnvY5SLopqiUTD5mV/rvJCwphJXF2E0V6losemUCumhrdNYk6oxKk4J60wuduD/68hBUoFS7c6fp7j1Sd/BU3cWHQy0MRYUaFO7gd+K9O1Abe0BpWN27M+ql6tzt0LNbYZ2m6NFKR8aiKysVWQ4vFsGmAth3CDvdE8ZbWdw1xFtBguKu+vtL9H6Q5yF6bEfegbw162hEhB93VShHKZr6ZLRhdcqwIkAZWtrQ0epKIyqVaBS70oqKRpr4arBGWKMbahpo1FxqzjitpmK1oqqnVZVFHU3tsMIyGot9eVUVzi5vhl3wZJg3u4I0mLsCB6mUoKoaXZV4/z9DnYVzKyrVvldeV7ci5W1Lhde3vWr5U+lwpcSbaA0aTBNNgtanjbVGjfd523g6Dp6to/3IrG5Y5bY3nIrn1Lugu4I3JuqFY8LcG6wocN8dWgDn/tAGKkPdIdrZt4ib9D2io4uQD+dJkgvIG0EQBEEQBEEQBEEQBEEQBEEQBEEQBEEQBEEQBEEQBEEQBEEQBEEQBEEQBEGokX8BUEsDBC0AAAAIAAAAIQCbB/iM//////////8GABQASUkubnB5AQAQAKBOAAAAAAAA7AEAAAAAAADt1k1v00AQgOHlSPgoFFoKpanTJLWdmDZJ4YI4cwNx4cAJRTQIpIqipOKCkPgP/GHGk9jxxl67aYgqpNfPJd7Yu97xzMh/3r5/8+7DDfPD/PRPR5NPY/+l57/6OvAjz/98Pr4YD799PB+fjuLx18OzyUjGJ1+G30dyHvR7ckRh5P3yrnjUzG8AAABcl6ZpmpZoq0Pli+mveKylmv9Mq0TbkjyLbwIVio7qqkg9W1nk0C3UmQlVUMFf0mGqXcAVqXbmvvVbdk+rqYpwXlihU6L4nbst5szq2Zh15HCc07P01UCdqOcFXqzF4ionqUGlvoO9u/muszFx1bVdt+FC3RbXnl1/V60JV24GOnvcUxvGM/umbvbMrngiHosd8Uhsiy3zUDwwm+a+uGc2xF1xR9wWt0RN3FQ1PY/H4//j6zbkrk2ZYUvm25H5d2WtuqzaMAfai+PdBVoT3TR/j9KMsuOfRLy6c7p6Z76PZt+G+00Uxbms5i9b1+ur3PVw9YOyzlCmV9g33JVnX5XtL5e1TPUXrZ3NwyQTo0x0jq3cdfWM6V1JFiY5M82I+UzFcRnM5rXnyuf8PPeymbnYd+xuM//WOlCNlFfKvmp/pq72cp7OXPe3JwAAAAAAAAAAAAAAAAAAAAAAAAAAAP4PfwFQSwMELQAAAAgAAAAhAMHzPen//////////wcAFABJSUkubnB5AQAQAKBOAAAAAAAAPgEAAAAAAADt2D1Pg0AYwHHQUhM/BRuadFBH4+ymcXFwMsRiNDHWgHExJn4Hv6WtImp9r5U7DrkgtNi0MTX/57fAvfHcccvd1eb2xtaOaZwZ507TC/Z8Z9V21g5XnIbt7Lf8U9893m35TU+Ur7tHgReXBwfuiRe/LywvxdFYbNgX9ogxf98vFsXupDB2K92USGpDKekT5fqnI4xfWU7j0pHayvXUaOd0FP2P/U44Mn1PRGp/PQzwWKCrPEnPA71MVP5rSUYityTT4TPT52FcAgAAAAD+B9MwDRH9OD6lniKeRYha0WqmslmlJllS/Ye5odKWlqKPZ1UYp14yQk3ll+ZrGukqJOuQrUWmV+hD8668Sa+aqmf1SUhP/93v0396f5bdgeVvwfS7pcRf71IAAAAAAAAAAAAAAAAAAAAAAAAAAABMhy9QSwMELQAAAAgAAAAhAMJlACH//////////wcAFABhVlIubnB5AQAQAKBOAAAAAAAApQUAAAAAAADtmFtPXFUUx9dMoTAM96FDuZXhUgaGe4GW0upj3zS++OCTIRajibEGjC/GxO/gFzPRxERjNDGRXoS2otzLdu219vWcOYBNjC/r/F6YM3P23met9f/vtfnm7Xffeue9HHwBX1Yfbmx9sFldq1Tvf3yrOlOpfvho8/PN9U/ff7T5cEPff7D+ydYG3t/6aP2zDfw8ubiA18zUTOWrymteLfC1IAiCIAiCIAiCIAiCIAj/NTfh3zGRQTUB37VPjcMYjEIFbsAQDEAfXIcyXIMSdEEndEAbtEILFKAZrkIjXIEGpBH/bsI7BfymiN9bikQLUUCK0I6j9EAv9OPYFZznJkxCDWZgDhZgEZZgBe7AKqzBfeQesUbcRVaJO8ht/N0KLCNLyC3DomHBME/MEbOGmYjZjPuaaUctYCpgsi7J2Caj7CM9TowRo8gIRqQCwxh3HfkhGMTo9yM6A72IzkIP5qEE3ZQLnY12zEcbRZpz0kRZacScXIE85ABAqTPklTpVJ+oYOVKHyIFhn/g7g78Q/2nfoZ/UoxwheswT5BR5RXPpC3DuHK4g7ypE14iuEl0nzVQNXBdFVzFtmbSmKEZ1Fv623dHh6CS6kG5Cx7CHuEaUCR3j65n0mhww/FxPkA+bEc5Ja5STMCucF84M54bzw5wa+NOZi6eOqI9pnsY5P671iGNtY9RJcSmZaJQpCn1YdwNYgUNYi8NYlSNYn2NYrewquponSQM1oxGrJlbbfMDCucSK9arNhtVsFVozStSaGiMFsWv1U8bKlJ9upxTOSJNxLp0HRbE/wXreU8/VM7WtflO/qJ/UD+o79W0G3+O3P6qf1a/42231VO2oF/j0AY5ypvI4dgFn0i53HdcxhNEbIZ+r4jqnceXz+L63yOluo5PdRW+7h173ZgZvINYLYx9cDfwwZvU1sc/fNqw4l112XrsUOG6YuzA3s1F+ai5Hk4EP2n1mLOV7seuVnc5YaSWjYFacVV2n0x7rr934Ylu0D/m9KK0I1oP1iq7AJawm2Bv6aG0DRhv1GQwYcPQ7+gw8Xl805uAlxw3HDkcNvSp2rNCzsmKZjmb9SPodvUCuww5nPY5dzpJP4L9pcDQmuJqiKUVzRMFkM5nLriCP5SCHNn/W3UZoBx6jHdl2Tuxy1udqkc95r5u70LHS/lWfpG7SzjZI/Rj7Whf1YkXyM+1mekfR0dXRa6ZeS0fB73phNZdNR8d9RLvpIHz/EHcPObzXgr8q4cwDuI5xXNWMeXN2cO1n2iFWEiwnWEp5SHbfFndu9f0ku/eaSPW0vtfibmuUss5UHMOGG64LG3Jq6zcK867kd/64E2txurhq4tlglOD3fe7IuCc7pk6KO7PDqDPbDzqx84g7tAPHoePIdWvcr50EnUayz+BOw5Ot4VDFaTV75drexGOVa/Vs/aPRjBK6R850THyp4Ar7Jx9P3+P6GHKc9pA/1UvkBe73u7h3/4E8w138qXqCPCa2id/rsB3w2PGE0GM8Q/SIOzj2Ls7wHOd5iTPuUY72KSNHlAHulrlT5i45eYpiFwv3ox7jY+F+lNw94h2E65hrmit8xLkdO55nPMHlz5n+GXtyDJ2qO+iPO6MdurWOXprIu1rp3fV76zfuJb9rx/vNGKU8nGH8DjGeBxjJU4xhDuOnXU97Xgc9U3bd1yiuasL4w1RELYPpC0meE/3JsL4LxWe+8MTn3SfLdwYz+4e+1MkkPpWUgr09PpsUo4gn/d6eFdmRWEWsIFaP1c8Lqu9dUtEOqSipo8eRXrYjBcXqscrZSahmz3jbvjtxHgdnTeX0kw+8qF4H4T0p7VchWX1LLoPLX+rc6+wCXkX4E2J4Tsyi/lV/XfGak+/p16pnBVJdAeupI/ivzv/9PzJBEARBEARBEARBEARBEARBEARBEARBEARBEARBEC7HP1BLAwQtAAAACAAAACEAollM/P//////////BwAUAGFWTC5ucHkBABAAoE4AAAAAAAAJBQAAAAAAAO2YW2icRRTHT+5pbttNs9rcNruby+aySTYxTb0U8doHoVIQhT6IhjZFRVpJpA+KIj4IPvggIggiCIIgCIIIIogIIkIRBEEEQRAEEUREEU0v8e/Z+Wa+mW929hJb9eXs7ym73zeXc+b8z3/y0t33Hjt+ooXO0ZPFU5vbJ7eKN+aKRx5eLZZyxdNntx7f2jjzwNmtU5uV749uPLq9yd9vP7Tx2Cb/PbtS5k9prpR7KvcPPz30jCAIgiAIgiAIgiAIgiAI/zZjZBkNMhIzrDnoYb6Pnqq8M0bjTJYmmBzlmQJN0hRN0wwVaZbmaJ4WqESLtETLVKYVWqXraI0O0TodrsE6/7rGT63y02V+a4nfLvEoCzzWPI84yxSZGWaa55riGSd53srsObWSrKKyMrvfenu7NsA1DfCfD8XJZ8QhlIGxBozXIBtgQpOLyWsKmknNVIJpjxmPYpDZq44/g7sGs7apOPuNKTTFZJNRCeFHKky9yM3FzMcseJQasrgHqt8O19mMs08bFRPXqPJycfWZyhuNK+6grqkMDTEHaJBJ034mRQNMP9PH9FIPs4+6qYs6qYPaqY1aqYWIUOPDP/HvFVqZNkU708Hvd/IoXTzWPqZHj2zproN5o5eprKuyvso6U0xl1Wm1gwNMZT8ZrQhR7dvqDlWuVSM3Ln5k0ioyJja1IhOKzV9NsKu5rLikuai4oNiJ+XMP7AS4EHMxwaUqLmuitdnVupmOPi1Oxk3O21TOO1TOTf6SuXOz5+cv42i6q+EjVdTS9nDHrO4nGc2QznfEYJz3tK6L/foEpOJzYOnXDDj7Mnsz+0vrMc0MQzGZmGQn8/dtT7DtMW5P8fuJ31F8XbyyjuH3geoOYJQoHyvRRMIHRHUX1ZypOKtEqViFerVOdKta61I60qEqztRcqz5/lKg9v77c02vPrT2z7fGJ7Yy1KtIrq0FWuYwa1cbqm1m5O5odsUfVxIA6L2l1MjJavYZV1iM/l9Nezrg54+csycxYL1bQPiyr1c7G3EQ8HWt/Uv37tNLZ/fTUyEWUjTAdDp1V0a0X4WQ8+zz6AwxUkfJwa9LWo1uHtatw2POKyV4S9nuu58vXqVHX44QdTNjphWnO+zTriZphr/oxVxPfaSV91JJiWVPWrChWG7Ki7jBldY9ZVneZJXWfsd5rIXZckevyKyupddbJhnXOVTrrMVyXkYmdRnXfSSc6h4v/hI/tOlfCYFVXtsoU3TVDylSMb5rzOnuLOmc2U+VEFuydMvK5Rr1MD8lp/co68RxN9H8/piaioT6eSuhcf1DjQvj6HlIiqzh+36/f+0P32NDt1b+vJv1ss/oTVp/wTSt842pWSZJv+e6g4HgE6xSqb8j5oHL6qw+ttrGemZPm90jTJZO3pMiZWFcS7oJt/zmtGuuDrBdyPZD187sJP7+r7hs7+AO/4zf8il/wM37Cj/gB3+M7fItv8DW+wpf4Ap/jPD7Dp/gEH+MjfIgP8D7ew7t4B2/jLbyJN/A6XsOreAUv40W8gOfxHJ7F03gC57CFM3gEp3ESD+J+nMB9uAfHcQx34Shux624GTfhehzCKspYxDyKmMYUJpFHDhPIYhyjGMEwM8KMYoy/yTIT/HseBX5yGjP81hy/u4ASlrCMFR5tDes4jBt49CM8xy0802083x24k+f9v/8PKAiCIAiCIAiCIAiCIAiCIAiCIAiCIAiCIAiCIAhXg78BUEsDBC0AAAAIAAAAIQAJoGYC//////////8HABQAYVZGLm5weQEAEACgTgAAAAAAALgCAAAAAAAA7djLTxNBHAfw4lsEFRT12Fs14SAejWdvGi8ePJlGajQxYorxYkz8H/yLhBZobYGWtwgILSCI+ADfj3E6Ozs7OzvTbbctxPidz2W3nc7M7zczu5M+u3r9yrUbTaFHoceR7ljvrXjkYjhy6e6FSGc4crsn/jAevX+zJ94dK31+OXqvN0Y/770TfRCj92e7ztPSea4z/CQcsDSHngIAAAAAAAAAAECjbZC3RuuSNeoNs8qsUMtUkSowS9Qi9dpgkVviClyRW+ZWmFXO6nNNsq5QR72heOey6fKe2dSwatttyBlZ0+ZiRWTEVvRRKGNJY1Ghq+Mmt6j2vlw1v3jKR+Qfn1+8Dms9LTDz1BzzipolL6kZappMUZNkghonYyRPcmSUGiHDJEsy1AsqTVLUEDVIDTBJLsH1i6uE+C7J65YMugx5pMSVXWfA2JOtT/Jco0+jXyuhaV3Xi76fWviPMaGV9OSlXB2dAcVgAGobOt55TzFprrTCMkyWGeZGJKMGOeV+pAJqCxZd6/Yv7BFlhYzEGb+OLia55zwzphhnrOs825XuMVYSp3/2vLG6I81IT4A0n7Uhzz417euUMs/WTKvZyiqGtfyiyknygjejjomy1Noyda5qVX5kk8KUMM3NMKVn+SxTerbPcfPcAmefLZx3nvN+c84SzjlCPjXY5wP7LPCB+kg+UVuSbZfPBtt1sLUD6jHOnWDKs9uXgCprvV7z2ojIq4v9q8E3j++KHx4/tX5V4HfD/BFMJWQsTYo9wl5mH7OfOcAdZA4xh42afRzx0SK0CkcDOPaPCRKjzc5Ti0arNo+tWrWMobb8HzdoE9pdTtTVSaGDO7VDOv4bQTN0elcFmz17NVmry71y2wJpr4q712r7MO0N3Q5xMnWmarv9PyAAAAAAAAAAAAAAAAAAAAAAAABAPfwFUEsDBC0AAAAIAAAAIQBBWqZL//////////8GABQAVjEubnB5AQAQAKBOAAAAAAAAXAMAAAAAAADt10lPU1EUB/CDs8Rx4ca4YGFSTFg4xI26M3FhoiEmLnShFoFIJKBgSAxqdOnaxC/gJ3AeMYqiyCBOKA6ooExCGSylULHH8+5r+4bevpYh0cT/+8WQvr53b++595x7vbB77678fVlURdW+wqLKwxW+LTm+bSUbfXk5vuLyihMV/rKD5RWFRcb9Hf7SyiK5X3nEf6xIPuduWC9X3rq8nNM507yy6SwAAAAAAAAAAAAAAAAAAAAAAAAAAADAvyTCE2JcCYsxJaSMKkHlpzKiDCtDYjAhYDPg0u/hh4e+hF6NHo1uD12z4HvMN6VT6VC+anyJ+ZzQ7sF8Iv6O2YLZttmP2afZv/WLrPHZ42BFqU/DK+bpWDM3oObamHtjJRgrwlob8fVhrpGAWgN9as66VfQ6ZWzt/IHf8mt+wc38jJ/wI37ANXyP7/IdvsU3+QZf56t8hS/Lv2vy6bZ8UyPP1PJjfipvNHITP+cWef+VtPJG2nrH76XNT9KyET8jYkZ8em1rSbd2dLFKNer4ig44DM5QwDN7MuWVZV7ZplshqbMuPV1epuKVr9PN3szbdNOvCr3Zy6jZNbX5Sz0Gr9H1J+WDOyNmlh9DHjLNpOSsyCRKupWZag/47qrF7lXkjLAzdvaoxevniNphg7LjGnuvsQuH1a5s7M8R/iUmxW8RFcZFiUt9VPej6gm76LSxrZesJNYVld8VUeeHkPz+ERlNQNX7HolOp9Rio9K3cauq9S1S7Zu4Qep3vdT8Oqn6tfxQ1fQ6+Vwv9xvk+2ZV2V/GKnurVPY2Vds/qtreIZXdrOsDEr1hW8zCKmZWxHQmk7gjppf8nlNyTxElff/erXiLpDGhMe4SdhhzCDnOgvHTYPw8mJ55Ykz9fdBmNKmXTPsKaumeGY2tlZBrnKmEMuB8I5ySFXH7bExl7uzzlf6kbj+lWzXRXQ/1e7/3+c9e8501ezhxErTq2WiinoUd43fmhzMbdTXIXYXmaM3VmJeB+bSAFtIiWkzZtISW0jJaTitsViasojW0lvJoE22l7bST9tB+8lMxlVApldFxqqQqOknVdIbOyd+jdIDy5bnNlEurpV2WqHRJPWvk+3K2vcQX+Tyf4nIu4EPsdyhIsN/92/9jBAAAAAAAAAAA+B/8AVBLAwQtAAAACAAAACEALPTVYf//////////BgAUAFYyLm5weQEAEACgTgAAAAAAACsHAAAAAAAA7dh5VBdVFAfw5woI5p4oS2QqmGjglqan1NTcM9M/+icjwbJMDMwWs7QstSzNLZfMFnNLM5eQVNxBSBFFRATEBZc0cs0lt9v33Xm/+c38Fuzo6Xg65/E5mv2Y38ybd9+98+5M7d2/V5/nyogRYmR4bFziwITwx8LC2w1uFh4ZFj4oPmF4QszQAfEJsXHy8y4xQxLj8HniyzHD4vD/EdFR+IlsFBk2KuwOfyqJ0ZqmaZqmaZqmaZqmaZqmaZqmaZqmaZqmaZqmaZr2/9eVulF36kE9qRf0pqehDzxDfeFZ6mfR36IfftcXR/XB8b3xzZ44R3ecqys9RV2oM3WiJ6kjdaD2+OR5SqTx9DWtonQqoovkJ0JFuGggHhIPigdEiAgSdURtUUvUUGqyWsr9Lmp75HrU7RjnrsnXqy6qQVVRBe5jlZUAxV9UEr7CR1QU5UU5UVYIQXSLbtINuA7X2N9wFa6wy+wS/KVcZBeU83flwm1dLFXp3/V21XNenXVxhv3JSuAPdtrmlM3vFifZCXacHWPFLo66OOLB4f+EpysZXMfkPi7nWQ4pRewgFEIB5MMByKP9kEv7IIf2Qjbsgd2QBbsgE3bCDvgNMiAdtkMqbaOttIU20ybaSBsohdbBWvoVkmkNJNEvsFpZ5cFKZYXy8x1YbvETW8aWwo+wRJH/lp/J38mjluO7K3Dt1RjjGox4LUafgvvYgP8m47OVOGIpLaYF9D3No7n0Fc2h2TSLZtKXNIOm0zSaSlNoMk2iz2gifUITaBx9TGPpQxpN79FIeoveoASKp9foFXqJYmkgvMhi4AU2AH9i8HksxdEgHDUYx75KQ/CdoRAPw+h1SEClS6ThOOMIepPepndw/ndpFK7zPq72Aa45Ftceh2o4gT7FaD6jzzGuSRjdFxjjFIx0GkY8HeOegdFLM2EWzIY5uLe5qKPz6Bv6Fr7DHc+nH3DnC2kRZmAJz9wyNWfGrK3ieUvimZMRX8vzt47WQ4qax42wiW2GLbAVtkEqpIFcR3I9ZWBt7eCVlolVl4UVKFdiNlZlDtZnLlZqHtZsPtZvIdZyEa/tw5wTxcje48jmk8hume2nUQ1KuDqc4XphVBFnZSutXp03K46jxjgqjFFVHBXEUT2OmbXCyL5DZq4VWLJM5pfMLJlRMpMyOH/k3TszyJlD61UOJdvyx5ErjhxxX/MOy80YreA4reTvrlbRSuK8tEbMiJcRLSNS9jg5omTEKZ0rQAbXAk92eGFENpOju4uri4xwLmbqCOayBDN/BU+8MngSBuB5WQvP7VBRD8/ySBElWoo2op14XDwhOoiOopPoLLqIrqKb6IG/24vWOCJChIlAfM8Xz8+rOFcJznkYEdiH62TgTlJwxyuwiudjlc9AVoynMcig4cgyI9OGqQxzzS6ZW2OQXWPpI2TXeOT4ROTVZM6n6cihWZw5Mndk9sj8kRlk5JDMIplHMpPmczYZFlgstFlks5jzzlG37JXLHuWVKhudEU7mCHvLSNecdEbbU15mWOJqj6IjjrtVthoZm81Za8jxaK8H2ebTZ486Y5bXZ1DpvK1Ab9zPkGGRrmxX0pRUF9vcbPViyz3ibTylc78vK+f9p5mstcK9WrhXhZ2WVWVdW84VluW2zrJt68xYVftYLtvP8tgBlq8UsELlIDP2SI4dk+suzLnXcu4Oj5mcO0j7LtJ9h2bfmRVZdmYH1WgKzP1Zvhr1v+W8N+t9We/O3aE75G3X6vokls/hs+hNrhFReeGH3qcGanQIqnoD1OvGognqdnPU9taiLWp7e/xpizrfCp9Fi6ao+41EQ1EfdT0Y36qJ2h6A6l4e9f0GeqFLeFafRZU/hasV4+qFmINcrIddWGFpWOsbUPmSUBOXW3Yt7nsW4+mXwvVwM2dIKq9cuUozecVlm7sP697jsFoVxWoFnHDZfRj7jzO8/zhn6arkbkN2a7Jzu2xxxeIqk/2e0fldZzcsbiq3mPEj+KcMlEUXWQ7zVAEdpQ9mzA/9pT/mrjLmvwpmsRp60hrcA8t+NxDP2brokoMRl1B0zGHom+uhe66PGDXE0zcCUXgYsYqEJtBUPAJRiFC0aAbNWQvWkrVSHmWtlTbM+LfxG+OolkoLpbnSjEUrUUxeuSmPIhIjaoxxNcL4IjDKhhhtfYy6HkYfhrsIhRDcUxDurQ4EchdvdOU11ZuA6mZvXtXSnzt7c3/uzCU/8GU+SkWLCkpFNz48/74cAz91Ln+ORQBfxXgfUIWjUpUjU42jU50j5Hhb4XhPYbyLCGR1bOqagkzBphCbUFOIm2CesSCetbrq3IHmGxBv7zQqm28x5CzJ+bC/x7jOby/km4tL/J7igq2nt/by9r32abNzd+3YnZX2qGXnbe157b2v5+7XVV4pjC45x+yQd/O+xPFMcj6RvPXQd+9ev8vTNE3TNE3TNE3TNE3TNK10/wBQSwMELQAAAAgAAAAhAGtjahD//////////wYAFABWMy5ucHkBABAAoE4AAAAAAAA7AgAAAAAAAO3YSU/CQBQH8Lojbqgorgm3asJBPRrP3jRePHgyRDGaGDVgvBgTv4MfVHHfsE5pGWZgOp2ZFtH47+9EoTOd5fW9crO5vbG102FdWJf2fqG0V7RXs/ba0Yqdy9oHp8XzYv5k97S4X3DPr+ePSwVyvnSYPyuQzwvLS+TILeayV1nDI2ldAwAAAAAAAAAAAAAAAAAAAAAAAPx9U9ZvM61lhpoNNcPQ6yUOPzmHGWqywURkaQXjEmPUKCelpX7dmBLZHblURiXHzlHjrIfLaDCJJJ044SMlnLgfWW+yFlpPfTTN8VuPKbPIiLKrxLtLdV/wT6D4d1u7mK8+uyPnOPOxmjOgEqVyJr0G340XGd7OiZJHVDJG8Pe1rOHlgBFqmDFEDVIDVNLXTyWoPqrX1+PrNtJjKKi9rgCdAh2WZTnOl1NxPp0P5514c14ZbxLvHNkvPfVWX6hnzhPjkXqg7n1l3x3nVgF/RbnJPafc1FNQu+6VD+Sun8mYXslsfJD5rFRnVazi+2KIDrI8ZI1c7mq5qyjeN72cPkaC0c9JcgY4gwwvSmpx40VSrbaqV006makxOzVnEbUcrJefdTOFeiUU5zM0vie7en0W/2hm/TyQqeaANFfPh9fdUeqeOConeSUVt+jvWkHjD3ub0aX+9qQ3v61iUkmHz4JJq7K9xz4D210lt66elpnyK0b+v4if2CG1ntLVijFVrQvd6i/BVHdBmVVMdJW4FT4nJwOxebnd/wMCAAAAAAAAAAD8B99QSwMELQAAAAgAAAAhAPeXROf//////////wYAFABWNC5ucHkBABAAoE4AAAAAAABLCAAAAAAAAO2Ye3BWxRnGn5gAuUAiIQkJJJCUy0dIIBCUi4BUoVSroGA7LSLaYIJAI2BCrQxlrI5YW7W17dDWmTqt7bQdpuPU28g4dWy1UqXeClMoAnJpoBBB7oQQCG+f8+6e25cvsdO/9/tNhnC+c87uvrv7O8/JT+Z+Zc6tC9JwP9Ym6hua725KXFOemLasNlFdnliysml1U92Ku1Y21Td4x2fXNTY38Hjz0rpVDfz/qPHj+KmurC5fV/5/frLxHYfD4XA4HA6Hw+FwOBwOh8PhcDgcDofD4XA4HA6Hw+FwOBwOh8PhcDgcDofD4XA4HA6H47Opk69b7rLcqSwi5jfvmDmjjiwmd0u9NJAlcg9ZKstkuXxDGuVeWSErZZXcJ03SLKvlm/xpJk08soqs5PcreFYjz17Oq5by6iW8Tz3vuFjqtA+m7TtkodwuC/izkL8v0h54bdfbVk2b8RabtC2f1Zb75QFZI2tlnTwoD8nD8og8Ko/JE/KUbJCn5Rl5Vn4rG+U5eVFekdfkr7JFPpTtskf+LUfkuJyTDukUIAN9kI1+uBIDUIQSlJIyMoQMJeWkwlIew/vWO6uMVwwmg3h1CYoxkPcpRAHy0Z93zUMu757DNrKQybZ6scV0XAFA5DJ7cEkukg65IO1ynrSxZ+fkrJwhp8kpOUlOkOPKp8oxcrQHjgV8avGu9e7i3e2U3tlr4ay2do6ttmnr7coFS0fARcslpZNcVrwPh4I0cgVJ5+gyOMZe6M2x9uGIszjyHNKXVcgleaxJf5LPeheQQlariDUrJl79BmktSxVTVVPXEj1joFbXq2+h1riAd/HI12p79b5Sa56nbXmVN/RVcpRsnQtvNjK1j71JLyVDSbd44/HGZT7eSC/rjHVqDcysdWid2u3cndc6mvkzM2jm8LSdScNJy4kusxrl2P9A8jXHdX5PRea2jb1M55gHsJLDUIUJmIqZuAnz8VXcgcW4B/eiCd/COjyMx/B9PE6ewJP4AX6Ip8iP8GP+71HqZA3Pa0QD7sTXMI93uB7XoBaV3BUlrHgmK3SerbdKi+yVnbJN3pe35Q3uu1fkBXlJXua/m+RV+ROPvC5/5jdvymae8Xd5j7tyq/xTdvCq3fIxr95H9ssB7tMWOSiH5D/KYeWI0mr5xJJq3aeqk78HTtgZOBXbCWftbmiz++CCrnuz3v2VnqZrPF3XeG9d3/4KN+vbrG5vbRfqqjbruUxd4nlkGIZjBEYigVEYzfmoxhiMRQ3GYTyrOQFXYSImYTKmsLpTMQ3TcS1m4PO4jvWeiVn4Ambji7gBN+JLnIObMYfMJbeQW8k8Mt8yT/GO3qJ4583hNTfx2hvJDbzTbN5xFpnJ+1/Hdmawvelsdyrbn8J+TGJ/rma/atm/ceznWDKGva5m30eTSo5jFEczkozg2IZzhJ8jFRFHGksaUxpKIwzugUERSlIwKEZ4XXj3siSGBAwNiFu9IoY3lmE6a968jdBxJnT2Kok3g2YOx2hlanQmzVx6s9k9tVrR8Xp2TVBXv7J+bf3q+vUdofX1K1yR9BQq6+LMqDGLYsbMD4wZdWZe4MzQl9mBL40xjTONNbv3pk9ajPhHt5R9jnQGXg25GCGN7eay/6UcfSVrNomrdBbX8FzcRh8tUpM1qsm+jfV018/xa/wBL+N1vI2t2IUDZD/2kb34mOzBbrITH2AzXsNL2Ihf4We03nqsxX1Ygtu5X2Zy/Y9hlYs4fqETztAen9BEB2ipj2isf9Bd7/LnAzXYNh7ZLv+ixXbRY3uIcdn+wGWezQ6q0Q5Zqx0OOBKjtQuh6Xyfhc/zZIv5DjsfPMv9J3iqp3dPz+/u3FZgn9jGbqW6r8p1t0T9lrw/aiKe80x3tXWdsZ3nO89409R6nvfizAi4Vh01Xc+cqqbyXWVsdRUJjVVjjVUV7KjubBWmudJgDxUHea5Qd06+TXV+rusX7JKuiSK6J7ri17mX3Ul9LJkxsgKyLTmWvtp6ru7aMFGZTFWg/TW5KpqswmyVnHDLP8N2vu/8GTVzmpqow8KaJ2zNjb/Ciof1DtNzka32gCBDh4kuNynRxdNctPLGP35y67RZ209sftI+Z5/8YWKLZrZUnElB15QXTXp+2gtTh79nT7P9S0xo2RxnMesxkvWbwPV8PZ/R87EAdVhGH63BQ/ges9jTeJamegGv4g1sodc+otVacZQ/h3EILeq4vXTbLpptK8/4Czbhj/gdnsEGprpH6LZGpr6buWeqWPEcXKRPdjODbZF3mMX+RjbLW3xTepP5zEtob/HIO/z2Xaa5D+k747gdarmddKAxnec5P7UZ23m+M7TEOBgQOjCe7XzXHU1pupOxyp2J5LazXd5ikr3XGXlfSbM5LiMpx8VNN8C+nfg7KHTd0G5sF090Udd5tvN8N1EtNbkLU5TJgcUmqslCl0XTwtjAaFU2gSViPqsI9tXgJIcVWIf1j70lhfspJ/LEzw7Mk5mCuJd8I/W1Vupn3/VyrZ/ytL3+1lKhpwotRZaBluLAWWHK69lcxl3GXsO6yWqjA3/5DvPp2WU+VbFnSCJFIos+RZL/KhA1W+i2ePqKey757TWcofg8ZSdlszCX+cks/iTqLp/1lM2ib7zhO2978M7brn9NEea0DLaeyd55f1Ep4MiHsDqjWMlarutpfMeYzXeQL2MhE9tymu0BPIjvMndtYGL7BX6J39BWG/Ecnmd220TPPY/f8+hP+Q66nsmumdcsphXnMvtN4U4YzurmscUO2uEIfdJC8+yjj3bRTjvoqm1ku75X7uU3LTRNK51ygtZo4wgusb/p7G0W65rXzYosSrlvUmXkrG73y38BUEsDBC0AAAAIAAAAIQC+0wNN//////////8GABQAVjUubnB5AQAQAKBOAAAAAAAAbAIAAAAAAADt1FlPU0EYBuCvoCwtWwVZEhMbYzxgGgNoDCG4K8YFRBMvvDKN1GhixLTGG2Pif/A3iYi4byiuuOFaFBER6/jOnK2nSjRows07z1VP55yZ75vznnPd+7t6DoTklJy2epPpQymrPWZ1HG214jHrcF/qZCpx/GBfqjepr3cmjqWTuJ4+kjiRxO/GlmaMeFM8diY2xxGWs0REREREREREREREREREREREREREREREREREREREREREREQ0n7LqX/3w+EMw1G+H/icEBVAIC2AhFEExlEAphCECZVAuFVAJVZ6oY5Gn2lHjWOyoNeqM+v+gLqDWYa+l19V70PvRu6vCniuw/3JUEUFFurISU6euV9et69d90P0IGW7vdEd1f7/DDHyDafgKU/AFJuEzTKhPno+O8RyZgA+ezF8Yx7MmsMYk1pvC2tPYSxbnWIgqwqguirrrZYkslWWyXCxpkriskhZZLWulTdplnWyQTbJFtsl22Sm7pVt6ZJ/slT3SJbtkh3TKVtksG2W9dGB2G+5aI63SjGfEZaU0ygqJSQP6GUGnsthDRo2pUTWihtUddUvdUNfUFTWkBtWA6lfnoV9dgAF1EQbVJRhSlzHnKmZex/ybuOs27r0Lw8Y9uG+MGA/gITyCx8YTeGqMGs/gueeF52WOVwFjxmvHG8db453x3vDPxT45+yz1uQbPQL8NM3gzsiZ3dqoKTJ5ys2Qnyc9RfpLcBPm5yU9MMDUNs5otJ/npcBNS7aTWzbGf7EqHm5sys383O65iT9Gc+Pf7zyz1hB0Rr3d29+z++R2MBjoY/P7U/PL9sfsy3198oj/7CVBLAwQtAAAACAAAACEAs9LEz///////////BgAUAFY2Lm5weQEAEACgTgAAAAAAAKcCAAAAAAAA7dhJb9NAGAZgJ01LG1qgFO65GaQcgCPqmRuICwdOKKJBICGKEsQFIfEf+JEsTZs0ZHOaYCbu7IvHNo4Q0jvPpbXimfEsn7/x1yfPHj99Xgk+Bp/Co3b3ZSd82AgP3zwIm43w1XHnQ6f17sVx56i9uv6o9bbbJte7r1vv2+T/O/fvkdK822x8bhQs9eALAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIDf1cC067VHXTNcz0G9c1XfbtJ+ndgJthNXNFuKTUmN25BUuYpElJiX34qlYqG5yGie8P1Kr32htb7U+raSrwRGqRiqHmLkxHgtk7GYx1E8i6fxJB7HI2rMTagpNUtE3Dy3SDOTTLkJJ/oyUvxKIZ5C9F2uXW+jDHrt62gjzbhkaU830+hzqq8L945Rd4lY7a41vWFRSyHiiBw7RMQQ8cEXB9TVy55qIfVd7jfr16ZCj356H6varpaLHBHkvl/QPRzxFS7Wg9gpw/icGMR94iw+JXqJE8VPhx+S79y33NidrK4s7anYL048epJTizMJu9ZTxsRXg0rcqT6Jq089Z/2sV/3EgFjN3DDhjnhD7pwa8L/+jtzyyJAlioiYUeyNUfQ9XI6s2UK2vui5QXqe4M4azJzAHilEtChe0tqSI7U/GtcsMVGPi272fNGWN9q5MyP5nWDLGMU6mNN8aUbjrJ6V5OPfzdmUs9P9UaAcLJKI+KDnE25mpuHPRCM+Y1NLpuSfl/Sxzjc/A0O/RGbtuuzza8uobZGfRXhXfHfPWDqRYS0yn6LsZySRO15Gny1yEt0mZ9R6clre8zLPyf+Weg6/4eU+vbMntH0lkL8p1KmdAti9Wb9OrIN/jn0zro/2PuEeYflXzE2rA+pW4nYhl/cecPaWstv/7/j3wHr9AVBLAQItAC0AAAAIAAAAIQA9SwZYjwYAAKBOAAAFAAAAAAAAAAAAAACAAQAAAABJLm5weVBLAQItAC0AAAAIAAAAIQCbB/iM7AEAAKBOAAAGAAAAAAAAAAAAAACAAcYGAABJSS5ucHlQSwECLQAtAAAACAAAACEAwfM96T4BAACgTgAABwAAAAAAAAAAAAAAgAHqCAAASUlJLm5weVBLAQItAC0AAAAIAAAAIQDCZQAhpQUAAKBOAAAHAAAAAAAAAAAAAACAAWEKAABhVlIubnB5UEsBAi0ALQAAAAgAAAAhAKJZTPwJBQAAoE4AAAcAAAAAAAAAAAAAAIABPxAAAGFWTC5ucHlQSwECLQAtAAAACAAAACEACaBmArgCAACgTgAABwAAAAAAAAAAAAAAgAGBFQAAYVZGLm5weVBLAQItAC0AAAAIAAAAIQBBWqZLXAMAAKBOAAAGAAAAAAAAAAAAAACAAXIYAABWMS5ucHlQSwECLQAtAAAACAAAACEALPTVYSsHAACgTgAABgAAAAAAAAAAAAAAgAEGHAAAVjIubnB5UEsBAi0ALQAAAAgAAAAhAGtjahA7AgAAoE4AAAYAAAAAAAAAAAAAAIABaSMAAFYzLm5weVBLAQItAC0AAAAIAAAAIQD3l0TnSwgAAKBOAAAGAAAAAAAAAAAAAACAAdwlAABWNC5ucHlQSwECLQAtAAAACAAAACEAvtMDTWwCAACgTgAABgAAAAAAAAAAAAAAgAFfLgAAVjUubnB5UEsBAi0ALQAAAAgAAAAhALPSxM+nAgAAoE4AAAYAAAAAAAAAAAAAAIABAzEAAFY2Lm5weVBLBQYAAAAADAAMAHMCAADiMwAAAAA='}
THUMBS = {'IMG_20261110_132617': '...', 'IMG_20261110_132655': '...', 'IMG_20261110_132704': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAA4KCw0LCQ4NDA0QDw4RFiQXFhQUFiwgIRokNC43NjMuMjI6QVNGOj1OPjIySGJJTlZYXV5dOEVmbWVabFNbXVn/2wBDAQ8QEBYTFioXFypZOzI7WVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVn/wAARCAEdAXwDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDMpaKK5Bi0tJS0AFFJS0AFFFFABTXXchFOpcUgMzoSKcKdOm2U+h5pFFWAopQKcFzQzJGMu6r/ALxxSAAKcBVd9QtI+s6n/dBNQPrVuv3I5H/IU+VvoBpAU4cDJOBWG+uSn/VwIv8AvEmq0+pXc6MjyAIwwQqgZpqm2Btvq1lH/wAtS5/2VJqu/iCIf6uB2/3iBWBilxWns4iNaTX7lv8AVxxJ9QTVaTVb6TrcMo9Fwv8AKqVFUopdAHvLJIcySO5/2mJptJkUZ9qoBaWm5PpRz60AOopuPejAoAduFG4U2igB24+lJub2pM0oUt90E/QZoAMn1pMVKLaY/wDLJgPfinfZn/iaNfq2f5UAQ4FFTi3QfemH/AVJp3lQjvI35CgCvRmrOIh0iz/vMTS7wPupGPov+NAFYc9BmpBDI3SNvyqbzn/vEfTim7mY9Sf1oAQW8ncAfUinCDH3pF/DmlCOeiN+VPEMh/hxQA0RRjq5P0FLtjH8JP1NSC3c9SBThb+rUAOtrbz0dl2qF7HqaLaPzWbJwF7L1oEAX+I/nTzb7BlkIB7kUAMGBIQSWUNjI7ituOXSFBEiRvz8pVW+72z71SsdLnvlZoAu1TjLHGT6CqjoUdkYYZTgikM2KWkFKK5gDFLTJZooADLIkeem44zVR9Ysk/5aFz6KpoSb2AvYpax5PEEY/wBVbu3uzYqrJrt033Eij/DJ/Wq9nIDosZpGwgy7BR6k4rlJNRvZfvXEmPRTj+VVmJc5Zix9Sc1SpeYjrJNQs4vvXMefRTn+VVpNctF+4JZD7Lj+dc3iiqVNAa1xrfmY8u3C+7Nmqranct0Kp9F/xqn+FOCsexqlFIB73VxJ96aQj03YqLGTk8mn+W1Gw4BOcHoaYDMUcU/aKMCmA2jn0p1KAT0BP4UAMwaXB9akEUh/hI+vFL5Ld2QfjmgCLbRgVP5S95PyWlCRDs5/HFAFeirP7sdIl/Ek0okI+6FX6KKAKyqzfdUn6CpBbzH/AJZkf73H86m8yRuNzn6ZpRFK3RD+NAyEWzfxPGv/AALP8qcLdB96bP8Aup/jU4tZT2A+pp4s3PVwPoKQFYRQD/no34gUuIh0hX/gRJq4LJe7sfpTxaRDqCfqaAKIk2/dVF+iClM0h43t9Aa0BBEOka/lTwoHQAUAZYR26Ix/CnC3mP8AB+ZrTpDQBQFpKepUfjThZn+J/wAhVw0UAVhaIOrMacLaIfw5+pqRnUdWA/GlQ+ZxGGc/7Kk/yoAaIkHRF/KndOlWY9Ov5f8AV2N03/bIj+dWo/Dury5xZMuP78ij+tFx2Myg+1bsfhDVWxvNtGD6uT/IVmaxYTaRei1lkjkYxiTKg45J9fpRcLCXJtWij+zhlccMCD0wPfk5z6U+xuorW881oy42kLgDKn1wazd7etG5v7xoEX9QuheXbTBNgIAx3OO5qOa4Mq7SFAznrVOjFAF621O5somjt5vLDHJIAyDjHB7VTLgnJJJ96ZikoEbwpy9aYKevWuYZyN/O11eSSN0zhR6AVBipbldt5OvpI386MAYrpWwiPHtRg1ZVogo/cMzY5JfA/Sl8/aMLFAv1XJH50wKwUnpz9Kd5TYyVI4zz6VZSa5diI3fLckRjHT6U42N253NDI2RnLEUAJ9gCiFnliVZU3glunOMH3pwtoFUF7mPOfuoCT+fSlFm4+/Lbxf70o/kKBBbjPmXq9OkcZJ/XApDHKbJSQUncdjkKRUMrI0hMaeWnZS2cfjUv+hIDgXEx98KP0NKktqgOLLzG6gySHA59BQBAJcIyBhhsZH0pgjdlx87KgzgKTtHr7Va+1yhUVBHGI/uFV5H49+9OlvLueMxy3DlG+8AAM/XHXpQBSCr7ml+UfwD8alEa04Io7CgCENjoAPoKXLn+8anGB6UoNAEAic9vzpwgc9wKnBo3KOpAoAjFt6v+Qp62yEgfMSf1qQP5m0KAxHHyrkn8qtraahcFWW1unKgBT5R4A6UAQfYPKmjjmiKbyOpzxnFXY7KBVR/J+Vm25LfT0x61YGi65cyiR7SQsvOZHVcc59a0IvDOsMwytnD3yWLEfpSuMxba0jkRGmu4YQ2eDywx6jtmooSgdDKpZe4BrpIfBM7bTLfxrn+5GT/M1ei8FWoA828uH5x8u1f6UXA4+XyxM4j+aPd8vXkfjzUl1NFKY/KhEW1cE4A3e5xXbp4U0eNW3pJJj+/K3p9auRaHo8Ljy7C3PHdN386LhY80EiE4DLn0BqeO3nm/1VtPJ/uxMf6V6fDDBEieVbon+6mKmLNtkwn8vSlcdjzSPRNVlI2afPz/AHsL/M1bj8K6vIMmGGMdPnl/wBr0Is+8cDoe/wBKad+zkgfN6f7VFwscVF4LvmbEt3bp/uqW/qKtR+CY8Ay6hIc9kjC/zzXV8CQ5kxwPSmArsX5mPI6E+tFwsYUXg3TVbEklzL06yY/lirEfhvRYlU/Y42PHMjFv5mtYBfMPyE8DqPr60g4jTEeOR6ClcLFaHT9Ohf8Ac2duvA+7GP6CpwVWL5UwM9l96k+beeB0Hf60wh/LPKjn096BjtzbxhT0Pp7U3c5D8D8/alPDjL44PoPSmZTD/vCevQ+3tSAU7vk5HX09q4Hx1/yMKZ/59k/m1d6QuE4Y8+57VwXjnH/CQJgY/wBHT+bVURM52lxR3paskSilopAIabTjSGmBtinCm0orlA5bUht1K4H+3mhdoKnBIxzmpNYG3VJfcA/pUalfLA2/N65roWwg6NkAde9SRzSpwm0ZOeEB/wA9Ka7hsYVVwAOP50qzOoUKxG1tw9jTGPa5uX6zyYPYHA/SmGJmTzWyQx25J60SzyTMDK7OQMAsegpu44xk464zQBI8ASON8qd4zgDpSMqKQFbdwCeOh9KjzSMxCnmgCeXyFl/cmR4xj74AJ9ake5tRNKVtlZGxtDMV28egrsoNI01Y9y6crHaCN+Tu/OrcVuIzmGxt4RnptGajmK5TiYbqYrtt7CJmPVlhLk0kWk6nKBssZyAOMrt/nXoysRgBQBk0u44GSBxRzBynCR+GdWfrAkf+/IP6Zq5H4Pvj/rLm3T6Zb/Cuv3ZB+fqvannHzffPT1FLmYWOZj8GIATNqDEDrsix/PNSx+HdFRSz3VxLjrz/AICujx975GPI6mpFG3dtRV5HT8KLsLGVF4c0VACLUPyPvuzf1q9DpmnQ/wCqsIBgjpEM9varPzfNnaOR/SlLAbsyDqPT2ouA5MICEiC4I4AA9KkLvhsKPzqM4w/zOfoP/rU4gfP8rn5e+femBIxY78kfd9PrSgjeMyDp7U3aNx/dDp3xT03ZQgKPl9fpQA1duI/mY8ds+ntTgBt+4x+bv9felUMAmWUfh7GjIwcy/wAXbHrQAuCBJhAOPUelPG8v0A49zTDt/efM547Z9PanYUsDsY8HqPp60AIMhUy6jn0+tBK4k/ek/THpQowq4ix83t60478SYAHHr7UDA7d4++eD6+1Nwuw4iJ+buB/eqUh945UcHt9KjJxG2ZQPmPp60AOGRJwgHHrSDeY1+6OR796XK+Z/rCeOx/wpoCeWvDnkdm9aBDgG8w5YDgdvrUfHlpmXuO49aeAvmH90eg7D3pBny1wmOR396BiDZ5h+YngdCff0ppVPKPyE89wfX3qX5/MPC9B3PvTSH8o8qOf7vv8AWgBOA4xH2PYD0pMttk+Ud+p9qeQd4zJjg9h7VGdu2TMp79x6UgFO75Pu9fr2rgPHOf8AhIlzj/j3T+Zrvzt+Tljz6n0rz/xvj/hI+B/ywT+tVETOf70tFLVEiUlLQaAG0lKaSgDbpRTaUVzAc/rq41EH1jH9app90VoeIBi5gb1TH61nJ92uiOyESUUUUwClpKWgApr9DTqQ9KAPS7N9+nWzbid0Cngewqf14Y8iqWjvv0WxO7/lgBwPwq72zlzwO1Ymo4Dn7nfuaeoIxhVHWmAc/dY89z/9enqvI+QdT1oAcCcDLKPlp2QQfn/h7DNIgI28KOCKcDwMuoyv+e9MQpC/NyTx708gfNhWPQ/5yaTPB/eH7vZRTjyGyZD8vpj19qAHbcbsRrxg9qkww34CjvTGAO/5WPHrx396cUHz/ux0zzj3piHNkB8uo49Pr704kZb95/D2x70YOWwij5fX6+1PG/d1AytMQgwWHLnI9DQgU+X8jn5e/wCHqacp5TLqPl9PpQhBEeZMnHbHpQMFUALiIfePXHvTxv2thVHzev09qQbcDlz8x9felIG1/kY89/w9aAHENl8lRx6fX3oz8y5lHQ+ntRtG5sRdh6e9OXd8mFA49fp7UAR5Uqv7xj83b6+1OIX95w549/SnAPsH3R83v60Nu/eZdRx6e31oATau5f3WeD1AowRG2EA+Y9/elyNyZlHQ9MU0lTG/zsfmPT6+1AEnz+Z/D09Sabh/LHI6jt7/AFpcL5nRzx703avlj90T83oPWgB2CJDmTHA9PeowU8tcy55H8Q9akAxIcRgcD096QF/LXCqOR39/pQA35DIeWPA7n1NNITyj8hPP933qXD+Yfu9B6mmkP5R+YDn+77/WgYmAHGI+x7D2pCW2yYX17+1OPDjMmOD6e1MJXZJ+99e49KAFO75OB19fY15943z/AMJIc/8APBP616ASmU+Ynn1Poa8+8bY/4SRsZ/1EfX8acRMwaKKKokSiiigBDTaWkoA2hS0lKK5gMfxCP+Pdv94fyrKj6Vs+IFzawt6Pj9KxY+lbw+ERLRSUtUAUtJRQAtIaWkPSgD0Dw6+7QLL5uisv5Ma0h93q3T0/+tWP4VfOgQjcBtkcfr/9etgH/aPQ9BWL3NEP9eHPSngDP3T97uajB4P3+g7VIByeH6jv/wDXoGPReV+QYyaegICgBRwaYo5Hyn73c09V5X92vU9T/wDWpiHKTgfMgyvr/wDXp2Rj768r9aRAw2/Ko4I5p6FsL86D5fT6e9MQuQcnex+X0/8ArU7g5/1hyvvTVbhSZF5XrT0IO394TlfQf4UAOwCfuMcr3P8A9enIoyv7sfd749qahBK/M5yvofb2pyY/d8SH5ff2oEPjDYQhVHHr9PanKHwmSo/A+hpigYT923T1/wDr05VAC/uhncfT3pjHZIHMi/e9Pf60Fl2yfvfyx6ClAIBwij5vX3+lOO8CT7o49/SgBMrvPzOeB0B9/QUi4IjOJDx7+lSfP5nLKOPT3+tNB4jzKvT29KAE2rt/1RPz98f3vrTtvzPiIdPb3pu5dn+u/i7Y9aXK7n+dz8o6fj6UAOG/KYVRx/e/+tQfM2P90cn1NNG393/rDx/telBCeXJ+7Y8nr/8AXoAkIfzeWUcf3ff61HnEYzKB83t607avmjEPY9h60gz5YxGB83r/ALVACZTzT+9zwO496blPLX5nPI7n1qXL+YflXoP4vr7U35zGv3eo9fWgBvyeYflY8ehppCeUf3R6/wB0etS4fzDyv3R/D/8AXphDeUcvjn096ADpIMRdj6e1IS2yTC+velPEgzL2Pp7UwldkmZfXuKBjmL5Xhevr7H2rzvxrk+JpM44hj/lXoZKZX5yef7x9K878Z4/4SabH/PKP+VOImYdBoNBqiRKSg0hoAKbS02gDbpRSUCucCjrgzp2f7rg1gR10erDdpk3tg/qK5yLrWsNhEtLSUVYC0UUUALSGig0Adp4Pf/iSuu4DbOevuBW8DyPn7noK5rwXJ/oF2m4DbKDz7j/61dKG5++OvpWT3NFsKD8v3m+76f8A1qkz1++ehqNTnHzHoe1PByOrfd9P/rUhkgHXh/vDv/8AXp4HIyhPzHv/APXpmeT9/qDTx9H+96//AF6Yh6KMriNep71LGrDb8g7/AOelRDHGVP3j1P196eij5f3Q6n+L60CJUDDZ91eP8KchPyZdeV9OvT3qNFwV/dIOo9T/ACqRFI2fIo68d/5UxCo33P3qdPb2pyMP3f7ztjt/hQu75Pu9xSgsNuXXqe3196AFUjCfM/U9vr7U4Y4/1n3j6+9IGwFzKv3j2+vvS7h/z1H3u2PWgYuBhso5+Yevt70rKuJP3RPHt6fWk3Lh/wB4eo9Pb2pSQTJ8z/d9D7+1ADwB5g/dDp7UJuCx4RR+Pt9KT5d4/wBYcj/a9qRcbY/lkP5+hpgP/ebG4X73qfWnYfe3Kjgdj71EQNj/ALpj83fHt70/aPMP7nsPT3oABuAjy6j8Pb601m+STMo6n0oHyxofKXAGe3pQCzROQigHkc+w9qAFLL5v+u/hPTHrSZXyx87n5v8A2ai5kkhjeQBPkQnqafiTZ/D973/vUAN+XzT/AKw/KP73vTQF8pflc8j19abdPJFFK4ZQVQc7ff61IVYRj5x1H8Pv9aAEwvmH90Tx6Cm4HlcQ9/QetJcOY4pn83G2MnoO2aU48rmbv6j1oAdyJOIwOD3HtTSX8uT5R3/i/wDrUpK7/wDW/wAJ7ioIJFkslkaQksm48+ooGTsX3LwvX19q858Zk/8ACT3Gf+ecf/oNeits3L8zdfU+leceMCD4nucZ4WP/ANBFOImYtIaDSGqEBpKKQ0CENJQaSgDbpaSiucCK+G7T7gf7Brl4+tdZKN8Ei+qEfpXJR9RWtPYRPRQKKsApaSloAKDRSGgDqPBL4F+m4D7jc/jXU7uT846iuQ8Fvi9vEyBuiB59j/8AXrqpJGDMRKAFVWxgc881lLc0WxOrcj5+57U8HgfM33T2/wDrVGrgswD/AHWwePb/AOvTg4DIpY5YkDj8f6UhkufvHc/Qdv8A61SfxH7/AFHrUWcKSS33P7v/ANanq4ZAwL4ZQ33f/rUxEw69HPzdiaeuOBtY/Mep/wDr1CHBldMyZXa3T1//AFVKHXzNhZ92cgH04H9aAJFAyvyH7x/r705FHH7odT6Z700fST73Y4p6gdNr/ePAb/69AhyqPlxEvUj+ftT1U8YRPvHofr7UzAABZSAGxkt6/jTwuQP3Z+96j/GgB4D4+6v3vX3+lOG8A/dHzD19qjG3LKI1yGGRkcU26kS3tpZWjXCkccdTgD9TTAnO/D/Mo/D/AOvSktlvnQfL6fX3qISRlGP7kEqCRuHHFDzhZ4wvlbHyCQfQH29aYiUN8y/vV+77e1AYYT96Op9PemrOnyZaMfL6/T2pkdw3nSIzJhX+XAJ7c/rQBISu1z5xIz7e3tTwV8w/vHPA/wA9KYZlxIPNj6/0FMtp3dQXbLgYbEZOOf8A9VADpNptxhpPun1/ummwbTZDPmH5B/e/uim3N0IbJnBDOsbMqlD82FPFFrKG05XaRFzErEccZUUDDUQDZXGFfPlN1zU+FKE+W33v/Zqgu5kaCULOjHYRgEe1P+0RCP8A4+FJ3dBgnrQBHqCqbWb91/AOw9an2jYMQ9/QetVryeJ4ZFE/zFRheATz9Km+0Q+WP3xOSMYHXntxzQAy/GbO6AjAzCw7ehqYbvK+4Bz6+9VruWJ4Jk8x9zJgAgjr+FPE8XkjDyNyMYBweex6UATOXBOFH3T/ABf/AFqrWJb+zIsAYCbep7DHpT5JV+YbZ92w4+Vjn8qh0/b/AGWmS3RvX1NAy427ev3fzrzTxgT/AMJRd59E/wDQBXpJ271+939a808WnPii9x/sf+gCnETMc0UUhqiRKQ0tNNABTaU0lMDcopKWuYBy8nHrXHgbZCPQ4rr161ylyuy8mX0kP860piHClpBS1oAUUUUwFpKWkoA3PB77dZdcgboWHP1Fdi+1kfc4wUwcelcR4WfZr8PONyOP0/8ArV3G7/bHQ1lPcuOwiqAWImb5iD+gp4XLKfOcYY+/86Tf1+cdqkDc/f8A4qkoUJkD96/3TT0QBABLLgJxjj+lIjcj5x37U9W+UfP/AA+lMQ4RKrOwklGQM4//AFVDeRqhMgadjtKKoBbk7SD/AOOmp93B+c/d9Kl3jLfOT07f/WoASHZLGW23Efz4wcg/pTkt4xLKSjFWZccEk+uePpT9wyfmPUf09qfuHPzkfMOg+ntQIZNbI8EiorbzwCc4BqbyYcn92w+YdMj0oB6/O3DDt06e1PyAD8zdQen09qAI1tk+0O3lsUwoGeec89adc20TwSKITkEMMDngg/0qQsPm+d+o7fT2pWYYk+d+np/9aqERWYb7MUkhdGVQMEnnjP8AX9KlkhDyqVjIO5XY8c4/GnkjLfM/K+n19qUEFx8z9PQ/4UDBcEIDGcFe5Hp9aYkKpIXWLA4UKMYGC3+NOUjEfzScj0Pp9KXIx1k+/wCh9aABlUrIGhBA55x6CiGLynYeWrbmZu3GTnFDFSsnMnT0PpTsr5o5lxj/AGvWgQ0xrJCqPENrDaeR0IIqnYQXEFuIZraHbGhAYEE5HA/TmrqsuxP9Z1/2qCy4l/1n/j3pQMc4ZjjYoypGc9OntUcKvFaJHhSUwuc9cH6U/wCXev8ArOh/ve1NO3yz9/73+160AOlEkiyRjaN6Fc7jxnPtSIHWBFG35cDqfX6Uvy+Z0fp/tUz5fLHyv1/2vWgCQ+YWYfKMrjqfeo41eK1jjBU7AFzzzg0o2eYflfoPWmnb5X3X+96H1oAkzJ5g+7096hijaG08sFSFyAcH1NP+Tf8Acbp6GmEJ5T/I3fsaQyQ7968r37f/AF68x8WE/wDCUX2f7y/+gCvTCF3j5D37V5j4q/5Ge/x/fX/0EVURMyqDRSGqEJSUGkoADTaU02gRuUtJRXOA4VzOpDbqM/8AvZ/OulFc9rIxqLn1VT+laQ3AhFLTV6U6tRBRRRQAtIaKDQBoeH32a9ZnOMsR+YNd6XCn5pAOSOa880ptmsWTZx++X+dehMTvT5jw/p7Gsp7lx2FD5XIkByuakDcn5x1FVUnClI2kAO1uoweDUrXCIjN5qnAB+tSUWFfkfOvWpEYcfOvQioBMmeJUPzduadFcoxOZU+V2XjnNAFgMMffH3Kk3/e+cfd7/AI1V+1KHiXzVAYEHcMdvepGu40KAzRnfhOOgPPXmmIt7/vfOvY0/zAN371RyOBVeOdXi3+ZENy5wTj+tSfaFMpRZYzlScr2x260CJ9/3v3i8Ecce1P3j5v3q9j2qIyqN372MZxjnkn86dHKZIt+5RvUHB/l1oQiUsPn/AHi9PanM/wB/96v3fb3qNph5rR71LMp6e3Xv71JLJ5UcsjsoRELE/TNUA7ILH94vI9qVT8yfvF+77e1VrKZfs8Z3pyCQO+Nxp0zkxQlHQ/OnKn3FAydWGE/eL+noaXeMH96vDe3rQCTtAKEgnjP1qEb/ALe+AufLHf8A2v8A69AExbPm/vF6e3pT937xf3i/dP8ASkw/z5Venr9faobUuzSZAOJHA59x7UCJQflT51+9RuDJIVkUgjII+lKNwVSUXhvX3+lVtNbzNNidVUq0SkHPUY+lAyw7hMM0ihVUk8fSjPyH94PvenvUd2pNpL8i8xOOvt9KcmTAD5a/n/8AWoAWSQR7nZxtVcnilIIQfOPvenvUF+p+yTjy15iIqY8p/q1+97etACsdpZmcBVXJ4+tN58v746+nvUN8P9DucRgEwnHT0NS4BiB8ockenrQMcch+XGNvpTOTExDgg5wce9JMAEkPlD/Vn0qK2H+gRZi52D0pAWSDvHzDv2ry7xQc+J9Q/wCugH/jor07A8wfu+x7D2ry7xKf+Kl1HjH73+gqoiZm0hopDVEgabS0hpgIaQ0pptAG4OlLVTT5/Mi2MfmX+VW65wFFYeuri8Q/3ox/M1tisjX1+e3b1BFVDcDPT7op1Mj+6KfWwgooooAKKKDQA+2by723f+7Kp/UV6Wxw5+Y9fSvLydrA+hBr09myc5bnB6f/AFqzmXERS245fPJx8vQelOJBUgsCCvQimg8/ePX+7/8AWpQ3H3+3pUFE2/BPzr1FPRsdGQfMen/66i3dfnHQGnhsfxr1/wA96AJVblfmTIyP880SbnhXb5buvzKrHAJFIjcj5l6n/PWnoTleU70xCwlyreasAJXsOoqcqpDZWL7o/h+tRqSQv3Pu96euf7sfK/57UCJWQEPmOPBTBOOSOeOlPMS/P+6i5X0+vtUYXOT5acr1P/6qftzn90hyv+e1AhxgjLN+4i+7gYH146U5raN1ZPJjAdCuQBxnI9PekA/6ZjBX1pydR+7x8vqPamBBpvmh5EktvLA+6T34Uent+tSXMY+yuwgQyCM7TxnOKlTHyZRvu+v096UEfJ8r9x19vrTAUwxYUC3TAY9h71C8CiZAIWK5O8k57DAPNTAjA4k+96n1+tBIxJ/rOvv6CgYGCIlgISnHVCB6+9QwxJ9pkUxuUAGATxkgEnrzVgEeY3MnQdj7+1IrD938z9PT2+lACGBCnAmXJIOHPr9ao6akcECWO65LxR7S3QHaMECtDcu3/WEfN6f7X0pd4LOPN7D0oAjeGN0CEzgFcZDNSCFFt9padtvBO5hn8qkDg+X+9HT29KTd8j/vV6n0oAZJbxSbk3TKGXkhmPf3pAmIlBnmOCM+/P04qfJ8z769PT/69N+bZ95fven+19aAIpohIGTz5VDKM55yM/SmqriBQbk5yAflHr2/+vmrHz7+o6ehpuX8sfd+97+tAyGRGbeq3bZZcfMFI/QCiMBLUIZfu8dR61Pl9/bp6mmEvsPC9T39/pSAXcPMH7zse49q8s8RnPiTUTnP7416nlvMHA6Hv9PavKtfOfEWpf8AXw1VETKFIaKQ1ZIlIaWkNACGm0ppKAHQyGKRXXtW2rB0DL0NYI44q/p8+CYmPB6VgymaArN15f3ELejEfp/9atRQScDrXP6pdSTytERtjjYjHqRxmnBakFSL7tSVHF3qStgCiiigAooooAY/3TXpMchaxjkBbJhVuntXm7dK9D01vM0e0PzfNbqOPpis6hUSypbLBnJIYchcdqeDyPmPftUDuRLhG+Ztp2kdqkBkBGGB57qRmsyyUN8v3x93PSpN3J+deo/z1qujyFOSoyCOQaeJJNx5j6A55pgWFbn7yn5qkXPH3OpquXkGSPKbnpnHpRcPLHbyvGkZZAzA59j2xTEW0U/LlF6H/PSnoPu/JH93/D2qCNnJJ+zFQGIGCOacJChjBgbk4+UA/wCelAiwoB2/uwTt6k8CnqAcfITleoPX2qBXHy5il4H4Dp70sNwjgFRN8uVJweMYp2EWk5K/K/3f73096cnVOHzt9T7e9VRcRx+Vv84bvlxtPtUyTRbowZWU45BGPT2oAfDIrcDzModnf0B/rSXEhitXkVnBUMR8vsfaoLCZZVkkDMA0mcED+6B6e1SXbA2Uw8wcq46D0NMC0SATh2+/6e/0qN5QsmzzPvAnnHYCnB8rkSL94Hp9PeopS32qIhl+6/8AL6+1IZZDDzD+9XoPT1NNjk3FQHXK4B4/2QfX3p43+Z/D096rQbjcTAqvDL3/ANge1MCxl9v3lPzenv8AWnfPvb7vQevvTMfKf3a/e9vWo7aRZ1ZxCAMkcEHOGYf0oAmXfiPhenqfT6Uh3bHyq9T3/wDrUzgIh8s8DPb0+tNicSWokKOC6hiATxkA0hkx/wBZ/q16HvTMDZ/qh9729aZdv5cMrqXDLExHXrTgymMHc459/WgB2F8z/VdvQUzC7B+7P3vT3pt1L5cEzpJh1jJXOOvOO1KGBjU+b1I9PWgB2F3n5W6e9NO3yzw/X/a9aSZiscrLINwjJHTg802Mu1rGxcEkAk496AJRt80cv0P972ryrXTnxBqOP+fh/wCdeqru8wcjp6f/AF68n1k51zUf+vmT/wBCNVETKdIaKSrJCkNFJQAhpM0pptADyMipbeGWdwIVJYc/SqxmCJzye1RG5m27RIyr6KcCs1G42zqrfMC7ruWJWJz1wFFcpcv5k8rZzucnI+tRnk5PJ96McVaViRYupqWoo/vVLTAKKKKYBRRRQA013ugtv0KzODwhXr6EiuDNdt4ZbdoUHGdruvX3P+NZ1Nio7mp2zhs4FOBwerfe9Pao1YtwUIBBxhs9Din9/wCLqKyLHqenzHv1FODcffH3e4pinkfM3X0pyt8o+cfd7imBLuyT8yHkGpB16KfmqHdkH5k6D/PWpPXhDyKYggtlicMAztyuWbJ/lVheq8dzUQHP3F+9609eMcMPmPf60CJU6pw5xkcH/wCvT0P3Pv8AAPHYVEpHy/6w8nofrT1P3RlxyeMUwJUYfJ879+o/XpT0cDZ857jkVCrY2/OepHT605X+7+8U/Me31oEVbJpILmSBFXyg5IyQOMHB6/UfhVx0kljIM4BYkEAZFNUN55k3IdwC/kW/xqXJ/uofmH8/pTAbDNI9usnlqVYKflOT0HamzMpuImITADck+x68UqRiOHYI1IUAD8AKSeIMwKRAEMrMRgZAzSGWFkQyL8gGRxkYz+YqKDi4lBjI5XuP7p96f8rEBlcgqQeT7e9V0iUXK/I4jCcZBI3ZP64oAteci5ykpAfGVBPOfaqemYhiW380pKdx8soSR8xPrVvcqowDlQD0wOOntSBYvtG/Me/ruwM56fyoAQq5iXbcPuK4G5Rjp9KRFaOFkFwm1eOUHTH1qRWO2PDKfw9j70pL7ZPun8/SgYyZWl3IJ1wykEFRjGRmlVJhEB56kA4yI+ev1p7FvMGVXoe/09qYfucxj73t60AEkUj7lMpwQAcjjGeaTZJ5SjzGwCOMgd/pTuPM/wBX29qZ8vlj5W+8P50AJJDv3ryCVxneT+hoCqsKqIxgEAdPWnfLvP3+g9femZXyx87Dn+tAyVAPNH7vt6CvJtWOdZvyP+fiT/0I16ujDzR+87eoryTUTnVL09f9Ik/9CNVElkFJRSVZIUlFIaAA000ppM0ARwRefKFZ1RSeWY8Cn3P2dP3cG5yp5kJ6/QVWopkiinMVHC5I9SKZTgpIz0FAxqffqaoF4cVNmgBaKKKACiiigBDXX+E23aQy4B2zHr9BXIGuo8It/oV0hxxKDz7j/wCtUT2KjubzIwUFFGQG7460qtIEG6Ns4GfnoKBiuVB2tkYNOA46Hp2NYliJI29srKAG4+UelO84q8Yw+DuByme3tS9z97qKepOR8x6nqKYCGbEbEHoh5K09JkeMMTHkqDSBuMFl6dxT+oP3TwKAHq6mRkCqcbSCG4OfT8qkZhGpZg4Ctzg5qExIWZjEhJxk+tK0MbKVMbKMj7px6e9MRZBxjmT7x6fjSq4zgMww3Ixz39qhCr/elHzdiaEj2u7ieQbmH3gP8KYiyZQigtIoAJzkdOtPDnjDJ94/1qrJGZIiiz4y38XNS/vT/wAtEb5u60AT84+6pG6lwOcxj7w/pVQCYXDsYEKkKOGwM5+nvUu9l3ZgbaCM4cZ7dqAJzgB/lbpng/8A16fkbz98ce9UoJXNsC8c+7YMk+uPrTpp2WSLHmqWYZ4ByOc0AWgwynzMPl7j6e1KGG1f3g+8euPeoFnGUALfd/uUi3IMuwPGV4Oc45LEY/lQBZLHa+HU8+nsPen5bzOx4qEnIdSIyfTPPSkR1eVtsf3SVPTqKAJB92PKKfx9vpQQNsn7sfhj0pqj5U+Vxz6//XpkcqywNIhkCsMjIPpQMmON4+Vhweh+nvSZGw/fHze/rTJJVjBdnICqzHjsKA4KZWRSCePzoAfuHmffbp/ntTdw2D953Hp60oLbzgqePSkJfYvA6jv70AOBJc4deg7fWm5fyxyDyO3vSc7zlB0Hf60zA2L+77j09aAJ4y3nDgdB3968hvTu1C7PrO5/8eNetxY84fKR0/nXkNwc3dwfWV//AEI1URMZSUUhqyQpKKSgApKDSUAV6cEJ9qeAM5xT+1XykXGhAvNRs+eB0pXfdwOlMxQ32GhvetCC6tQkay2wygILg8nPqKoqPn6ZpQjFQQMgnAx61Ay7tsXbieVBz1Sqz7Qx8tyy9iRg1G0brnKkbTgnHQ00D0oAsTKscm1JRIuAdwqPcajyaM07gSbjV3TdXn0zzBCkbCTG4OD2/wD11nk0maT1A6VPFsn/AC0soz/uuR/SrEfiy2xiS0mXj+Fgf8K5LNGaXKh8zO3j8Uaa33jcR/VM/wAjVhdc0mYbTe7RnPzKR/SuB4opciHzM9GhvLJ9oi1WM9eDKP61oRlJIxtkil+XGQRzXlGBR05HH0pcgcx64Yz8x2dh0p2MZ+VxyOhrzvUbW40pYmgv52WQ44LJjgH196I9R1mNlWK+uWZl3ABt+R+NN02hRqKSuj0F5ZEfCxyOuRyP8/Sj7TtzujnHzDkpmuHXxHrkH358jP8Ay0hH+FWI/GOpL9+K1k5z90j+tLlY7nbRy+YpYHjdj5hgmn4JzlVPzD+lcenjWX/lrpyNz1SXH8xVlPGdmf8AW2NwnI+6VP8AhS5WO6OoxwfkxyOh+lKSAG4cfjWBH4u0hwQz3Eef70Z/oTVpPEOjyhtuoouR/GCP5iizC5rFuX+Zh8vp9fanh/nH7wdO9Uo9QsZifK1C1fI/56L7+9W0y5UoyOMfwnPpQA5WbCcqeP6UbELb2hjLFsZ74z9KQIQEyg/yKTGB9w/e7fWkApjj2SL5IA9sDtQkEaOcGcbsn77ev1oJGH++OPf0pdw3j5yOD1/CgYgSPauGdTn73fv3NNSJI4mRJWwBjO7npT1Y7Vw6nn0pSWw/Q/8A6qAGyxF1KJJt3IR1JHb3pQsqxBQ4YA4+Ye9B++MovQ/0pDjafkI+bsfegCO5SV4pUCI5dNoxwRnin722Lm3IyR0YetOyN5++OB60zcNi/Ow5HUe/0oAZPOY4ZmCSo4T5TyeecdPepQ6mNDvIzg80BvnOJB0FIC2xMEHpSGSxMPO+/wCnp715A5zLIfV2P6165GzeYcgdB3+teQZyzH1JNXEli0UmaCaskKSjNJQAUUUlADM0HkYptL061dyBmMUtKRmkqShucMCOop/mFT8jEAdKaeTSYpAS/aJCMMQwznkUwuCrdix/SmUUAPDep/Og/dz8p/CmUUAL2FJRmigAooooAKKKKACiiigBS7EAFmIHQE9Kcs0ikEMRimUUATm6lcYd2YZzgsTSCb1WoaKAJxMvcGl8xCP/AK1V6KANjSbzSraxuY76zM80mdjYzt4OMHtyf0qLSobGeKVLqby5iRsZm2qq85PuenFZmaM0Aaujafbai9wtxdi3EagoxA5OfSk+w+TrL2IuFAQ/64AjjGcgA1mA4II6j2pUco4ZcZByKSXvXb0B3asjZnk1Cyvobaz1GWUzBTE0czAHdwAQTwau39/4l0Xy/td1KFcnaSVcEjqORXOee4uFnQ7JFYMpXsR35q3qWs32qxxJeSK6xklcIF5PXpTYle2ptDxVr0Fuk8yo0MvCvJAAG/LFWoPGuosnmHT4ZYxwWRXA/Pmucl1eWXT/ALI8EJ+RU8zad2BjHt2q9ofiMaXYS2rwCRWJIOfWiaS+HUqnd/FobkXjyIgCXTu+fkmB/mKtp400xw3mW1zHn0VT/WvPIjH5ymVSY93zBTzj2q2qaWx5nu4/rGrfyNKyC7PQU8V6LIRm4lj4/jjb+masprukyjCalCOf4m29/cV5e0cAjQpc5YkAqVI28c/rSywmKMOLiKQE8BTz9cUuVDuetRXlrK2Yr23fgdJFP9alVWKrtIbntXkqabeSxLJHCsisu4YZSf8A9ftVeOWVeYyVI9GxRyhzHseHDHKjtUePlTKenpXk8erajF/qr65T2Erf41aj8TazHjF9I2P7wDfzFLlHzHp6kAudpGF/xryJTlc1sJ4x1dVYM8L7hg7oh/SsRZABjBppWE3clpKZ5i+9G9fWqEPpKbuHqKXI9aAFpKKSgC2sUUMYkcfMBzznmqUjq8hKKQD6nJNNydu3Jx1xQCVORwaq5Ni6IYoQXdznHGe1UWxn5c496CSTySaSk2MKKBRSAKSlooATFAGTycUtGKAFZQOjBqbS4pKACiiigAop8RRZAZELqOoBxmpJ2tmUeQjo2edxyKAIKKMUUAFFFLsbbu2tg98cUAJRSUvFABRR+NLigBKKMGigAooooAKSlooAOaMn1oooAM+w/KlyP7v5GkooAXK+jfnR8vqR+FJRQA4AZyGH6ilAYdCPwamUUAP2P/dP4UhBHUEfUU2lDsOjMPxoAMiinea/ds/UZo8w91Q/8BoAaOvpUkflBz5u8rjjYQOfXmm71PWNfwJFGY/7rD8aAJ7GO1lnK3k7wx7eCi5JPp7UwLDHd7ZCZYVfBMZ5Ye1R4jP8TD6jNL5Yxw4/EGmBbv4LaBQI1u4pSc7JlA49iKqKHYZGTTn82XG+XzMdMvnH503ypP7ufxpDG0lLSUxBRRRSAKKKKACiikoAKKKKAClzSUUALSUUUAFGKKKACnBMxs+9QQQNpPJznkfl+tNooAKmhupoF2xvhc5wQCKhooAmnuZJ8bwox/dGM1CMZ56UUUATuLXySUaXzPRhgGoKKKADmjJoooAM+1GR6UUoOAeAc0AJRRRigAopKKAFopKKAFopKKAFopKKAFopKKAFooooAKKKKACpEZQuCCfxqOg0DJiYuo359Kiz6Cm1PIdpAHoKAP/Z', 'IMG_20261110_133206': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAA4KCw0LCQ4NDA0QDw4RFiQXFhQUFiwgIRokNC43NjMuMjI6QVNGOj1OPjIySGJJTlZYXV5dOEVmbWVabFNbXVn/2wBDAQ8QEBYTFioXFypZOzI7WVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVn/wAARCAEdAXwDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwCrilxQKUCuQoMUuKUClxQAgFLRRQAhFFLRQAlFLRQAUmKWigZnzptlPoeaaBVq6TKhvSqwqkIUCnAUAU4UgExSgUtLQACnCkFOFIApRQBTgKAAUtKBSgUgK94m62f1AzUCjBRh3q+y7lI9RiqEf+oHqvFMZYrN1ZMqjfUVpjpVXUU3WpPoaaBGXprbbtPfIrUnX5lNY9sdt1GfRhW5c4VMnsaqQEOKhuFyAfenGZe1QyTbiAcAVA1uYt0uLp/rUWKt3ab7lmUgj1zUYi9W/SuiOwnuM82TAAbgUhJPUk1KI0HUk/jQY1B4JI+lOwuYiXG4Z6UskZR+fzqTYvoafkkAHoOlFhXKrgk8An6ULDIxHykD3q1mgkdzTC5C8OSNzqo7UFIMAbmP0HWpDJHnkgn86PNH8KsfwpgMCJ/DET9TTgrdgifQUb3PRQPqaMSHqwH0FAiQE4wTmkJA6kCmeXnqzH8aURoP4RQAnmL65+lJvPZG/KpMAUlAzrhThTBTxXIMdS0gooAWkoooAKKSigBaSiigBaKSloAa43KQao4wcVfNVJlxJ9aaAaKdTRTwKYBSigCnAUgACnAUAU4CkMAKcBTgKcFoENApwFKzJGMuyqPVjiqsuq2EP3rlCfRfm/lRZsC1tqmI8STJ/tZH41Wl8RWq/wCqilkPvhRWDdX9xdTO7SMoY/dU4AHpVKm2FzqDLFFGPMkRP95gKpXepWRhdBNvJH8IJrm8fnQK0VNCuWTcAPuVTwe/FTXOqzTAAhEUegzVAketJvA4HP0q+VBdk/2iVhzIce3FMJJ5PP1qPeeymk3P7CnYRNk44oz71D83djTSBTAnLqOpFJ5y9smoOBUiRSP9yN2+ik0WAUzE9F/M0eY57gfhUos5z1QL/vMBTxZ/35ox/ugmmBW+Y9WNAVfTP1q4LaAdXkb6ACnhIF6RZ/3mJ/ligZUGBT1Vm+6pP0FW1YD7kaL9EFSj7Q/aTH5UBYqC2mP/ACzI/wB7inC2b+J41/HP8qtC2lbqAPqaeLNu7gfQUAVBBGOshP0WnBIh/AzfVquCzUdXY1YXT8W5n8vMYOMk/h0pAZm5VPyog/DNXbd7TyR57S7++0HH6VcsrFLibYWWIBdxOAT+FRXMaxXEkaOJFU4DDvQMvCnimCnCuQY6lpKTNAhaKSigApaSigBaKSloAKXFKKcBQMbio7hMpnHSlkuraD/WzxJ7Fhmqc2u6egIEjSH/AGF/xppNiHhaeFrHk15Qf3VuT7u3+FVpNavH+6Y4x/srn+dXyMLnSBaY8kUXMkiJ/vMBXJy3lzL/AKy4kb23YFVyRnmn7MVzq5NWsYv+W24/7Ck1Vk8RQr/qoHf3YgVz2QenNPWKRvuxtVKmguakniG7biKOKP8AAk1Vk1O+l+/cuB6Kdv8AKols7hlZtoAUZPtUbQuiI7fdkGV56jOKpRSAazFzl2LH1JzSbgKXaKesbN91CfoKoBm70Bo+b0qcW0vdQv8AvECni2/vSoPoCaAKuG9aNnrk1dFvCOrSN+S/408JCOkIJ/2iTQFjPwo9KkSGST7kbt9FNaKeZ/yziC/7qAVKttdzfwuw9zmgLGaLOc9VVP8AeYCnCzA+/Og/3QTWkdPlQ4kYIT2xmnrp6lgpZiScY6UAZf2eBerSv+S/40oWBekKn/eYmtb7DEkmwp82ccnNOlhjt5CiiNsfxL0oAy1Z/wDlnGq/7qCn+XcydQ5/3jV7fSGQDqQPrQMqLZyHqVFSCyH8Uh/AVOhaQ4jR3P8AsqT/ACq3Fpmozfcspz9V2/zouBQW1hHXcfqakWKJeka1sReGNWlIzDHFn+/J/hmrkPg66baZbuJM/wB1SaVwOfBA6ACl311cXg+2UAzXkzc4OMLVtPDWkxhsxvIR03OTRcLHEFwOpA+tCkucIGY/7IJ/lXokWl6dA/7qyiGB12D+tWY1VRGEiVePb0pXHY83e2uY4Gme3mWJerMhAH51B9pYIUBbaeq7uD+Fd74p3Hw5dZwBuX/0IV51TQmSGVj6UwyN6/pRTaYjcFOFNFOFchQtFFFAhKWkooAWikpaAFoFFKKBjlFZHiW5eG0jhTIExO5h6DtWwtY/ilM2UD/3ZMfmP/rVUNxM5fil59KnjXdGcLznrmrsTXEAKrFCrH5smIMwB+vaukRnpDLJ9xGb/dUmrI0q8KljCygZB3ELj86vhL+Xer3ZjAIDK0oQZx6D60z7DbBibi8iLf7AaQ5+vSkBTtrSFryOK4nVYy+12Q7sD196uraabHwXlfDHlUHIycYz+FOEVghJT7VIAcABVTj60oNvjEdiGPrLKzfoMUDD7RZQ8R2qtg5BlbJ+mBxSTzvcFVS1CYyQsUR5z/8AqqVZrjYWhhgiRcAmOAHH4nNSKL6RUkN04ErEAmQLyOeaAK621xsyU8pS21vMYLj6jrjvUE1hFBEzme2kYY2ojlifX6VcFjGSTNdQAg4yWLmoPlX7vT6YoAqIsn8EePouKkEUzfeP5mpw+Tgcn0HNWYrO8m/1VrO/0jOKAKS2p7uB9BUi2sY6sxrXh8O6tL/y67B6u4FXYfCN84zJcQJxnAy3+FFwsYCwwr/AD9ealGxcY2++B0rqIvB8K/6++dun3FA/xq7D4X0qMjcss3OPmc/0ouByEc0CSsZUZ4+wzjv3/CpbK6VCcJ5hByFCls/lXc2+k6bCV8uwizzyVBP61bRVRAFjjQFPpSA4MQalLIXtbOcZXBLR4754z0qb/hHtZunZ5Y1VickyOM/pXcGTO75x93PApWbl+ZDwD6UXHY5FfB15IS1xeRKc84UsatR+DbRM+deTNggYXC10jD73yeh+Y0jNt3covIouFjIj8MaRFnMLykEfedjV2PS9Pg3eTYQqQRg7BVpmzv8AnJ6dFoYff4c8euKQw27Q4WNFGKGc7my6j5e1DL9/5B93ufrQeGPzIvy0AIGyV+Zj8vYU1RwnysfqacDkp85PHYUigbU4c/jigBMEDIRR839aRmOJAXUUuzj/AFY+93PvQ2QJOUXj+lAC5y5+Zjx2FNXkR/Kx+p9qdn94P3nbsKauCE++ef6GgDN8Sr/xTt78oGCvP/AhXndei+JB/wAU7ffKeAOSf9oV51TQmLSGlpKoRtCnimCnVygLRSUtIAooooAKWkpaBhThTaUUCJBWf4hTfpDn+6yt+uP61fFV9VTzNJuV/wBgn8uaqO4HMaeHljeJCMHkgkDOBVtoi8YkeRWyduM8/wD6qzrRiNwHtVrJ9a6BFmKK32gyEg7gCFX+HuadL5CsPJZyuOd+Bz+FVaKBlpp4x5RRApQc99xznNI945naVfkZs/d9xiq1IaANXR7CfVBPHFcLAibdwYnDZ6cD6VsJ4WhUlZtQdyCfljT/ABqp4Jcf2jdRnGGhzyPQj/Gu3B549ey+1S2UkYlt4V0vaGf7RJ04ZiP5Vfi0LS4R8ljGSAeXGf51dGcD73Qe1KR6gdxy1K47BHDDCcRwxRgHsAO1P3/Jww+6egpoPoV7dBTuTxlj1HTFADJIY5ZNzqzEEd8cUqRpGMJGqg570p9dp7HlqBgMPuD5j70CDPHBX7o6Cn85PLH5h0GKaD8uAT93sKd68Oeh64piFCnI+Un5j1NIABj7i8EUu3nlR97ufalU4K4ZByRwKYCA5AG7qnYUrDO7iQ/L9PWheQo3N90jgUuMj7r8p/epADL9/wCQdM/MaGON43IOM0FR82VX7vc/WlJwGG5B8vQD60ADnO/5ieM8LQw+/wAOePXFKx+/8zH5ew69aCM7uHPy+v1oGIUyzfIOV7n60A4ZeUXK0u35v9WPu9zQpwU5Rfl/woAapyE+cnjsKQDhf9YfmPt605W4j+fPHYe1IOi8ufmPbHrQA0rw3yZ+YdT9KGUgSfKg4/pSkZDfK33u5+nvQy8v+7UfL/jSAN37wfOvTtTVxhPmY89h7GnBjuX5kHy01W4T5+/YfWmBn+IgD4ev/vfdHX6ivOBXpHiHnw/qHLH5PT6V5uKaEx1JS02qEbIpwpgp4rlAWlpKKAFopKWgAooooAKcKbSigCQUTp5ltKn95CP0pBUi0DOEtTh8e1XBVTb5d46f3WK1aFdJI6lpKWgYUhpaQ0CNvwc+zX0Xn543Xj6Z/pXfDPo3bvXnHhp/L8Q2R9XK/mCK9G4/2On171Mty4i8YOdvQ9TTx97gjr2X2po69e5HC04ZP949D2FSMBnb1boOwp2Oeh6nq3tTMcc+hHLU75d38HUH1pgJwF/gHy/WnA88N/F2X2pAfl4J6HotO5JJ+c8g9hQIVQSAPnPBHpRj5Sdv8IPLUBTkfKep6tRgbf8AlmPl+tMQ7gE/cHzCnA8j5v4j0Wmlup3Dt0Wnc56ucN6e1MQi5+X756+1Kq/d+Q/d7tQAcr8r/ePVqFX7vyKOD1NACADjOwZSl3ZU/OMFewoU42/NGPlPQfSlB4Hzn7vZaQwznPLnK+mKAMn7rnK92/8Ar0AZ28yEbfTFKq52/Kx+Xu1AxAuWX92vK9zQpxs5jHFKqcp+7X7vc/SkX5fL5jXApAIG4T94O/QUmeBy5+b0pwb5V/eDr2H1pM8fec/N2X/61ACMOH+VzyOp+nvSFclv3Y6dz9aVuj8OefX6UMvzN+7P3e5+tADRkFDhF+X/AApN2APnX71KFwU+RB8v+FJkgD5kHzf1oAo6+f8AiQaj8xP7s9vpXmqnivSteOdA1Ibwf3R4H0FeaLVoTH0lFIaYjZUhlDKcg8g04Vm6TcZBgY8jlf8ACtTFcrHYKKWigQlFFJQAtFFFAC0CkpRQA8VItRCpFoGcbqK+Xq84/wCmpP5//rqQU7Xl2avIf7wVv0/+tTFPFdC2JH0UlLTAWkNFIaYFrSpPK1ezf+7Mn8xXqJzkj5j1HTFeSRvslR/7rA/ka9aJBbPHJzy3tUSKiKM5zhuo6t7UcAfw9B1b3pqkf7Hbtmn9uPQ9FqShRjPBUckcLmlBOOrdAeFoyd3Vz83oPSjB29G+73b0piFAJI+/1I6gUY46fwjq1JwG/h+93b2oyu3+D7p6DPSgQ8bQ3/LMfN/SlUjgBh0I4Wkzyee46LThkkcv1PYUxC5JBOXPyg8CnEcnhz8wPXFMAJXo/wB31pzDrlR2PLUwFxyPk/iPVqQYBXiMcmkOAT/qx8wpQwBGHUfMeg+tACo3KfvF6EcLQhOE+Zz8vZaFY5X52PJ/hoX+HmQ9enFIYJk7OJD8vrj0pVX7nyH7vdvpSIM7PlfoRy3/ANehV4T92Oh6mgYKuCnyIOD1NIrYC/NGOT/WlUAbPljHUUBvu/Og5PT8aQAH4X5x949B9aTPB+Zz83933+lLu4Hz/wAXZfekJ+98z/eH8P09qAEbpJxIfxx2oYfOfkPTu1DdJP8AWHiggGTG1jx3b/69ADFX7n7tfu9z9KQZAXhB839aFX7n7sfd7n6UgwUUhUxu6/jQMqa6c6FqILL/AKo8D6V5kvSvTdbP/Ej1Hlf9S3T6V5kvSqRLHUhooqhEETlGV0OGByK6K2lWeBZF79R6Gubxtb2NX9MufJm2Mfkf9DXOy2jZoNKRSGpIG0UUUAFFFFABSikoFADxT1qMU9aBnO+J0xeRP/ejx+R/+vVKM5QH2rU8UJ+7tn92X+VZMJ/drW8PhJJhS00UtWAtIaKQ0ANboa9Xs5PNs7eQE/NGjcL7V5Qa9K0d/N8PWZ4J8gD73px/SokVE1Ocfxng+nrTsHOMN1P8XtUAKoJB8gILYGfbNSK6nBwOoPCE9qkofx1wvbq1HH/TPv71Grt82VYjHy4TtTmaQsu0SAB+eAMjFMQ/POQR1HRacCfVu/aoiZNv+rb7o/iFRQzuTDG4XzXUsQX54Azx+NAi3z/tngelOAORw33u7e1QEOE58vO3oST0pVLMAS0S5IOMcjimImAHAIUcEctS/Lg/6sEqKiTeWI8xQoJCkIDmnYYp/rX5TstMCTcMthlyCOi04Nk8Mxw3Zaoz7obglTOwlZAT2Hb/AAq0YlycLJjcOM4FADy2Nu5mHzHrgetGQuM5GCRy3196hhjG6QbGOJMct2xnH60BAlzGBGAPm/i/z60hkqsmEPy4yfvPSB0AUhVIGckc04KODsj+8eSfrUduQC/+qH71qQyP7SpmVAY1CsMnsQQ3+FSJNg/PKgBbKgjp1/z+NRXMLO4eKWJG3AE7ck4bPr9atbv9v+L+7QBB5xaWRVlJyRtwvT1pyttR1JlJVv7vbj2pSp87zN74Hy42+49qc38fMnUdvpQBBdNmKTCy4GGP0HNO2EzLJsYtjkbu350+VQ6SqRJgrg8/Wlx84+R+n97/AOvQBEA21MQ5OO7D0pkCbYEUInytj8jj0qRV+WP9327n2pAgC8Rpjf8A1+lIZV1r/kCajwo/ct/KvMVr07WB/wASXUeFH7hun+7XmC9KtEsfSGikqhEXDrQvoeopsbYOO1PYchhXOam5p9z58G1j+8Tg+49asmsC2nNvMsi/iPUV0AZXQOpyrDIqSGrDTSUppKCRKKKKAClFJQKAHinrUYqRaBmZ4kTdpqt/ckH9RXP25+T8a6jWk36RP/sgN+RFcrbnhh71tT2JZYFLTRS1oAtIaKQ0AIa9E8KOZPD1sATlS6dPc152a7vwS4bRmQkfJOw5bHUCplsOO50i53Zy/JHGB6U4bsD7/QdxUSMpxgpxjo2afhcH7nQ9veoLHnuDnof4qQ9ckAcjq3tS4Ge2M/3famjoOew6JTEJ8uP4PunvmkSKFHLKkSndnIXnmn856t3H3aXJ65fsegoEKCMYBHQjhafuPOCex4WmDOQPn6kdqXnbkh/ug9aYiQE5HL8N6e1Azgf6zoR2pMHP3W+8P4valAAIyo6kfeoAGXcpyrHCg/e7/nTmB+b5D1B5amAAqOE+53akLIdw/dZABPPSgY8qMn5V+8OrfSjAB5WIfN3NI7IoYlosAg9KUMOcMg+YdB9KAFBA7xfePSgNyPnT7x7fX3pqShhlXXG/+6aVpNikl2ADf3DSGKXwATJgb/7tG48/O33uy1FctmLG9j+8U/d/2hUpJ+bmThh/D9PagBpf52TL5yD936e1K3R/9Z2qJv8Aj6b75+T6dxUj/wDLTiTp6igYy4fy45Xw5wmeuPWn4+cfK3Q/xfT3qG8H7iX5X+7/AHvf61IPvJlWyV7t9PegBoXiP5Pzb2pkeJIg4jUZfuf9qnqvCfJ6/wAVRQIVtwCin5s9f9qkBFrC/wDEm1H5VH7hv/QTXl69K9P1df8AiT6j8qj/AEdv/QTXl69KuJLHUUUlUBV6Gp423DHeoiM8ihWwawepaJejY7GtPS7jBMDHryn+FZxAZaRWIIIOGBqQaOjNNNMt5xcQBx16MPQ080jMSiiigApRSUooAeKeKYKeKBjL1PM0+4T1jb+VcXbn5jXdhN6MD0IINcHF8spH4VrTEyzS02lrUQtJRRTAQ12fgOQ/Zb2ME/LIjce4/wDrVxhrqfArgXF6hxzGrc+xP+NTLYcdzslDqejkYbpj+9kVIDJyCrnk9x0qMqpbOFyM461ICuf4ev8AdPpWZYuZC4OxscZyw/z60jiTyyApDbTzv4BpRt2j7vQfw+9O454Hf+CgQ3LbhlB1/wCentUKPIQwbYCGGMv1HWrOee/UfwVGYoyxcoC+CM+XzxTEO5DdIvvf3z6fSmru2ncYTwcc/pUhPzdT1H8BpQT6v3H3aAGkEvu3QgccYyOtPGQRhovvf3D/AI0bjjOX+6D0FOy2T9/7w7D0pgVbiRoWSQyRqmxgfkPJyD/LNWFB2BlkGWTJ+TrRKC8LLhySGABA96chPlr/AKz7mO1ADNztNIryFgAGGExRJuQqPNfG4LgLj0/wpxDeczYkwYwM8eppLgMSpCyHDqeCPekMfg5ba8ijcOAo9vamJkzSKzyttYEZA7/hUp/j+WTqP4v/AK9RhWE8rbZNp2/xd/z+lAFa6Ro5A+6UqFKYC5PUEfyqzsJ3ODLuJHJ79O1POcN8r9R/F9KGB+f5W7H71AyNojvZy0ucY6ilaIASBRIox0DU9gcP8p/76pGBJf5D93+99aAGPFuJDeYwI6FqQQrhMxsTjqW5qQj5vufw+tNC8p+7HT1pARiLhPlbqf4/rShAEwI1wGA6+9KEO1PkXr6/X2o28H5F+8O/09qAKurjGj6j8ij/AEdv/QTXl69K9Q1cf8SfUflUf6O//oJry5elXEljqQ0tJVAVkPalIwaZ0NSA5FYlD4nwcGnMMHd+dQ9KnRt689aljRZsbjyJuT8j8N/jW0RXNjg7fyrY02482LymPzp09xSYpItGkpxpKRAUoopRQA4VIi5+lNRc1h6xq/mA2to3ydHcfxew9qqMXJ2QN2DV9XMm62tWxH0Zx/F7D2rBTiQVsWWlRSW/2q5n2QqCSmOSBWTJsE58vds3cbutdPKoqyJTuWBRSCloGLRSUUAFdD4JfbrMiDPzwMOPYg1z1bPhN9niG3H94MvTP8JpS2BbnoQf94FxJyRzkY5FSLnj73buKgbABYlVACnJQ9jUi7iAQUIx/cPrWJoS84/i6HuPWn4O7v1/ve1QnfzgRk5P8Bp43Z/gPIP3DTAXHA69B/HS4Oencj79MG8oflQcED5T2NRyO8c53NGiEcEqeDQImxx0PQH79OA5HB+8f46iBbywcRkbeu080o3hyT5eMjjaeKYh4B29/u/36cc5PB6j+OovmO3HlY5B4NLubBOIegPegCUdRx/Ef46VcZUbecH+Kq8zyLtKiH/WDP0x9KImEixyjyycntx3pgT4+X7v8H96jhgSFBBUHIeoS5LIm2NQykZprsY0wBHgD9KQyycEyAL0xn56Vh9/5fT+M1Afl3kCNty5OMdeaR33SPGREoKg9vfNAyd8BXbA2gA53UMOH+UdB3qnOSpWIBNjbgSMYxg4qbdgMCsZwo6EUgJXwCwKpkjjnr1oZeW+RPu+v1qFwWfcFj+Ueo5zTyTlv3cf3f71ADyAOSiABcnn/wCtSKuRGQidPX/61MbLjb5afMpHWiM4SMCJeFx1FACheF+RPvHv9fakABDEKh+fHX3HtQCcL+6X7x/iHvSKuIzlUzuyefpSGQasP+JRqPyr/wAe79D/ALJ9q8tXoK9R1fH9kah8qf8AHu/Q/wCya8tU8VpEhj6SiiqAq0oOKKCKxKH4yKFba2aarY4NKw9KAJyNwyKdDK0UiyL1U1FC+DtPent8rZ7GpKOhjdZY1dejCnVl6ZcbJPKY/K/T2Na2KkzasJSqMmnBcmqd7qNnbkwySuW/iWLr9M9qaTbshEmqxSvpEzwPt2jJx/EveuSUKvPU1uN4jjig8i1sgI9pXEjZ4Pt/9esCuqmuVEPUkLF2GTknjJNNukijkURSeZ8oLHHG7vj2pKZJ1qpagiwOlLTEOVH0p9SUFLSUtABWhoD+Xrti3/TUDr68f1rPqexfyr+2k/uyqf1FD2BHqmR7d/4venjuBjuPvVDKXzgMw5I4x6ZpyyA7Cc/MfQelYGhOCc9e/wDe9qVTwOvQfxUxWOMn0B+7TgwK5HOQcfLQA/k5GG7j71IyhyCyZwQRkg4oBUsRxkHn5PUUgxj+HoP4aYDsHGMNjBHUUvzf7fY9qarKeRsxuI6YpNyYAJTJUd/TrTEPG7I+/wBT6Uc7f4/ue1JlQf4PvevtQMYH3eh/ioAVxuDBtxBIyCKbApijVPmADHoPXJ/rS9QTweB/Ead3PB+9/eoAYdxkiPzYBbPy9qZOT9nPzN9zP3fpUwzkYDfeP8X1pFH3Rtbv3oAM9eT93+6ajct54OTjyiCdp9RUgB+Xh/u+opMnH8fKUDEmCyqyuQRgHG0+9A2xoURgFCYAwfenHODy/wB30oJOT8zfd/u0gAtyfmX7vpSbhkfMnK+lLknPzN93+7/9agNyPmP3e60AIrDKfNH93/CkBXEfzR9P6Uqtynz/AMP936UK3yp84/759qAGgj5eY/vHt9aCRhuY/vD+lLu4Hzr949R9aQnhvmT73p9KQFbV/wDkEaj9z/j3fp/umvLF6CvUtWI/sjUeU/4937f7Jry1elaIkdmkJopKoCKkoFLWJQlPUgimnpSA4NADiMGrCESJg9ah+8KEJRs0nqNEy8HGeR0roLKf7TCCfvrw1YJG4Bh1p4vjYgyR4LsMBTU2vsEtjS1jUlsoTDC2bhx1H8A9a5+zsJrwPICqRJzJK54FVndpHZ3YszHJJ70vmyeX5fmNs6bc8flXTCPKjFu4MArkBgwB4Yd6fGilSzyKoHbufoKipRycDk1YhSRnjp70x6kZNgGTye1MfpQ1YESRcoKko0+3ku5GijaNWClvnbaDjtVuXTL2LaRF5gYZynOKm5RVpaWeGe3IE8LxkjI3CmyiSHb5kbLvUMue4PemIWjO0g9xzUfmj0NBlBHQ0WA9YVg6q3HzAHp6ikjA2jcqA8dBXL23jK0jt4o5La43IiqSGHJAqwnjPTj96K6X/gIP9ax5Wa8yOkCpjovT196kCr2Cjr0aueXxhpJ6vcr16x1N/wAJVo0gwbuVBnP+qOen0o5WK6N4KpOQD1H8ZpdvHV+h6PWJH4h0c4P9q9McOh/wq4mu6Q/3dQtu/XiizC6NAJtPHmD5v73tSbCQDukyFIB44qsuqaa33b60PT/loKe11byACG7tc8/8tRRYCdl3Ags5BI4IzSgNkfvG6kfcFQh5GGVlt24HRj/jU8avgbhlt38LUANOSv3wfl7pSbQhYB05YHlal2tjkP0PekbcM/f6D0oAjK7iDuiG18jA9vrSEEjbuj5JGQcEU8k5PLfeH8NGenP8X92gYiB/l4ToejEUBXLIQo4UjAc89KUYyvK9SPu0Lj5f9X0PakAo345Q8r2emoJFRQVZiExkv1pVCnbxGfl9aUD7vA+7/e+lADD5hZMK4AHIyORShmyvyScr6inKDlfl/h/vUig5Thvu/wB76UANDttXAkzjjKihHYJHu8wkdTtFKuRs4fp60AnC/f6n096AAScDl/vf3DQrsUYuwB3dMdu1AJx1f73p70Ena3zN17r9KAKusN/xKNQ+ZT/o7dv9k15ateo6wSdI1D5j/qG/h/2TXlwFXEli5pKWmmqAiB5p1DDuKQGsShcUhFOFIaABGwakIyM1AeKt2cT3cvlRlA+CfmbAwKGguRfaRAhzyewqizNI5J5Y+laMemMUee8lEca9QpyfYfX6ZqecJpahFQLdONwTr5K9s+rH9K1hFIzlK5QNoY1/fHEh6RjqPr6fSoCrL1BqUuSSepPel3GunlRlzMYkRbluPapTtiTIHH86QNUMrFn54A6U9IrQWrYhJY5PWkb7tFBHBrI0GhchcZJPYCnrLKv3ZHA6cE0kMpjCsCA6Nlcip49QmjCBSmE6fKKyuUV2kd2yzFie55oaV2A3OTgYGTnAp3n8ltiqdpA2jHJ4z+WabHJtHzFvw9Kd2FhN/uKCSDjjinSSq0eFYknqCoph6gY6AUXYNC7qXd7U2incQ7d7UbvY02ii4D9wq7pN1aW155l7B58Owjbtzz2Pas+immJq6saa3VmZ7k7CI3ZjED/CO1SmXTmTgyB8egxmsekobuC0NTdB2KfnT1kUfdkI+jn/ABrJoyaQ7mvLeXEMe6K7mUjsJW5/WtDUbu5sYbVrbXZLiSUfvEEhxH0756dfyrmNxo3e1A7nVXOoazaXtvAuqNKLhgEfdkHnGe/HpU17quv6ddwQS30chmPyMu0jrjniuQ3e1OaUsQSTxwKVtbhfSx2M2v69b3dtbpPBcSXBHl7UXBJOMdsc0688S6/p11Hb3UEBlcZQBM7snHGD61yDXkhnScOUlQgqyjBBHQ1Jc6jdXk8c1xdPJLEMIzdV5z6etNpX0Ji5W13OsfxhqtvcJbzWVsZTgKADzn3BxU1z4vv9PdEvdLijJHGWPI/WuNn1C6uZopZpw7xfcOAMc59PWpNS1O61R42upFYxrtXkUpL3tNi4v3fe3OqTx6nG7T0OBjiT/wCtUqeO7X5d1i4wMcSj/CuOjvZljVPKtZAowN8SMcfWmSXDuBm2g+9uyseM9eOO3P6ClyoOZncp440/5d1rOMejKf61KnjTSyBmO6XB9Af61wEsnmMhFmsYXlgob5qLiWGRAIrMQMDksHY59sGjlQcx6GPGOkY+aS4XnPKf/XqVfFuisCPtjrk94z/hXmDMhI2gKMUgI7UcqC56dqPiPSLjTL2OK+VpJIWVVKkZODx0rgdyn+IfnVCk/CmlYLmgSPWkNUMfKTkcdu9PmEQYeTI7LjneuDn8zTC5cpjLg57U+nKN3XpXOWRAilpW2k7EA9zTZMIuaYDJGCqTVu2WK5uglkhiVU3S3E3OwDqQOg/nWW7lz7VrSqIvCkLR5Vp52EpH8eM4/AVqo2M27ll7mO3RL5kPlpkWUL8l27yv61iNJJLK8srF5HOWY9Sa2fFC/vrGTs1suP8AP41iitILqQx2aTNJmkzWlxWH5oPPBq5pzWy8zmPeT8ue39KqzSi4utyoqBj0z19yaVwsRkYo7VoXctjHbGGBd5JyTk8H61ll8fSpZSE2Y700gg0/O7kdKTFZjG5ozTqTAoATqOhx9KXNKrOv3WI+hpDkkk5JPrQAZpc038KKAHZop0JiWQGaNpE7qrbT+dPna2YD7PHKh773Df0oAiopPxooAdSU6KNpW2qUB/2mCj9anexnRS37plAySsqn+tAFeim5p2GCbzG23+9g4oASijcvv+dGV9TQAlFL8v8Ae/Sjj+8KAEooPHfNJmmAtFJmigYUYoooAUZHQmnB3HR2/Om0UAP86X/no350vnyf3s/UCo6KAJfPfuIz/wBs1/woE5/55xH/AIBUVFAEvmj/AJ4x/qP60nmJ/wA8l/76NR0UAaaLu+lJK/OxPxPpRJLj5E+93PpUYG0VzpdTVighFpI7c3I82aQW9qpwZWGcn0A7mri2Db0SVd0jDd5OcYHq5/hWibVFtpALVYp3QbfNePKj2RegHv1NaxXUzkxiW2l7sI9/dn0jiCg/zq3cQXl5p8VjZaTcRQxvvDSHJJP1xVJ/EGqOMC7ZB6IoX+QqrLqN7Mf3l3O31kNXZkm/4js5Y9E0ySVSskQETjrjj/61c1mmtI7fedm+pJo3VUdEJimkoH5VPNOhUpBEI0PU9S340xENJSbqAaBhjmnUgorNu4xaaadSUhje9JSmimAlFFFAhcmjNJRQMXj0o49KKKBBge9OBXYy4BJxhj1FNooAMe4pNv0paSgA2t6U8yTGMRl5DGOQpJx+VNoyexoAbg+hpKk3N60eY3sfqKAGcelHFSbx3VT+FG5O8Y/A0AR4ox708KHkAHyg1N9lz0f9KBlbFGDVj7I/ZlNH2SXsFP0NAFfB9KSrH2acdI2/CkMUy8mN/wAjQBBmlzT+RkMD+VJkeg/KmA3NLS/Kf4RSgKeMY9OaAG0Uu0e9GPegBKMUbfejb70DLoGBVnTy73JMSKzRjdvf7kf+0309PWnQWe+Brid/Jtl43nufQDuaoz3jvALaMCO3BzsHVj6se5rOMb6jkyxqF8rBre1dzETmWVvvTt6n29BWdQBQeDitSAzRSZozQAUZopO+B1NFwHorO21FLHrgUzdmtlbiCG1O1YtyoA+zv+NY7FWYlV2L2Gc4ouIQU4U2lpNjFJwKTJxSdTVi3iZ8v5ayKByu8A/l1pJX2Bu25DuNGTVprcANM0LxRKOA/wDE3YD/AD2qoKGrbiTT2FNFGaeI32K7fLGxwGNIojzRT3BTGcEEZGKERpTtjjLkcnaM0xDKKcdnTBHP6UBUPV8fWgBtGaf5ecYYUGFx2B4z1oGNpKcEYjIUkeuKaeOoNAC0lFFAgozRRigYUUUGgAopKKAFo5pKKAHCRx0Zh+NPE8o6Oajo70AWFvJh3B+oqRdQlXsv61TooA0Rqh/iiB/4FUg1GA/ftwR9Aay6MUBY1hdac334AP8AgH+FOH9lODkBfxYVj/jRQFjaFppb9Jiv/bT/ABp39j2r/wCrum/Q1h80dDQBtHQSQdlyD9VqM6DcjpLEfxP+FZizSr92Rl+jGpRfXYGBcSY/3qAH6hfyXzruASGMYjiXog/x96pmiiqEKrFelNpSMCgUAFGBRTSaQCk02jrTsUwE6UtSIoKkmo6HoAUHpRQagYCnEbnAVevQdakEH+jGbd0OMYqSyUr506kB4E3LxnnOP61SjdkuSSuQsm1nRmI2cY9TTVUswUdTSIxSRXHJBzzU8bFbe4mPzOxCZPbdkk/Xj9aErsG7IjjBAMhiMkY69QPzqRWtWY7451XrhHB/mKmthu0+YyFmjRgAgOOSCc5/CoVQLp8kv8TSCP8ADGf8KcRSBjHczMzyrB2UEEgAdBxVkI9w0RNzaokKhdyMEIX1xgEmqsk6yW0UawxxmPguB8z/AFNM2gWzOeTvAHtwTRdPdCadtGSPHJPK0yQyNEW/hXOB6flU7fZzdLFZxOVkOMXABI/L86NRszpskAjmZjLEJCRxjPaoReXQiZBcSbG4ILZzmjTqPXoQuAZSIxwT8o/lVqa2S2mMMkxaXA/1QyASOh/HFQRz7E2NDDIPVl5H4jFSRXVvGd4tAJV5UiQ4B7cHr+dCSfUG2uhEzPE7xh8hSRSrJI+VA3ZHQLmo0G+RVJ+8wGavJJJpFzvjKyEgr8y4GPzpW6jv0ISsoUTSW58s8Zxx/niod0ZIypA56VLFI0rXUrH5ijMfqSB/U02ziWW6jjbox/pRYEwYW5LbWYccA+tJ5SEqFlHPrVi52wXDQRxpsbj5lyRk4yD+FUnADsB0BNIZN9mclQpU5Gab9nlPRc844NSR2w3ruY4MZk+XjoKScPb3DxrIxCk859KAITG69UYfhTenWpRcS7cbsj3FON45ADKpxxQBBRmrayxyrgwKCO9TG2hbb8m3PoTQBnUtap0uIkbXdcjPPNQy6b5Y4lz/AMB/+vQO5QopXXY2M5pO1ABRSUtABRRRQAUvWikoAKWjvRQM/9k=', 'IMG_20261110_132921': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAA4KCw0LCQ4NDA0QDw4RFiQXFhQUFiwgIRokNC43NjMuMjI6QVNGOj1OPjIySGJJTlZYXV5dOEVmbWVabFNbXVn/2wBDAQ8QEBYTFioXFypZOzI7WVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVn/wAARCAEdAXwDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwCnS9qSlrjKClpKKAFooooAKWkpaADNFFJQAtAoooAWmuAyEUtFAGYBg49KeKdMu2U+/NNFUA6lpKUUgFpRSCloAcKWkpaQDs0opopRSGPFLTRThQBRuhi6+q1LbH5SKbeL+9jPqCKWAhC24hR7nFMRYrndRXFy/wBa2ZL60j+/cxD6Nn+VYl/dwSzs0bFlPfFXBO4Gno5/0Zh6Gq1+pJk7Yaq1pqgtUZREXz6tiorjUXnZj5aKD9TVqLuFxPLz3NTwLjOKo+c574+gpDI56s351Ti2CdjU68UuKykdkOVJBpTJI33nY/jU+z8yuc1CVHUgfU1EbiMN98fhzWdilpqmg5y410hcHkgegqRNQEe7bFu3DHJrPyB3FG9fXNPkRPOyxNevIhTbtHfFQpJIjbl3A03zB2BNLvbsv5mrSS2JbuSGe4bPzsM9ecUzax6tSbn/ANkUYc9X/IUASDgYoJA6kUzy89WY/jSiJB2zQAvmIP4hSeavbP5U8Ko6AU6mB1dFFFcZQUUUUALRRRQAUUUUALSUUUALRRRQAUUUUAVbteVaoRVy4TdGfaqqqaaAUClApGkij+/Ii/VgKhfUbJOtwp/3cn+VOzAsYpaz31u0X7okf6Lj+dV317/nnb/99NT5JMDZxTgK5x9bu2+6I0+i5/nUD6jev1uHH+7x/Kn7NiudXjHXio3ubeP788S/VhXIPJJJ993b/eYmm4FUqXdhc6C91xYnCWqrLxy5zj8Kovrl833WRP8AdT/Gs3ij8KtQiguTy3t1N/rLiRse+P5VCcscsST70c+lHNVYQAUAUYPrRimAvFGR60mBRwKAF3D0o3+1Cgt91SfoM1ILec/8snA9xigCPcfQUZb1qX7LJ32L9XFL9m9ZU/AE/wBKAIOfU0YFWfIiHWRz9FA/rS+VCP4Xb6tj+lAFXj0p2cVZAiHSFPxJP9acHA+6kY+iCgCqDnpzUixyN0jc/wDATVkPMem/8BS+XM3VXP1NAyAW8vdMfUgU4W7d3Qf8CzU4tpT/AAgfU08Wj9yooCxXEIHWUfgKeI4x3Y/pVgWnq/6VILLEYch9mcbscZoCxUxH/cz9TU8MUUiZLRJzjBqzb6eZ5RHEm5iM8nAA9TRc2f2WZopUUOPQ5oCxqilpMUtcZQlLRSUALRSUtAgpaSg0AGc0tJSigAJCqWYgADJJ7VnvrlgnR3f/AHUP9aZ4ikZNNCqcB3Ct7jBNcuFJ6CtYQTV2Js6KTxHEP9XbO3+8wFVpPEVy3+rhiT65NZIidjgKSfYZqZLC6cZWCQj124rTkigJpdYv5QQZ9oPZVAqo880n35ZG+rGpzYSqCXaJCvBDSqD+WaIIIDcxpNKPLL4dkBOF9RVWSAqcUVqrBYL955jhiMqg5GTjqfpThJYxZ2WzyYOQZHxkehAouFjJCk9FJ/CnCKQ/wmtG4uYJNoigjgAJPysWJ/OlhS4ZP3UUrLL8mQnDc9M/hQFjM2tgE8A9KNvvWjNpt3BAXuIpkijOBuwACfSqwWMfwk/U0wINoowKsgL2jH5Zp6rJ/ChH0XFAFUKT91SfoKcIJT/yzI+vFW/Knbrn8WpRayHqVFAFUW79yg+rUotx3lX8ATVwWnrJ+QqeLTWll8tQ7Sc5XpjFIDN8mLu7n6ACl8uEfwMfq3+FbNvpkTSMkgYOGCY3Y5z3NTS6fBHMsYEaEFgzSOCvGO9FwMIeWOkMf45P8zT1L/wIB/uoB/StiaGOBwEkjkBUHKDgH0p0whG3yWYgrkhux9KAMnZct2k/E4oFnM3VQPqa152t9kYhV1YA7yx6+lQGRB1ZfzoApCxk7uo/WnCw9ZPyFXkVpP8AVo7/AO6hP8qnTTr6XGyxumz/ANMiP50DMwWMY6sx/Gni0hH8Ofqa2E0DV5Oli4H+06j+tWE8K6s2dyQR467pf8BRcDBEEQ6Rr+VPAA6AD8K6OPwbesQJLu2T/dVm/wAKsR+C+AZNRPP9yID+ZNK4HKUZrs18HWKhjJdXLY/2lX+lTp4X0dGAKSScfxTH+houOxwmaTeo6kD8a9DTQdHQArYxNz1ILd/erMdhYxE+XZQLjHSIUcwWPNA6kZyMVKbx/s/kFwY+w2jPfv8Aia6nxqFTSrNVQJmXsAP4TXGYpoTLEd28L74nZGxjI9PSo3mMjl3LMx5JY5JqPFJigDdopKK5BhRRRQIKKKKAClpKWgBaBSZpRQBleJP+QfH/ANdR/I1hRRs0RIHy5wTjvW74j/5Byf8AXUfyNY1qqtExZ9pHIGPvV0U/hESDz4cxid0HUhXwKVbdJc+bdxg5x8xZs+/0pzJGFQqxZiPmGOhp0Ztxs8xJG679rAfTFWAi29kh+e4d/wDrnCf5mnYsVBIS7fnoWVR+dMkMZdvKVlTsGbJFSi6VLkTRxquOinkdMUAJ5lqANlgG95JmOfyxTvtJXBjsrSMdj5RYg/UmohOyhwpwHGGwO2c1q6dpGo6pYq0EsS229sLJJj5h1OMUDK8UmqtIhhDb5F3KUiXJXP09TSNZ6jNG0srSlMFiXkCg+vfrWsPC92Qgm1KBQq4UDc2B6dqsJ4MT/lrqBOM/djH9TSuh2ZypVWxuJbjjJzTgFHQD8q7OPwjp6n57m5fnsyj+QqzH4Y0hBzBLIcfxSNS5kHKcMCKPMUdWA/GvRI9D0qM/Lp0J5H3lz/OrEdokT/uLSzjTnB8vkflRzBY82TL/AHFZ/wDdUn+VWo9Pv5f9XZXLf9smH869JjZ1jw/lg4H3eBUnmHP3x1FFwsefR6FrEu3FlIMcDeyrj9auQ+GdaaTzCYonOcs0xz+grtgSSPmbqeg+tKgJKZ39CT2ouI5O38JX6gl723TcMn5C/wDP61Z/4RASY+0ak5wM4SIKBXRhdw+6cbe5pxjABwEHy0BYwU8IaamRJcXL8Z++B/IVZTwzo6E5t3fA/ikY/wBa1ieXUugO3j9aVjnd8/8ADjgfWi4FGPRtKiPy6dB06tHn+dW4raCIr5VrEnH8KKKfj73Ln5fSlA+YfK/3e5/+vQMVS2EwoHHrRub5eVHJ/rSKv3PkH3e5+lAUgJ8qDn+hoAQtkH94PvdvrSNjD/Mx47D2+lOycffQfN/X60hORJ+8H4D2pDDA3jhzx70wKCqfITz3PtUnHmfeY8dh/wDWpoAxH98/n6UANKkLJhFH/wCqnEN5n8I4pCo2v8hPPc//AF6Xb+8/1a9P60AMz8q5dev+e9GRl/n/ACxTlDbEwqj8f/rUNu+fkf5FIZy/jk/6FYDJP7xjz/uiuPrr/HZ/c6eMg/M/T6LXI1a2JYnam4pxpKYjZooorkGFFFFAgpaSloAKKKKBi0optKKBGZ4j/wCQav8A10H8jWDB9z8a3vEX/INH/XQfyNYMH3Pxrop/CIlpaKKsApaKKAEPSu68IKp0GNioJ82QZJrhT0ru/B3/ACAE6f65+1TLYqO5sqiAYCxjpT88HkdD0FAPA59Oi0vODy3Q9qgscM56nr6e1KM7f4uh9KUA56N19falC/L909D1agQ7ac/dPUdWpQvT5V79TRgZPCde5pRjjmMdaYBwAeUHAqQHJPzfxdhSA9cMOg6LThnPVvvf3fb/AOtQJirkkcvyT2+tKoyF4fkHvQoOR9/qfalUcKdrHC4GW/8Ar0yRNuVwqcBO5pxjI3YVB8v+NIVwpUIM7O5+tOZGG75UHy/40wAtneu9AwH+NKzZ3nf1XsO/NDNkPh0DAf570rHPmfvByOw780ADfxcv93sPr7Uv8XR/u+tIxzv+Zvu9h9fanH7x/wBYfloGIq8qNh+73P096aq8J+7X/I+lPA5X5XPy9z/9ekVQRH+77dz7UgEwQBwg+b+tDMdsnzoP/wBQ96UJwuI0HzH+vtSkMFfAUc/4UDG5/eH94OnYU0dI/mbp2Ht9KkJO5sso4H9femg/c/eL0/pQAw42t/rDlvf1pcfvD8rngd/r70uRtP7z+LsB60cb2+ZzwO319qAGBRtj/d+nXHpQV4k+Rf8AIpRjEf3z+fpQQNsnyt+ft9aAOT8ecHThwP8AWdP+A1yddV48/wCPmwGCPkc/qK5arRIlNpxpKANdWDoGByCM0VR0qYvCYieU6fSr3euVjClpKKBC0tJS0gCiiimAUopKUUDM3xD/AMgz/tov9a5+D7v410HiD/kFn/rotc/B938a3p/CSyalpKWrAWikpaAEPSu88G/8i+vX/XP0/CuDPSu88Gf8gBf+u79/pUy2KjubgBwOG7d6djg8Hoe9NGMD7vbvTvlwfudDUFj8c9B17n2pQFx/yz6HvQMbuqdfT2pwPH3h0PRaBCjG7qnX09qcpOPvdj0X3oyc/eb73932pR06v0Pb3pgO555bt0Wnd/4/vf0pvr9/+H+dPA56P1/ve31oEKvLLw/JJ+9/9ehV4XKHhMnLUqqSR8h7/wAXvQFyB8i8rnk9qZIhTCMSi7inc/X2pzEgucRjgf1o2gliVjAC8Z/H2pzOAWGYx0FACM2d5V0BGP8APWlZsh8OORxgUrNncQ6DkA/5zQzcPiQckdB0pgI5yH+ZunZf/rUrdX5f7vp9aHOd+Hb8F/8ArUrH7/zP93+79fagAx83R/u+tIo+58jfd7t9PenfxH/WcD6UijlOJPu+v096QxoXhf3Y+8ep+tG35W/dp97/AA9qUD5U+Vuv976+9BUYP7v+LufegAIbL/Ko+X/Gl+YFRlRwf6U0oP3n7telLt+df3aDg/09qBjcnaMuv3vT3+tIWG5/3i9B6e9OAbavyqPm/rQ28eZ93p/SgBmR8n7z8selBI2v8zHn0/8ArU87ty/MvQ9v/r03J2n5l+96e/1oA43x2f8AT7MZJxCTz/vVzFdJ45OdXthkHFuP/QjXNmrRIhpKd2ptAEFpN5Nwr9uh+lb3UZrmx6VtadN5tvtP3k4rnkUy1RRRUki0UUUALSUUUALQKSlFAzP8Qf8AIKb/AH1rnoPu/jXQ69/yCn/3l/nXOwfd/Gt6fwksnopKWrAWlptLQAHpXd+DMf2AOn+vfqPYVwZ6V3ngvP8AYPf/AF79PoKmWw47m8vQcjt/DTweDyeh/hpi5wPvdvSnAHB+90PeoNCQE7urdfT2pR93q/T0pADu6N1/ve1OCnb0bp/eoEP/AIv4/ve3pTgOOj9D3pu3n7v8X972pdvH3R90/wARpgP2nn5W7fxe9OC8/d/i9etNKjP3U6jqfenBQSPljPzf0NBLHKvK/IueTyelJtCocLHxHjrQoX5QBH0P86XjHWPATNMQrKMFR5eAoz+tOZlBf5o+1IWA3APHkgdv/r0p+837xByO3/16YrgzAbmEiYyM/p70FuWHmDqMcfSjftLZkUgsO3096XdjcPMH3gRx9KBiFh8+HPUfw9KGbiT5m6f3f/rUrH72XOMjGB9KRmB3/O34L/8AWoAUn5m+aT7v936+1AI3qMv93/Chjy/zPnb/AHfr7VGp/wBOYbnwIgen+0fb2pDHqeI+JPzoP3fuv97+97/WhcYj5k6ehqN2/ewL+8wzMT17UAPYcSfI3T+97fWlIHmD9327mg4xJxJ+Z9Kjjwb2YYf5VUdT7+9AxwUbEzGOvekK8Sfu1/yPpTlAKx/K/wCft9agtiXtWZlYks3U9s/WkBMV+cfInQ/09qQBtv3V+96+/wBKUgeYP3Z6HuP8aQAbR8n8Xt60wOH8bknXIwccW6/zaudre8aH/ioCMYxCnH51gVaJFNNpSaaaAKh4Oas2M3k3Kkn5W4NV/vLSDkY7isSzpTSVBZTedbKT95eDU9ZkhRRRQIWikooGLSim0ooEUdd/5BUn+8v865yD7tdHrn/IJl+q/wA65uH7tb09hMnopKKsB1FJRQAp6V3fgrH9hdv+Ph+p9hXBnpXd+Cj/AMSI/wDXw3bPYVMthx3N2Jt2M7OQp6+9SKVZNw2YIJHNRqORhmUhQOF96cisqhVfCgED5eazNCYFS+35MggkZ9v/AK1LlBgfu8lTimBWDllcliQPmT29vrTtp4bzWyqnGU4piJhs3f8ALP739KX5dvHlfdPamAMWy0r53dAuB0qqrRwz3IdpnZyGwEJx8qjPp2NAjQ+XnmPqP509SuR8yfePaoyHyf3zdR0jFNjDOzb5JDiQgfL0460xMmUgAfPGPlPb6e9O3DbgOv3cdP8A69QyM4MIErDccEhPbP8ASpANqttlYfIM5XPPNMkezYLt5ijp2pWYndhx1Hb6VUmBmtZWeVvlcN93ptft+VSGcmCWRZ5PkJ4KjtT9Q0tYsb8ggv8AxD+H6UbiARvz8w/h+lQhN8efNcSZHzkHPamkN54jaaQry3T0I/xoAsMQc/OR8wHC/Sgn7/zk8j+H/wCtTNhXIWZwN/QDjt7VGo8xpg8jMFcKMj2BP86Blhm+/wDM/T+7/wDWqKNla6lKyMcIF4HufanFcB/38nT29PpWbbyrBfTW3mv5kaNLkKQCWdzz68Y/OkM1VI/djc/3f7v09qhbm5g5k4Dn7v8A9anoFdss27qBlfTA9KqyhPtllEHYB97Ejg8AY/nQBbPR+ZOv936e1RxH/S7k5k/h7e1CkSgl2zjHG3uetQI4F7d7ZCoh2DA6HIyc+tIZbX7sZzJ+R9KrWf8Ax4r9/n2PrUqosoXdI3OQMHHHTtVa2nH2USCbKMWwMDHDcY/CgC4ceZ/H09/WmLjan3+vvSoArcyEtgZJ9aRGG2P95/L0oGcB4wOfEc/XhIxz/u1h1seKzu8S3mDnGwf+OCsetCANNpTTTQBUibtTyMHNQKcGrHVaxZaLWnTeVcbSflfj8a2K5wE/iK3baYT26v36H61LEyWlpKKkkWikooAWlFJQKBlPW/8AkEzf8B/mK5qHoa6XWf8AkEz/AIfzFczD0Nb09iWTUtJRVgLS02loAD0ru/BX/IDPX/j4bofYVwZ6V3XgrH9iNnH/AB8N1+gqZbDjudCucDhu3enjODw3Q96gjYNjIQcA/rUkbB4wwCDcpPWszQsDO4cP19R6Uozt/j+76io0IMhXC5BB+97f/Wp4IyFwMlSR8x7f/rpiJhnd/H9729KXJ8vHz/dPpSAfMPl/i/vH0oXlSACcAg4bpQIkPU/f6jsKcp+YcyfePpSAHJ+Vuo/i/wDr0AEsPlfqf4vr70yRy/dHzP8AdPVfp7UpYjdkv93P3fr7U3+AAhz8p4JH+NPxww/eYwBjNCEKc4bDPkkfw/8A1qGAwwZmIJAOV69Pagnlj+86jt9KTGTndJ94dvp7VQWHZ44Zvvf3f/rUnBYPubIO37vbI9vakOR0Z/v/AN3/AOtS5xj5n5b+7/8AWpDAtjrI33sfdpAAN5Vz8zgnI+lLnp87Z3f3f/rUpYcgyfxDt9KAsBbh8yD8vaoF5vbtjKMNEidP94/1qctw37wdR2+lRK+ZroiRf4f/AEH60DJIsqsatIu4Lzj1qu3zajav5q/JFJ29Sv8AhVoP8/8ArUPH+e9QBj9sj+dP9Sf5j3pDHQ7kWTdInMnH0zxVeJT5+pt5iZkYBfwjA/nVzcdv3l+9/X61BAWP2o5T/WH+QoAljYpHHmRMquT/AJzVGzidNJsowyZUKD9D171emLBGPy8IxqO3DC0gGF6CgCTcd7fOvQf196RC37v5l/L2+tBDZf5V6ev/ANakVTlPkX/P4UDPN/EzbvEd/wCzgfkorKNaOvtu8Qaicf8ALZh+VZxrQgQ02nU00AUW4NSxNkYph5FNU4NZlEx4NXtMm2SmM9H6fWqR+ZaFYqQw6g1AzoqSmQyiaFXHcc06oIFpaSigBaUU2nCgZU1j/kFT/QfzFcxD0NdPq/8AyCrj6D+YrmIehrensSyWlpKK0AWikpaBAeldz4KP/Ekbr/x8N0+grhj0ruPBeP7Efp/x8N3x2FTPYqO5vqBuUZkB2jp0xmliDiNVLkAKf4eR7UL0/L+I08ZweD0PesjQVd6zsQQ2dv3l+tOO4SRsCjfKwOUx6f4U5c7v4uvqPSnqTt6v09BTEOUknJ8ofN0I6cVFAGSS43tBmR8qCccbVHH5VYDHcOW+96e1NMcbHeyqX2/eKc8e9AiTD5P+pPzD1oijkQn5EO52bhyP6U4kEnlc5HUUoK7hzH1NMTGNHK3lfIuEG7AkPpipS0nzfum6f89OvNIMbM7Y/uZ4NPK8t8g7dGoEQM8kESho5CWmUH5gerD39DUskrIrMYpgFbJ5H+NLJH5iOpDqWYcq+COnIqOZPKsZUHmE88lsn19aYEqyOMbkkyW7YNN84+ZsCSk5zyo96lB5Aw/3j/WoVz9q6yfd9PdvagB4lHBBl5bOTGf8Kas6tnDOFD4JKdOalDcj5m5Y9V+vtUNs2Ff5+s79V9zQMk89Pm/egDcOqH2qG0uBcQyTghBKAwVuuMfWrAfGf3i/eHb6e9Z8UcsEk+Wg8mJEWPHUAe+fSgC95rFyV2so44I9frUcZZroEqBiPAz3qVM7Byh+QE/rUEkRkuUyiHbCSPZsjB6UhkpbtsDYbkgZxyait1JinPlj5nY4Ix2HtT4AHjD+WnzuT/T09qgx5ZuJfLH+tIPPGNo/rQBJcugikBVBlCo9z09KdGgEMQ8te38qdtCREeWeI/Wq8LP9mhDqd5CkEcdR9e1AE7KP3h8vt7elKoHmKNh6Hv8A/XpuMJJnzD7k89KeuPOX7/f19qBnlurtu1rUD/08P/6EapGrF827UrxuuZ3P/jxqua0JENNNKaSgRSQ9qDwaYODUnUVAySJsjFKeD9ahVsGpz8y1LKRf0ybDNET15FaVc9G5R1cdQa343EkauOjDNQ0Sx1FFFSIWlAoArPv9VjtcxxYkm/RfrTSb0QEmsyImmSq7AM4wo7nmuZh6Gp5PMuUlnmfLAZyT+gqCLvXRGPKrE3uS0UlFUAtLSUUAB6V3Hgr/AJArc/8ALw3b2FcOeldv4K/5Ar/9fDd/YVM9io7m9Gxbrt6Ajg+tSAqRkFMEGogAHVf3gJUDIPHWlhDeQnzuPlPYf4VkaFpcbh93r2PtUig7Rx27NVZd4ncbgfun5lPv/hTtzCRAREQUPHTuKYi2Ad3RvvevtThnZ/H932qFWO4fu16/wsPSm28xcTBkYeW2wYJ5GAfz5oAuAkMeW+8P4f8A61OVuR83c9VPrUW/B5WUcjtSQzBlJZ2BDNwQOBnj+lMklBGzlk+53H0pxwd3EZ6fzqBrlQUAkGChyWHTg1O0iBiGkj6jr/8AroEOAG5vlXhgOD9KPL3D5kOC3TdUUcivFEz+WDIy8D1I6VKRjGEUDd64pgO29MB/vH+L6+9IDwp+fOT/AFpACCvyN949G+vvQCCwxv4bnn60AO/u8vjcf4fr7Ui4UYDkZcnke5oH8I3P1I6fX2oDfd+c/ePVfrQMXfwf3i/e7/X601ZN/njchw239B7+9O3cf6xfvd/r9ahhyftByhzKf6CkMnYZL/Kh+X/Gmjmfb5aZVQc/X8Pahlzv+RD8v+PtUUa/6bN8i8KnQ/WgCRI1VI1WJQM9AfrTGSOWFx5bbWbnBxyCPf2p6DiP5G/A+31qG1ObVSQ/LE9f9qgZOQC7ZD8jnn6+9QGP57cjeFUcjn04/rRduUglZWkBwozj3qTIBQbmHHcf/WoARj8r/M/X0/8ArU9D+/Hzn8vemFhtb5x97v8AWnI2JSdy8DP+eaAPJJm3XEzf3pGP6mmUgOcnuTmitCRDTc0pppoApUqmg0nQ1ADmFSRNkYpg5FNU7WzSGTEYP1rT0ubIaEnpyKzjyuadBIY5FcdQah6jN/FLwBk8CmPNHHB50jBY8ZzWHPrk7Ofs6rGnYsMmlGLkQzorbbcKx2ttyVGRjPuK40hUdhg5BI5qeTVL+T711IP93j+VVeScnk+projHlJHO2VNJF3pCPlNLF3qmBJRRRSAKWkpaYCHpXb+C/wDkCv0/4+G7f7IriD0rt/BX/IGk6/8AHw3T/dFRPYqO5vqRx93t7U4dOAOh6NUMT4IDO3Kg8/U1JGxaJWOMlc/drI0LCj5s4br6+1OA6N82QvHFQqwMjL8vBHfHUVIp6Lg5K54b0I/xoEWA3I57919qi8iIyeaT8xAJG4gZHHT8KepO4feHPt6Uqt8v3j93utMCYdeAPvdjRGm08hjlmYHPTJpmQTyUPzdx7U5QMjCr36H3oEOcFthJcbRnGM54I/rUnA3AcDI/g96j5CdG+4OjU/JBPLjkdgaYiBoAIVztZ0kDYB27vrz70siD7HuKhW3Z27uAc8VOGORlj97PK+1R3J3wN8yH5vT3piJAh37duI8kja5HP+f51EkZWdwPNJwGU7+/P+FThASpCoeT0NRRg/aX+RhhV6N/vUD9SRBIdhDurHOQVBAqG2Z1giIkdvmZMFfTOD+lWFPKffHU+tQ2Zxbx/M3336r/ALR9qQx00kqQOUkjZgGI4IOefeo7ZyIcPtY5Xc5xyxAJqwJOF/eL949R9feoUtoUD7VUh2GQWJ6GgB5EhDHyosEdMkH+VNjwt5KGQKzBSAG7cj+lSMOH/dr+FNeMFpCFIZlAzn0zjvQACQAIQkpxx8vP9aZajbaRqS4IODxn+KnxrsWNfn4X/CoTFtjUIxG5yGJGMgkk9qBiXciNC6iXJJXjj1zVnd8w+deh6/hUVxl4JlSUKxGAcdKDvaYRlgyAE5PccYH86AHknb95T839aSQlUmYgfLGT+ho24jUbVxu/rUN6QljePsHywOeP900IGeTx/cH0paRPuD6UVoSBplONNNAFQ0h6U6kqABD2pWHemdDUi8ihjHwnI20rusXLd+gqu0nltxyahd2kbLHJoULiciSa4knCq7nYv3V7CpWktY4AsUbSSsMM8nQfQVVxjrRWqViAp+4YGBz3JpoUt0FPEYUZc07CuMPSiLvQxznHAp1p5X2hPtBcQ5+Yp1xUsaH0Vca1tXiU214u7JyJTtPtx+VQT2724UtJC+f7jgkfWgZFS0rJIkCTMv7uQkKc9cdaj8z2piHHpXa+DMf2PJnH/Hwev+6K4jeK6Xw1rljp1i8F1JIjmUuCEyMYA/pUyV0VHc7BT8w+UEYHIb36UqRhUA+bgHv39azode0mXG2/hHThwV/nV6K5tph+5uLeTIP3JBWVmXcnVWWVirHBx94Z5wf/AK1LgiRGBVvlIORjuKeqNkEKcZ6g+1MDyrkGJyMcEEetIZOu8kcJ16BiO1NiYoJAVfOexyMnFKh3YJXb83RlwelPXpwB0HQ0CH+YwPIf73pnt7U2CRPLO5kyC+QRyPmpwznow59fahAOMZOCzcjOCTTADIN6AEAFDnkj6VOzhS+SQcj+L3+tQkKxUnYSoyMjHYipERV4CDG4HhvpTELCzGCNiX3NjtntSzS+WiDIOZAMMvqcVFFCESNTvwDlueGPrSsmUypYNuyMg+tAidW5X/VnrSRsDLxHgrgEg9eM/wBaapLsoMi45xwMn61HEjpK3yB1IByOpPt+lA7WLKdU4cY981HbspghKlgpGeR9fakTzSI+ACVIxk8frTIJAsUK/vBxheB83agCwH4X94D8x6j60feH8DfN/WmLKcJnPU9gfX3qO2x5QU+WX3sSM8jLE/1oGTleH+Qdex+lB6tww49fr70zcjBsbT8w+630pkbEm4zvwGIHOeMUgJhwV5ccen0pobhfn79x9aSR9is25vlQnkf/AFqbG5McRLqCRk8e1ADyxKyYdD/+r604gmT7qniopJMDB2ku+0fln+lPx85yi9KAGgfKn7sdexqtqzbNG1FsEYgfv/s/WrA4WP5WH0Pt9ao68wXw/qR+b/VEc574FNAzzIdKKQdKM1oSIabSk000AVqMUlKTgZJqAEIz060gExcxRxuXBwRjkVYgXN1GLVzLKvzZHyqnqST2FXDKHR5ZZ3FqDh5Rw07f3U9veqSJbM1LJyGeV1SNThm64Pp7n2FLujTiJSo9T94/4UlzdvdSA7RHEgxHEvRB/j71FmtYksewVxz1pFjUe9NzShqrQkkJCCq7MWOTTnyeaZUyY0gPSlVSwRVGSx4oPQ0itheCQQeDUMoPLkIBCMQxwMDrTRntUqXMyY2yHjpnmk8znOACBgYGPxoAZuPrRmlU8cn9Kc23ZkbTn25FFx2GZozQ3GPpSUCF4pNooooAmiuLiE5huJY/9xyKvw+INXg+5qM5Hox3fzrKpc0aAdFF401mL78kEoH9+If0xWzL4q1OyWL+0NKiAlHyEMRu78dfUVwhPFXLrVbq8WFbhlYQ/dwoHYDn8qOWPUTcrqx28PjS2J/fadOnOf3bhv8ACrCeJtDlbdI11A5zyyN/TNcKuqkuzzQRzFgAd4z06dMUj3sMj5WMQjGNq5x9etLlRd2ekw6zpEv+q1dBwOJHx/PFacc0E+TDNby5P8Dg+npXkfmxt/Ev404JGTkBSfUUuULnsGxh/AR83Y0cjA+ccn3ryzSFvrnVUtLbUHtgQWLmRgFH4GtCPU/EEOrXNjDqHntbAs0jEFNoxk5Ye9LlC56CpI2/MPxFOX5ghAQkDtXB2ni3Xjcvb/ZUnmiBLx+SdwA6/dNWofHMgl8ufTFMi8FVcq35EUrdB7HZrwV+UjjPDfShOAgBcDHpmuYh8bablRNbXMRHHylW/qKvQeKdFk2/6Y8eBj95G3+Bosx3NoPwvzj8RTBErIAQj5kLfmTVaDVtNm2iLUrVj6GQA/riraYkVShjkGTypz60rDBkUg5iH3h0/CohGym4K705+UZyuNo7fXNTFCP4CPm7GmtnD/fH69qQCOgIYNLIcrznv+lMhklMUHMZLID0xt4+tS7sOfm7dxSAkhOVPFAEM4cNCzCN/wB7gADGSc1IHJdgIjuAGecAde9KRx9xfvdvrQT9/hhx6/8A16AFBGI/vjj+lZniZ9vhq/8AmPIA5H+0K0c/MvzN0PUf/WrH8Wvjwzc/Nnc6D/x8f4U1uJnnnaikzRWhIGm0pNNzQBBIQnWqzuXP8hVsWkRBNxfwRP3TBcj8uKfAumwTRySXck2xg21IcA49yaErCbLotgmoQaVH/q22tcN3c4yQT6D0rN1Cfz7t8cRISsaDoqj0rU0Vzd65dXe042s4z7nA/SsPOeT3prcQoopM0mau4h1TJbyuM7SBjOTVcHkZ59qsNeOYdigIfVfSi4ER4ODx9aRlIAOMA9KfbyrFIXcFjjp60s9y0+AVVQOmBSuBCelNwKOc0tSMbRS0UAJmijFKMDqufxoACSTk0UHGeMge9JQAtFJRQAtFJmloAKKKKACiiigAozRRQAu45z3qS3uJLaVZYHaORejKahpaANCx1e6sr+S8ilPnS5DsQCSCcn+VSxaxMdYfUJXCyy8MyAjHGMj34rJooWj5luKS5lZm9bapay+JBf6nCk8BzvVEBBO3AOD15wabqU2k3PiGN7QPb6c2zzNqlSP7xA7Vh80flQFtLGxqUdmupQJZ3Ktby7d+1iwj5weW9uaveI9Mt9GFtJYXZlE2c8jK4x3GK5rPtSlskZJOBjn0olrK5UdI2epuWt/rqQLLbyXxiOcNG7MOPzq0ni3XIOJJ5SP+msSn+lc7HPLF/qp3j/3WK/yqw2o3rhw9y8m8bWLsGJGCOp56E0WC500Pj6/X/WRWsn1Rl/kavRePo2I8/T1OBj5Jv8RXFNeStbGJ4YWyMB/KG4D6inW9zZJCsdxYCVhnMizFWOfbpxSsh3O/i8aaXIAHguouc8AMP0NXI/FGiS7h9uMbHtIjDt9K8rXZ82VPtz0puBS5UHMz2OLVdOnYeTqVs/B48xQazPGMgPhw7ZEcNOgypz6mvL8U5WZejEUcoXL9JVMTyD+LP1FOFy/cKaoLlkmkzVcXOeq/kaUXCnsaBE41aRP9Xa2afSEUv9t3o+6YV/3YlrOoosI0xr2o4x56kenlr/hWaKSimAtPETbQxwqnuTUeaCT3NACnGeOaTNJmigBaKKKQBRRRQAlFLSUAFGaKKACiiigAxTkdo3DoSrDoabRQBNJcvKpV1iJP8XlgH8xUFLRQAlFPyPLK7VyTndjn6fSm4oAk8ibZv8p9vrt4qOlUspypKn2OKGZmbLEsT3JzQAnFGPelRlVsugcemSKaevpQA7BpKT8aXJoAKKM+1GR6UAFFFFABRRRQAUUUUAHTpS72/vGkooAduPcKfwo3D+5+RNNooAfuT/aH5Gl+Xs/5rUdFAEm3PRkP/Asfzp2x9mPLzz1HJqGigZPNcTTOrzsXZAACw7DtWhcavb3TiSbTbfeBt/d/IPyxWUHdejMPxp4lbuQf+AimnYTVxlFJRQAU6NfMkVMgbjjJpmaVHaM5XGfUigCzJaeXGx3b37AdKqVLLM0oUHgAc+9R0gCiiigAooqQLgcpuB7g0WAjop7oVQM3BboD1x60yhq24BRRRQAUUUUAJS0lLQAUUUUAFFFFABRRRQAUuaSigBePSjikooAXHvRtNJRQAYIopQx9aCc0ANoqQICOvNBiPYigCOinmN/TP0ppBHUEUAJmilo4oAKKMUYoAKKMGkoAWlpuaWgApaSigYtKOlJSg4HSgBpNJU8Fv52Pm25OOlTahAls6RIOi5LHqTQIpUuKKeOKYDMUUE5NFIApKWnxpvzzjAzRuAzt0/GjBA3dPQ0qLudV6ZOKWRycLwAnAAp20uAjElssxY+pOaSgnhQBj39amkRUuPL6hePrxRa6uF+hEq7gfmUH0JxUqxux3EbIwOWHSonILHAx7UjDHFNW6idw+83pk9+1O2AOV3BjnAK8g0gfC4Kqw9xz+dPEiLGSiFXb5c7s4HtQkn1BsipaVDskDYVsdmGQaeh3yegwSQPpSSuMjopM1KIsg89P8KQEdFJS0AFFLikIxQAUUUUAFFFFABRRRQAUUUUAFKGI6E0lFADxKwp4nx1FQ0UAT+ZE33lH5UoSFuhx+NVqKALX2Yfwv+YphtpO2D9DUIJHQkfSnrNIP4vzoADFIvVDTeRUwumHUA1Yjk8wcqPx5oAoUYFaPkRN1QfhxSGxjb7rMv60AZ+KMVPPb+SSN2fwqGgYnNFOHWlK0Af/2Q==', 'IMG_20261110_132723': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAA4KCw0LCQ4NDA0QDw4RFiQXFhQUFiwgIRokNC43NjMuMjI6QVNGOj1OPjIySGJJTlZYXV5dOEVmbWVabFNbXVn/2wBDAQ8QEBYTFioXFypZOzI7WVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVn/wAARCAEdAXwDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwCrQaKK5RBS0lFADqSijvQAtFFBoAKKTNLQAjruQg1n4wxB7Vo1UnXEmexoQDBThTRT6YCiloFLSAUUopKcBQAopaQCh5YoRmWRE/3mApAOFOFUZNYsIv8AlvvP+wpNVJPEcC/6q3kf3Yhf8apRb6AbVKVypHqK5qTxFdN/q4oo/rljVSTVr+Xrcuo9Ewv8qpU2M6BRti5OMHHNOa/tIh89xHn0Byf0rkWZnOXZmP8AtHNJiq9kB0k+tWmxlTzJMjsuB+tYr3OWyEx9TVfFAFUoJAXzqt15YRXVVAxwo/rSRanPGCCRJn+92qlkDvSbhT5UBcl1C4lGCwA/2VquzFmLMck96j3+go3H2ppJASUVFlvWk+uaYE2QO4pPMX1qLApaAJPN9AaTzG7ACmZqRIpX+5E7fRTQAmWP8VJjPUk1MLWbuoT/AHmApwtsfemjH0yf6UAQBV9KUACrAghHWR2+igf1p2yBf+WbN/vN/hTAr5oHPTmtG4tltoYpQsJEgzwM4qaSNYmVVkWTIOdoGB78dj70AZYjkboh/Kni2lPbH1redrIaaFT5rk4OcHIP8We2OmKSzu47WWWQQCRiAEyOnr+dAjKtdMnu7hYIVLyN0Aq5c+HL22kCSQuxIyCg3DH1FW9O1KXTr57qOGM7wVKHgAH0qW813UbmbesiwqBgIg6UhljNFIKWuUQtFJSmgAopKWgAopaMUAJS0YpcY57UANqOdcpn0pJby1h/1lxEvtuGapza5YKCA7yH/ZT/ABppNgSingVjPrijPl25P+81V5Nau2+4I0+i5/nV8jA6MChmSMZd1Qf7RxXJyX13L9+4kx6A4/lUB+Y5Ykn1NP2YWOqk1Oyj63Ck+i5b+VVn1+2X/VxSuffAFc7gUtP2aGbEniGY/wCqgjX3Yk/4VlTyvcztNKdzsck03B9KMGrUUtgExRil2n1pdopgNozTsD0ooATPtRzUqwSv92JyP92ni0l/i2L/ALzigCvg+tGKtC0H8U6/8BUn/CnC3gHVpX/Jf8aAKeBRxV4JAvSEH/eYn/CnBwp+RI1/3UFAFBVZ/uKzfQZqUWlwf+WTKP8Aa+X+dXs3Eg480j8cUq2k7f8ALPH1NAFIWb/xSRL/AMCz/KnC1jH3pmP+6n+Jq+LCU9WQfrUg04fxSH8BQBnCG3X+CRvq+P5CnARL92CMfUE/zNaS2MI67m+pp4tYV6Rr+PNAGYJnHCEL/ugD+VGJpO0jfXNa4VV6KB9BS0AZK2sx6R4+pFSCxlPVlFaNGOKAKIsR3kz9BViDSjNuEUcsu3lsdqtT3PmxxowjXyxgEH2H+H86INQktt/lSAbzuPy5wfUfnQBWgsI5HISNcgZOasWdj9ovUtV2xliQTjpgEn+VQrcbCShYE9cUwz85wc+pNAF2+shZXkluZFcoM5Hr6fWk2W8VnuciWaUfKFbHlYP8VUvNPYCk8xvWgDU027tbWOUzwmSRvu/KDx6c9Ko7hknaBk5x6VXLMepNNPWgDcoFFFcoC0UUUAKBUNzeW1mB58oQt0GCSanFchrLvJqs+8/cO0D0AqoxuwNmXxDaL/q45ZD9ABVSXxHMf9Vbxr7sSf8ACsUKfSlCk+laqCAuy6zfyf8ALbYPRFAqpLNNMcyyyP8A7zE1LHY3MmNsEpB5ztOKcbJlUM7wqCAeZBn8hzVWSAp4FLV2zitmkIuHZV8skFVzhuwqzH9hUKWiuGOOQGUc/WmBlBWPRTTxDIe2K01vI4ivl20AK9C43k/XPWmyyy3TqBFyowqRRYwPoKAKQtJDGz9VTqR29KY0W0KTjDDIwc1prZ3LRhjFsRiVy7BcEdcjqKZPp/2WFneW2JBACLLuZvpj0oAzsClCk9AT9BVhc/woB9Fp4WVuzfnQBXEMh/hI+vFOEB7ug/HNWBbuepUU9bX1f8hQBVECd5GP0WnCOEfws31b/CrgtY++4/jUot41UHYuDxyc0AUBsHSKMfhn+dPVpcfKGA/2Rj+VakcUQmjVnRUbG51H3c/4VZiEaXTrC+9QvysR9M/1oAxRBPJyVJ92NSLZSnqVWty7liEaFmLpuBKGQZOBz06VWkuI59iwwIhXIPl5Ytn1+lAFEaccAu7YPIwuM1ONMVIlldJDGxwGPQmtBbfULmONEs7lwudp8oj9T9KtR6HrU0aobYpGDwJJFH8s0AY62sC9I1/HmpVVV+6oH0FbsXhPUXx5k1tFk46lj/IVci8H/wDPbUGPOMRxAfzJpXGcvmjNdknhXTYxmSW4kIOPmk2/yxVpNB0iLcRZo5A/jy/86LhY4EyIOrKPxp8aSSn91FLJ/uRs38hXpEVpawNiGziTj+GMCplZsJhQOPWi4WPOo9J1KX7lhcfVgF/mRVpPDOrPkmGKMD+/L/gDXd5fbzgfN/WmseHzIOnbHpRcLHIx+D7skebeQJ/uqW/qKtR+DoQAZb6Zsnoiqv8AjXSHBccseD0z7elNwNq/ITz3/wDr0rjsYyeFtKj3bxNIR/flI/lisTxhp9nYLY/Y4Vj379xBznG3Ga7XBG/Cgfj7Vyvj3O3Tc46yfyWmhM4/FGKXFLimIbijFOooAbilxS0UAJimkU800igDaFLSCiuUBaKSigBy1yuuLs1eb/aCt+ldUtc54kTF/E/96PH5E1dPcCgqM5UKvPTjvVhJbs4jjkZewCAL+tNiEjQrJztX5QfSnSwlCA+DkBuDnrWwAtrJMis80e0jI82bFOW2tFP728Q89Io2b9TgUscMBK75dgK5JCZwfSmuEDMEYsoPBIxkfSgB4WyUD/j6kJ6gBUA/nS+ZbrjZYofeWVm/linCaKO58xIw0Y6I+PT8ajE20OFwA4weO2aAJvtkynMMVtBzwY4Rkfie9Sx/2nIUEck5MwJXYQNwH0+tS6dp+oapastsyfZ1k5DuAN2Pz71fHhe7KItxqECKuQqjLY+nSi47GUNMmkzJM8SDoWmmH5VVUIuMAfhXVQ+D7bAMl9IwP9xAv881di8MaUgywmlOP4pD/SldBY4sMKPMXONwz9a9Bi0bS4j8mnxHB/iXd/OrsUUUQxFbxR8H7qgfyouOx5xFb3M3+qt55P8AdjY/0q7Fomqy422Moz/ewv8AM130hlbBWbysYzxnNKmVJzO75PQ444ouFjjYfC2qPjcIIu/zSZ/kKuxeD5yMy30S8Z+SMn+ZrqV5x989R6UoGV+6T8vc0XCxhR+ELRP9deTvgjptX+lW4vDWkIeYXlOf45GP6CtXBG7hV4Bpxbr86/eHQZ9KVwKsOlabCQYtPhBDdTGCfzNXowIwoSJUGTwMD1pgwe7H5uwxTgOnyn73c/8A16AJNz8dByf603eMgeYM7j0/GgL0wij5vx/lS5I6sg+agQdhy5+Y9j70Y6/IT83c0ZGP9YT83YUvHzZ3n5h6j0oGGCAxCqOR3+lDEjfl1Hy/40Mow/7v8yKcQwL4VQNv+NACAjePmJ47D/61NGMR8Ofx9qfk71y6jj/D3poIwn7wn6D2oATbxxGPvd8etOIbEmAo4/pSYG3/AJaH5vcd6CoO/wDd54749KYASd4y6jg/0poIKj5yfm7fX2p4B3LhFHB7/T2pPm2jJUfN6e9IBpA+fhz+fpXK+OwAmm4XHMn8lrqyeX/eD8Melcr47xs03knmTr9FprcGchS0UtUITFFLRQISilooAbSEe9OppoA2KXNJilrlAKBRikoAcKxPE68Wr/7y/wAq2hWX4jTOnxv/AHZB+oNVDcDDhY7BUmTUMJ+Wpq3AMmlpKWgApaKKAOq8FENBeoQSUdHGD0yD/hXSiNATiNRnOc1yXglv9LvY+u6IN19D/wDXrr+A38I5qHuWhQQOhUdOlOBz3PfoKYDx94dOwqQderdfT2pAKP8AgR6H0pwXkfL3PU00D5ejfd9aeFwfur97uaYAOF/gHy1Jnk/N3HQUxegGUHBFPBzn5ieAeBQIcvUcueT7U5VzgbGPBHJpAOern5vp2p6ryPlY8nqaYBtO0/Kg+Tv/APqpxbBb51HQ0ipwPkQfL3p5OFb5kHy+lABnlsufvDov09qd6/fPzDpx6UE/ew57H5RmlP8AFzIeR2x6UCAL1/d/xDqadgjPyovzD+lG3r8jfeHU/T3pdvXEa/eHX8KAAnrmRR8w7fT3oP8AF87HkdB9Pal5AblV+Yf0oZvv5kXt0FAxHHEnDnj1pSgLt+7H3e5+tDniT5nPy9h9fanEDceHPHrigAUMCuFUfL/hTeQEy6jn09j70qqPk/d/w9z9KApAXCKOT/X2oAbkYP7zPzdsetKcZflzx70p3ANllHzf4UhYZb94OnbHvQAmAWT5GPB6/h6mkA+UYjA+b29aXI3J8zHg9B9Pak42jhz83v60AOO75+AOPWuT8eZ2abnHWT+S11RA+f8Adnp3xXK+Ox+7035QOZP5LTQmcjRiilqgEooooAKSlpKBCUhpaaTQBs0tIKWuUAooopgKKpa2m/SJv9na36iroqK+TzNOuU9Y2/lTW4jj4T1qeq8PX8KsCtxi0tJS0ALSUUUAbfg5wuuFTj54WHP4H+ldvkZ4K9ugrgPDL+X4itOcbiy/mprvyfdunpUS3KQo6dW79qeOv8XamDr/ABdacBx0PQdWpDHhePu+o5NOGB2QdDTQOeVXr3NKCAOqD5aYEityPmXqelOB+X7x+72WkDc/e/i7D2py9hl+hHT3+lADu5++eR7U8A5HysfmPVvamDkHhzwO9P28n5Dww6mgQqr935F7jk04HCD5kHyUirgj5EHzH/PSnIcBeUXgimApbIb5wflzwOe9Ob+LLSdui/8A1qQNlf8AWLgp2GaceVbDP93sPr7UCAj73yv26n/69KV+9+77g8nrQw5fh+mev1pWH3/lY8d2/wDr0ABUgP8AIg6f56Urkjfl0HH+e9Iyf6z92vTv/wDqpzAjzOEX5f8AGgBHYZf94Pu9gPelBBcfM549P/rUrMct86D5f896QMCynzB93sB7UAIoB8viQ/L6n296TaPl/dZ+Y9ce9KpH7v5nPHYf/Wo4wOJD8x9fegYYIVsIg+b1+ntTjv3N90cD+tMYDa37snnuR7e9Lt+dsRL0HXHvQAmeUzIo49Pp70mRt/1n8XbHrTgG/d/Ko49f/rUEtt+8o+b+tADTjL/M549D/hXKeO/uabjPWTrn0WurJ5fLr0Fcp475TTfmzzJ/JaaBnJUUUVQgpDRSGgApKKSgAJpp60pppoEbYpaSlrmASlFJS0ALS7d6sp6MCKbT1PIoA4aL5Xx6cVZFR3K+XqE6f3ZGH608V0AOopKWgBaKSimBb0mTytZsX9J1/U4r0hs7iPm7968tR/Lmik/uOG/I16k/Lk44Jz96okUgAOeh7fxU4Dj7q9D1NRjAHROnrTwRnqnX0qRkgwD/AADkU9T23DoRwKjVuPvdh0FSKTkct1I6UwJAf9pux4Wnr1H3+pqIHK/xn5RUg69GPzevt9aBDgPl+6/3f73/ANepNvJ+TuOpqNVzj5D0I5an7eCdi9B1P/1qAHhcH7qD5v6fSnKQCvzIOTTMYJ4jHzCnhsH76D5iP880xDlbIX94M7T0H0pQcr95jlOy/wD1qRX5X94O/QUqnIX52Py4+79PagBW53ff5X/GlYZ38Ofl/vfX3po5A+/9z0peDniQ/J6//XoAVlzv/d5+XufrSsn3v3afd/x9qQqCT8jHK/3v/r0uzJ/1S8r3NADuQf4B8tIH5T94g+X/AA96Om0lEGRj+Xt70KSPL5QfL/hQAisMJ+87dhRkYHzOfm/u+/0pFkyVHmpkHkfgfelLDHMo+92HvQMQ4w/+sPPuOwpcAufkc8Dqfr700OHiLh2wwBHHqB7UruqONxk+bCjg9ST7UAIAP3f7odO+PSjHy8RoPm9ff6UoAxHxIePU+lRlh5oj2Mc5br/tAevvQBJ82X4UcCuU8eZ2abnHWT+S11RUbn/d9h1xXKeOxiPTflC8ydPotNbgzkqKQ0VQhDSUpptAAabS0lABTTSmmk0CNwUtNFLXMAtFFFABTxTKcKAOT1hdmrz+5DfmBUYq34iTbqKt/ejB/mKpqeBXQtgH0tJRTAWikooAbJ9w16fBKr2kMpK/NEjdPavMG6V6LpDmXQLM5OTAF4Hpx/SpkNF8MCOCO4+6aeGOep6/3ag3MJ/+WhUk9F9h/gafuYKSEkJAGBwKgomBOOrdPSpBnP8AF19qhVmOCEJU5x82D/Oli3gH923XPL+pzTAnXnIw2Qp71KB/snqOrVXRX80nYo3DHLZ9ai84wsEmkhQvJtUE5ODkj/CgC+q8jKr1PU0oA29I/uYqGTeiOwaIFckYX2qRM7VxMuMHsOaBExIG7mMdP508NyfnXhh2/wDr1WjZz5mZ8bfYdMA/1p7b/OQGZyCcnAx6YoAnDYK/P1YgcfWnL1Ubn7j7v/1qqXKHdDIZJ38ubJCj1UjsPUipotsiRuHnXJPc5HWmImXovLn5cdPpSqM7eH+72P096ijjO8EvKRjA+YjHrQkR81WLTbSmMbz1/OgCVVzj5H+7/e/+vSquSnyDJXufpTEiXKZSQ/L3kPt702G3RTzECG5A444HH55/OgBt2yRW6M6KBuQDB7l1FT7lTaD5SkDB56VBcWiT23lrCqsVyCCAQQQR+op9ukgVfNghUgnG054x9OvWgBscgW4kbdHglQMc5OD0p7zgIcuB839w04RASB/LQNkjP5+1O+YZ+4PmH8xQBXgmAtVXzM7UAO0ZxwKS4lVmTLsuHVjle2anH7uN1VkAA4H+TSkhmIZ0Ix0x7/WgZGJMiPAlPHBwBnioS/8ApcZ2ygMCvPb5gasB+I/3q/h9KQkEf6zkN1wPWgYpxlvv9PeuU8d/c03AI5k6/Ra6on5m+dunp/8AWrlPHZ+TTeSeZOo9lprcTOSopKKoQU2lppoAQ0hpTSZoASm0pppNAjdUggEHINLWdpdxlfIc8j7v+FaJrnAKKKKQC0optKKAMPxMnz2z+oZf5Vlxn5RW34kTNlE/92TH5isOI/LW8NgJKWkoqgFooooARq7zwu/meHrfqdpdOv8AtH/GuDNdl4OcNo8iHb8k56+4FTLYaN8DnODk4/ipwHHTsf4jUQZDkAxkjrTwRnGU79qgolUAYACgA+vtTgRt6J0qNSPVe3ang8dR37UASgru/wCWfX+lLtjzkrDuweSozxTQ3P3vQ9KkUn+8e46UAS7hk/MvUHp/9enq3Iw46noKiDHH3m6A/d/+tUgJz1b73932+lMQ9WymA55U5+Wn7uvzN0B+7/8AWpinoCX7jGP/AK1L1XJL42elAEmcFvmf7w/h+ntS56cyHDentTTkk/6zqDTsez/eHemIcD93755Pt60JxtGH7j73/wBek9OH+8ep+vvSjkL8r9T/ABfX3pDFQcplX6Hnd/8AXoUfcGxun97/AOvSL1T5XPJ/i/8Ar0qj7vyMeT/F/wDXpiBQMJiP8zShOE/dr1Pf61HMStuzBDlVYjn2NSbeF/d/xHv9aBhsxj92n3j/AF9qTacH5EHzD+ntTB/rjH5S4GGzn3IqQp1Hlr94f09qAEYNiThRx/Sly3mfwdKihy9vuMaZK/4+1LMWj2ssaZLKvPoWAPb3oAdubEfzL/kfWjcdp+dPvf1+tAVsJ8i/n7fSom3C5Vdq8qx6/wC0tAyQt8zZkTp/nvXKePD8mm/MDzJ0+i11h3Zb7vSuT8ek7NNzjrJ/Jaa3EzkaSiiqEJSGg0hoAQ0maDSGgBDTTSmm5oAZG5Rg6nBByK34JlnhWQfiPQ1z/wB0+1W7C48iXax+RuD7VgxtGwaKDSVJAopwptKKAKWuLv0qQ/3SrfrXNQng11moJ5mm3K/9Myfy5rkoepransMnoooqwFopKKACuq8FSfuL6PJ4dG4HsR/SuUrovBb4vbyP+9EG646H/wCvSlsCOqfeB8u8nLdB6nNSBzn7sh59BSFSSPvDDZ4b2pwzjoe38VZliq7YOQ/the1PV28zpLj0296aM46Hv/FUgznoeo/ioAN7fIQsuO/A5GD/AFxStM4laMJIDt3LgA0ozt6Hof4qWSPzSMtIoB/hfGfrTAmD8H5ZSNv92nI7bTuSYnd6ds8fpSIpAAO48EZLU8Dj7p6A/eNAgV3Ew/dy7e3Trz7/AEpxZii7Y588Z6dM896cAc/dP3v7x9KcgPHB7/xH1oER3Uzx20sixyDaobLMMAZ57+lWCSGb91N1B+8Pb3qMoHjZWUkNHgjceaWKARK6rvbJByz5P+eKYCmVvNVfKkwSSMt349/ekLsssa+U+GZh9/qfz9M1IV5J2c7hg7vpQU5BMeSHyDuoVgF3EFf3Mn3j/EPf3piu/mlPJbjlQW7c9eakwRj5P4j/ABUBTkHYfvH+L60WAgvWkTT7hkgbesbkZYEZwfepY3LRIywsQTkEtin4OBlD94/xfWo4bdYN2wOdzDO580DGkMLncYCSRgfNxwfX8akw43ZhTqM4fr09qcR1+U/eHf6UMow/yH86AIIFcQlPIX5QQST15PSidHKf6hDjDcNk4BB9KnIGW+Q9PX60gUbl+Q9PX6e9ADNsmE/cxD6nkcfSo2R/tCMYUxypIPTkHPT2qXAwnyHr6+x96QgbT8h+96+/1oGOKjc3yL0H9fauT8eDCab8oH+s6fRa6vA3N8h6ev1965Px5wumcY/1n8lpx3EzkqKTNFUIKaaU0hNADTSUpppoAQ000pppoAThhSD070yJ8HB6VIw7isCzW0+482Py2Pzr+oq5XPwytFIsi9RW9HIssauvQ0mQ0OpRTacKRI5l3xuh/iBFcRDw+D9K7K5u4rOLzJT9FHVj7Vx27dOWAwGYnHpWtMZYFFIKWtQCiiigQlbPhF9uubeMPC45/A/0rGrR8Nvs8QWh/vMV/NTSew0d6SoIyUBOOopeMfwdPSmyBuSAxIXjp2NAZ+co45PdayLJQV3Yyn5U4EY/g6CoQ580MQ6jHcCniTAHL8jj5etAEw2/7HenArj+DoKj8wDJJcAHP3D6UpmCBM7/AJ/lB2HimBYXbkcR9TTl27ekf3aiWZTyG7/3DT1mTkeYnAOcigCYbcniPqO9SLjI4T7xqDzk/wCesfOMcf8A16eJo8n97Fwc/p9aBEoAIA2xn5SOv/1qdgEH5E+6O/8A9ao0kjZlAkhJOeP8miN0kiR0KEPHkZ64oAmK4LYRT07/AP1qXb1/dp94d/p7VEzID/yzJbgAfQn+lOkKqfuxnLAf549qAH7Tn/VqPmHenbf9gfe9aYyjnEcf3h3+ntSE/vPL2Lk/Nwfce3vTAk2/9M/4vWjaBn5P4h3+lG3r8gPzDv8ASobUh7KJwu7cFIIPXgUATkfexGeo7/Skbo/yt0/vf/XqJ8+eq7TtYEkbuuNv/wBepX4D/K3Ts3/16AFZRub5G5H976+9IB8y/I33f73096jgJYOSr5DMPvejH3pJ3KQbwHBCj+L3HvQBIBwnyv1/vfX3pD0b5X+9/e/+vS9NvEn3j3+tQu2J1X95ghj+q0hkv8TcP0/vfX3rk/Hn3dM6/wDLTqf92ur/AIm5fp7+9cn49Py6Z1/5adf+A1UdxM5KikoqhATTTSmmmgANNNBNIaAENNPWlNMJ5oAiIxU0bblwetR/eFNUlWrLcomxtPtV3Trjy5PLY/I3T2NVAQ65pFz07ipHY6PFPjQuwFZ8F/Ctv/pEqxsvGW7/AIUxNasbUu0Xn3DsQSxGBx2HoKcY3MmY160st5KZ2yysVwOgAPaqjcScdM1LPKJriWVV2h3LYznGTUL9a36AixRQOlFAwooooADVjS38rV7J/SZP51XNCP5csb/3WDfkaGB6c4GSPl7jrSIAoAznHctzTpW+fI3HJ4AA700NkA8846rWJY4dOnb+9TwPY9x96ox07dD/AA08Yz/D19PagCQZ9G7d6V1ZkwMhhkgnnB/OoxjH8HQVIuM/w9fWgB8e8L87s7ZBzgCplJyOW6nsKgH3eg6D+KpRnPQ9f73tTEPzlc5b7vp/9apN2cjJxkdVqIA4xhuhHWnZPJw3Y9RQBKpHGcEbjwU4qG1iMarE4jMaqQrEcnnj9KkBOf4+vt6U5WOV+Zup7f8A1qAGyJkKyOuRg89PQ9T6E0sqAhirpuG1vm+vWnKxwPmP3f7tBf5T8wGV7rQA4oPm5BzjuKRoQZN3y4HB+Y5I4pWb73K9Aen/ANelYjLf6vt1oAcYVBJEar8wxtbGOntVWyjMUItmSX9yoXf90HGBxVk7fn4Q4I5pWHEnyL27/wD1qYDXg+feN4K/7WePx/CnNGBvGJPu/wDPQ+/vQ4xv+Tt60rDlvlblf73196AGpFsYgNKQcn7wprxF4wu+Xle5B9KkH3hw/T+9/wDXpFPCff6etACbWwpMkud3OAPemNG3mCQSOQpIwV7HH+Ap+eF5f7x7fWkJ4bl+v936e1AxxPzN8zdB2+vtXJ+PTldM5P8Ay06j/drq8/M3zHp6fX2rk/HpyNN5z/rP/Zaa3EzkqKTPFBqhCGkJoJppNAATTTSmmmgANNJoptAEaHtT2GRkVGaepzWRQ6Jtpx2NLPIIwCPvdhUUrBMep6VXJJOTyaajfUTZJHDNcsxRGcgZJ7CosVYkvJXtltxhIVOdqjGfr61XrYjUeEO3djC+pqN6dknAyTjpSOCAM0gJUOUFOqOI/IKkpjCiiloEIaZJ900+msOKBnpULiW1gk4+ZEb73sKVVdS2MEH/AGjxz/8AXqto7mTRbJuf9SB09OP6VbCgE4A5Jz8tYFinzM8Dv2NOJfacK4yOORmkAGf4evpTuApPGAM9fSgBwZsfdfofSlR2x8ysTnuv5VGs0RP3wDkjBYiplyVBGSCAcg0AIrnLAg47fJTzKvmKPkAPJyMelKM/7Xf0pQTjqegPSmANMipuBQkZ74qQOmMZU/L2ajJzye/dfagEYx8nQjpQIckoZcnjLY+/6HFSRtuzw3yuV61EAnOFj5weOKXy0zxGo+bscUAS79ihmLgDI6D1FK0myMsWfATJ+XrURiVgAQ4BJzhz/jS7NygM0pBUg8igROW4bJP3fSh5ACQWXLDAyOvU1Dg7R+8mB29R/wDqoYE4JmkJUbhuX2I/rQBOzA7+YycUEhvMKiNuOue4zmo2LfNmYD5em0e9NCsokHmRHOWPy+uT60wLDgfP8q8jA569aUjk/KenY1XkjkZ1OIm2/N6diMdPen4lz92P7vbNAyReq/K3T+99PekU8Jw449agiEiFj5Zy5yP3nA4H+FEjSGEhUkVipClWzzikBPngct97096CeG+Zuvp9Pao/M4Hyzfe/u+9MaRvMUZkUZO4FfpjtQMnLfM3zdvT61ynjxsjTec/6z/2WupLfM3z9h1H1rk/HjZ/s3kHiTp/wGmtxM5TNJmkzSZqxAaQ0E00mgAJpCaSkNAATTSaWmmgBuOtMZ9h460juc7VBLH0qSOyldjvwgUbmLfwj1Pp/OpS7g2V8s7dyxoKlTg9atFo4/lgBC92P3m/wHtTCVYcitlEjmK9PWIt14FSKqjp1pzMFXNNQ7icuww7YhkDJqFyWyTSkljk0h6GpbuNIls51t5C7wpMpBG1+nNW5LuxmVN9tJGVXH7thiqKKHEYLBQWwT6e9SR2vmBMSxAucYLdKgodO9sWH2cSqv+2wNNkaLZH5Rfdt+fd0z7VEYnDgNxnv7etNC5HUU7gP3H1o3H1phUgZ4/Og+o6Gi4FyHU7+3jEcF5NGi9FDcD8KsJ4h1ZOl65/3lU/0rKzRmloBtp4q1Zes0bf70QqdPGGpL96O2f6oR/WudzRmiyA6lPGt0D89jbN9CRU8fjbOFbTASeBsl/8ArVyGaVHKOrr1Ugj8KLILs7r/AITCCNwk+m3MT9du4Z/XFTL4t01lw8F4uVx90H+tcZfatNfXcdxKiB0GBjODznvSLqCDfvtonLMWyQcjPpgim4x6CjKTWp3CeI9EJ/1t1FznlG/pmrMOvaMDldRI6jEgb+orzxrqNnLBdgJztHQUonjP8X6UuVFXPTE1jS5B8uo2pOO7gfzqyl1ZyZ2XVq/I6SL/AI15X5kR6lfyqxp40k2N4b+WYXOD9nCdM44zx64pcoXPUlAbG3Y3P8LZp4RgR8jd+hrya3t7d9Kln+1iO6jJJjOBxxgDuScnp0xV62tZm0Q38WreXKqsxt95DHae2DSdkC1PSgCMfKw47GlBOB8z/d9K85trvUf7La8TWJUdASImlbJAOOCeM+1WoL7xN/Yx1NLzdbKCcMylsA4JwR603GxKknsd8GOOCRlepFLu/wBpT8vGR1rz6HxPr/2I3e5JIVO3LRpn34GCQPWrlp4p124tXuo7GGaCPIZwhGMdf4qTjbcad9jtev8AcOVpR1X5F6dj9K4aPx9Jx5ljA3GOGYf41Yj8d25I8zT+g/hmH9RRYdzsF42fKenZvpQDjbw3X1+tctH420443WtwuBj5Sp/qKsR+L9IbGTdJz3TP8jSsO50OeBy/3v60hbh/mbr6fSsdfE+jMP8Aj8dOc/NE3+FSrr+kPnbqcAJ/vHb/ADFFmM0y/wAzfMOncfWuT8eNk6byDxJ0/wCA10KanYSE7NRtG4/56r/jXM+OZY5Tp3lyxyALJkowP92hbiZy+aM03NGasQpNNNBNJmgAzSUZooASkpTTaBGmWlkumRUiiuWG6VtoAgX1b3x2qhe3aSkQW24W0ZzlvvSt/eb+npT9Sb7MXsISSitmaQ/elfrz7D0qgBTihDs0uabSVpcmw/dTW+YU+OGSXGxSR69qSRGjcqwP5daLgRdKWpPKdo2fadq+1RZqHoUNDEAjsaTNOpDipGAYjOOMjFAJHSkxRQA4uSuCB9e9Ieg+lIAScCnMrJ94YoASikozQAtFJRQAtJRRQAUUUUAFFFFABS5NFFABk0bjgjseaKSgCQTuIjGCNhOSMd/rUyajeR2bWiXUq2z53RBjtOevFVaKdwLcOo3cFq9tFOVhfIKYB69e3Gas2Gt3thYTWcDJ5M2cggHrway6KT10YLTVE9tMbdy3kxSgrtKyLuFWHvYnADadarzyUDKTxj1P1qhRz6mgC2k9nukM1oSCSVCSkY9B9KLU2OGF0twTn5TEV6e+aq7m/vH86Nzex+oFAExMYlfYZPLydmeDj3qLGaN3+yv5Ubh3QfgT/jQAYpVJXoSPoaTcv90/g1LlP9sfkaAHebJ2dvzp8bTyuqI2Wbpkgfqai+Q/xMP+A/8A16U4PWUHHHINAFq1gvbySRLeIytGpZwAOBSWqzXTsqvAhUZ/euEz7c1XjLRsGjmVGHQhipp8DSwTCVPJZhn75Vgc+xpgSTmW2naKQIXXrsbcPzFM88/3KYEl8zeqgHO4bcYB9qs211fWl01zEGErZ3Ex5Bz7YpAQfaF7gijzl9DVhNSuYvPzGh8+TzH3IQc+2Mcc9KpSPvkdyFUsScKMAfQUAWLxvMvbh/WRj+tRUsxUzylDlS5IPqM0yrQhSabk0dacyleowaQFiG88mMoqduDnnNVw5aQPIzNznOeabQaLhYsz30soKj5F9utVDmilpN3HYKSlopAJRRRQAYoxRRQAUUZozQAlWFlt/K2ta5cDG8SEc+uKgo4oASilxTlKhXBXcSMKc4289ff0oAZVlbGd1DJ5TZ7CRc/zqvik2+1AD5I3ifZIu1h2zTc+1JinxSyQtuico3qKAG5HvRx60MzOxZjlick+tJQAuB60Y9xSUUALikoxRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQAVOJUwFdGOOuH61AOtKVJJwKAHvtcnykbA5O45phDrjIIzT42KBhtJ3DH0okfcirgjaTyaBiCaQdJGH4ml+0y/3zSEqVXAxgc8daG25OMYoAOlFJSqrOwVFLE9hVCJElZFKpxnqcc1HmkzQOaBC0hoNJSYwpaSikAtFJS0AJRRRQAUUUUAFFFFABRRRQAUUlFAC0UlFAC5pc02loAXPtRx6UlFAC4FG0etJRQAEY75o2n0ozTt5oAbSVJv9qXevcUARUcelS/uz6UuyM9D+tAEOBRj3qfyQejH8qQwHswoAhx70YNS+Q/bB/Gk8mT+4fwoAjwRRzTirDqpH4UmaAEopaKAEopcZoxQAmaXcfU0baMUAGTRn2FGKMUASQxtLMsaruJPSujl+x2Vs8iRqqsNp2clvbNcwCexIp25tmzJ25zjtmqTJlG4shQuTGpROwJzTScUdqbSKCiilAycUgFQLuHmFgvtUqwCSQiJyyKu9mIxtFRrLIgIV2APbPFONxIYTFkBGIJAUDOOmaasJ36EeeeOlGaSikMWgjHUYpd58vZgYznOOfzqa8jELxxrk/u1Yk9yRn+uKLCvrYgoqeWAQw28pO7zVLbemKikAG0gYyM0DuNpKWigBKKWigBKKWkoAKKKKACiiigAooooAKKKKAClpKKACiiigAooooAWlDsP4jTaKAJBM47/AKU4XDDqAahooAsC5Pp+RpftIPUH+dVqSgC2JYT1Rf8AvmlH2Zuyj8SKp0UAaC28DdGYfQ5qrKnlyMpOcHr61FS5NAC0UmaBQAtGaKXFAH//2Q=='}
TRUTH = {'IMG_20261110_132617': {'page': 'P01', 'shot': 'A_straight', 'tool_light': 'GREEN', 'true_hr': 55.5, 'read_hr': 55.6, 'none': False, 'wrong': False}, 'IMG_20261110_132655': {'page': 'P02', 'shot': 'A_straight', 'tool_light': 'GREEN', 'true_hr': 98.6, 'read_hr': 98.6, 'none': False, 'wrong': False}, 'IMG_20261110_132704': {'page': 'P02', 'shot': 'B_angled', 'tool_light': 'AMBER', 'true_hr': 98.6, 'read_hr': 99.4, 'none': False, 'wrong': False}, 'IMG_20261110_133206': {'page': 'P20', 'shot': 'B_angled', 'tool_light': 'RED', 'true_hr': 82.4, 'read_hr': 108.8, 'none': False, 'wrong': True}, 'IMG_20261110_132921': {'page': 'P10', 'shot': 'B_angled', 'tool_light': 'RED', 'true_hr': 85.1, 'read_hr': 60.4, 'none': False, 'wrong': True}, 'IMG_20261110_132723': {'page': 'P03', 'shot': 'B_angled', 'tool_light': 'RED', 'true_hr': 100.2, 'read_hr': None, 'none': True, 'wrong': False}}
COL = {'GREEN': '#2e9e4f', 'AMBER': '#f0a30a', 'RED': '#d62728'}
for vid in EXAMPLES:
    leads = {k: np.where(v == -32768, np.nan, v.astype(float)) for k, v in np.load(io.BytesIO(base64.b64decode(EXAMPLES[vid]))).items()}
    r = T.check(leads)
    tr = TRUTH[vid]
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={'width_ratios': [1, 2.4]})
    ax[0].imshow(Image.open(io.BytesIO(base64.b64decode(THUMBS[vid])))); ax[0].axis('off')
    ax[0].set_title(f"photo ({tr['shot'].replace('_', ' ')})", fontsize=10)
    sig = np.asarray(leads['II'], float) / 1000.0
    t = np.arange(len(sig)) / 1000.0
    ax[1].plot(t, sig, lw=0.8, color='#333')
    ax[1].set_xlabel('seconds'); ax[1].set_ylabel('lead II (mV)'); ax[1].set_title('what the digitiser drew (lead II)', fontsize=10)
    fig.suptitle(f"{vid}   ->   {r['light']}", fontsize=14, weight='bold', color=COL[r['light']])
    plt.tight_layout(); plt.show()
    print('Reasons shown by the tool:', '; '.join(r['reasons']))
    if tr['none']:
        print(f"For the demo only (the true signal is not used by the tool): true heart rate {tr['true_hr']} bpm; the digitised signal gave no usable heart rate.")
    else:
        print(f"For the demo only (the true signal is not used by the tool): true heart rate {tr['true_hr']} bpm, read from the digitised signal {tr['read_hr']} bpm -> {'WRONG (> 5 bpm off)' if tr['wrong'] else 'right (within 5 bpm)'}")
    assert r['light'] == tr['tool_light'], 'the notebook gives a different light from the evaluation run'
    print('-' * 100)


## 2. How often was each light right? (the tool was frozen before these tests)
*Wrong heart rate = more than 5 bpm away from the true value. The tool was trained only on simulated damage; thresholds were chosen only on the training data.*


In [ ]:
from io import StringIO
TABS = {'comp_with_output_all': 'light,readings,wrong,wrong_rate,share_of_all_wrong,share_of_all_right\r\nGREEN,917,2,0.2% (0.1-0.8),4%,79%\r\nAMBER,204,14,6.9% (4.1-11.2),30%,16%\r\nRED,82,30,36.6% (27.0-47.4),65%,4%\r\n', 'comp_incl_no_output_all': 'light,readings,wrong,wrong_rate,share_of_all_wrong,share_of_all_right\r\nGREEN,917,2,0.2% (0.1-0.8),2%,79%\r\nAMBER,204,14,6.9% (4.1-11.2),14%,16%\r\nRED,139,87,62.6% (54.3-70.2),84%,4%\r\n', 'own_with_output': 'light,readings,wrong,wrong_rate,share_of_all_wrong,share_of_all_right\r\nGREEN,22,0,0.0% (0.0-14.9),0%,55%\r\nAMBER,15,1,6.7% (1.2-29.8),7%,35%\r\nRED,17,13,76.5% (52.7-90.4),93%,10%\r\n', 'own_incl_no_output': 'light,readings,wrong,wrong_rate,share_of_all_wrong,share_of_all_right\r\nGREEN,22,0,0.0% (0.0-14.9),0%,55%\r\nAMBER,15,1,6.7% (1.2-29.8),5%,35%\r\nRED,23,19,82.6% (62.9-93.0),95%,10%\r\n'}
names = {'comp_with_output_all': 'Competition real captures (1,203 readings with an output)', 'own_with_output': 'Our own phone (54 readings with an output)',
         'comp_incl_no_output_all': 'Competition, counting "no output" as a failure', 'own_incl_no_output': 'Our own phone, counting "no usable heart rate" as a failure'}
dfs = {k: pd.read_csv(StringIO(v)) for k, v in TABS.items()}
for k in ('comp_with_output_all', 'own_with_output'):
    print(names[k]); display(dfs[k][['light', 'readings', 'wrong', 'wrong_rate', 'share_of_all_wrong', 'share_of_all_right']]) if 'display' in globals() else print(dfs[k].to_string(index=False))
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for a, k in zip(axs, ('comp_with_output_all', 'own_with_output')):
    d = dfs[k]; rate = [100 * w / n if n else 0 for w, n in zip(d.wrong, d.readings)]
    a.bar(d.light, rate, color=[COL[x] for x in d.light])
    for i, (r_, n_, w_) in enumerate(zip(rate, d.readings, d.wrong)):
        a.text(i, r_ + 1, f'{w_}/{n_}', ha='center', fontsize=10)
    a.set_title(names[k], fontsize=10); a.set_ylabel('percent of readings with a wrong heart rate'); a.set_ylim(0, 100)
plt.tight_layout(); plt.show()


## 3. How to read this and what the limits are
- **GREEN readings were almost always right** (competition: 2 wrong out of 917; own phone: 0 wrong out of 22). **RED readings were wrong far more often** (competition 37%, own phone 76%). AMBER is in between.
- Together AMBER and RED hold most of the wrong readings (competition 96%, own phone 100% of the wrong ones with an output), at the price of flagging some right readings (competition 21%).
- **It checks the signal, not the truth.** It says "the beats in this signal are unreliable". Irregular rhythms such as atrial fibrillation are flagged more often than regular ones, even when the heart rate is right.
- **Limits:** only 46 wrong readings on the competition real captures and 14 on our phone; one phone, one screen, 30 ECG recordings; the intervals shown elsewhere treat images as independent, which is optimistic; the digitiser was trained on pages like the ones we tested. The tool was trained on simulated damage and thresholds were fixed before the real-photo tests. One safety rule (out-of-range guard) was added after a unit test showed noise got GREEN, before any real-photo evaluation.
- **Not a medical device.** It is a research prototype for a capstone study.
